<div dir="rtl"><h2>03 — מצב לפני שריקה: פעימה A</h2>
<p>לפני שחוזים משחק, צריך לדעת איך נראתה הליגה בבוקר שלו. <b>M0</b> בונה לכל קבוצה שורה מתוך תוצאות בתאריכים קודמים ממש, לפי שעון ישראל. משחקים מאותו יום אינם רואים זה את זה.</p>
<p><b>טבלה על המגרש, בלי הורדות; עשויה להיות שונה מהטבלה הרשמית.</b> נקודות הן שלוש לניצחון ואחת לתיקו; בשתי עונות החצייה מעגלים למעלה. 11 המשחקים הטכניים אינם קלט לפיצ׳רים.</p>
<p>בפעימה זו: חוקי ליגה, טבלת as_of, רישום הפיצ׳רים ודגל מתמטי שמרני. Elo, כושר, התקפה/הגנה והקשר מהעונה הקודמת יגיעו בפעימות B ו-C. הסטטיסטיקות כאן על development בלבד: <bdi dir="ltr">2006/2007–2023/2024</bdi>. עונות ההחזקה מופיעות רק בשער האימות ההנדסי המאושר.</p>
<ol><li>הגדרות וקלט</li><li>חוקים ואימות</li><li>הטבלה בבוקר המשחק</li><li>רשמי מול ספורטיבי</li><li>יעדים פתוחים ושלב העונה</li><li>שמירה והמשך</li></ol></div>

<div dir="rtl"><h2>סביבת העבודה</h2><p><b>מה התא עושה:</b> טוען את פונקציות החבילה, קובע את שורש הפרויקט ומוודא שימוש ב-venv. הטבלאות מוצגות בעברית וב-RTL.</p></div>

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import HTML, display

from ligat.analysis.prematch_review import (
    championship_examples,
    dead_game_summary,
    postponed_example,
    registry_overview,
    rules_overview,
    split_comparison,
    sporting_official_comparison,
    stage_end_distribution,
    validation_overview,
)
from ligat.analysis.prematch_validation import load_technical_ledger, validate_prematch_tables
from ligat.analysis.prepare import DEVELOPMENT_SEASONS, load_analysis_tables
from ligat.data.repository import Repository
from ligat.features.artifacts import input_fingerprint, save_table_state
from ligat.features.registry import validate_feature_columns
from ligat.features.table_state import TableStateBlock
from ligat.simulate.rules import load_season_rules
from ligat.viz.prematch import dead_games_chart, review_table, stage_end_chart

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Select the project venv kernel"
started = time.perf_counter()
pio.renderers.default = "plotly_mimetype"
display(HTML('<div dir="rtl">סביבת ה-venv אומתה; החישובים מיובאים מהחבילה.</div>'))

<div dir="rtl"><h2>טעינת הקלט לפיצ׳רים</h2><p><b>מה התא עושה:</b> טוען עונות פיתוח דרך load_analysis_tables בלבד. הפיצ׳רים ייבנו ב-DataFrame חדש; הטבלאות הנקיות אינן משתנות. עמודות EDA קיימות, סיכומי סיום ושדות שנדחו אינם נכנסים לחישוב.</p></div>

In [ ]:
tables = load_analysis_tables(seasons=DEVELOPMENT_SEASONS)
rules = load_season_rules()
display(
    HTML(
        f'<div dir="rtl">נטענו {len(tables.match_table):,} משחקים '
        f"ו-{len(tables.team_matches):,} שורות קבוצה×משחק בעונות הפיתוח.</div>"
    )
)

<div dir="rtl"><h2>1. חוקי הליגה וקווי היעד</h2><p><b>מה התא עושה:</b> מציג את הפורמט, החצייה, שוברי השוויון וקווי העליון/הירידה לפי עונה. בפלייאוף מוצגים גודל הקבוצה ומספר המשחקים לכל קבוצה.</p>
<p>ב-<bdi dir="ltr">2011/2012</bdi> ירדו שלוש קבוצות. בשתי העונות שקדמו לה מקום 14 הוביל למבחן ונחשב כאן חלק מאזור הסיכון. מפגש ישיר נשאר שובר שוויון בתקנון; שוויון מוחלט מקבל מיקום ממוצע. בפיצול משתמשים במיקום העונה הקודמת אם עדיין נותר שוויון.</p></div>

In [ ]:
display(review_table(rules_overview(rules, list(DEVELOPMENT_SEASONS))))

<div dir="rtl"><h2>2. שער האימות — טבלאות רשמיות כבדיקה בלבד</h2><p><b>מה התא עושה:</b> משווה 9,956 שורות מחזור בעונות <bdi dir="ltr">2006/2007–2025/2026</bdi>, וכן טבלאות סיום. זו בדיקת הנדסה; אין מחקר או בחירת פיצ׳רים על עונות ההחזקה.</p>
<p>מונים נבדקים במחזורים ללא משחק שחוצה מחזורים. המשחקים הטכניים משוחזרים בספר נפרד לצורך הסבר הפער בלבד. מיקום רשמי נבדק ללא הורדות. בקצות השלבים האימות קפדני; באמצע העונה מותר סדר תצוגה אחר רק בתוך קבוצות שוות בנקודות ובהפרש, עם סף עצירה של 1%.</p>
<p>ארטיפקט מקור מאומת ב-<bdi dir="ltr">2006/2007</bdi>: בטבלה הרשמית חסרים שערי חלק ממשחקי התיקו. מותר רק פער שווה בשערי הזכות והחובה; נקודות, נ/ת/ה והפרש נשארים מדויקים. התוצאות עצמן אומתו מול אירועי השערים. שום כלל דירוג אינו משתנה בגלל אופן הצגת הטבלה באתר.</p><p><b>תוצאות האימות:</b> במחזורים הנקיים 7,572 מתוך 8,196 שורות תואמות ישירות (92.39%). יתר השורות מוסברות ב-467 פערים טכניים וב-157 ארטיפקטים; כולן עברו לפי המדיניות שאושרה. 1,760 שורות של מחזורים חוצים הוחרגו. נקודות הסיום עברו בכל 280 השורות אחרי הסבר הטכניים. סדר תצוגה: 12 מתוך 9,432 שורות ביניים (0.13%), מתחת לסף 1%.</p><p>בדיקה הנדסית בלבד של rules.py מול הטבלאות הרשמיות; אין כאן מודל, ציון או בחירה, ולכן היא אינה חלק מהאיסור על עונות 26–28 במחקר.</p></div>

In [ ]:
engineering_tables = load_analysis_tables(seasons=list(range(8, 28)))
repo = Repository()
official_rounds = repo.read_filtered("round_tables", [("season_id", "in", list(range(8, 28)))])
official_finals = repo.read_filtered("final_tables", [("season_id", "in", list(range(8, 28)))])
validation = validate_prematch_tables(
    engineering_tables,
    rules,
    official_rounds,
    official_finals,
    load_technical_ledger(list(range(8, 28))),
)
assert len(official_rounds) == 9956
display(review_table(validation_overview(validation)))
display(
    HTML(
        '<div dir="rtl"><b>שער האימות עבר.</b> ״זהה בספורטיבי״ מציין התאמה ישירה; '
        "פער טכני וארטיפקט מקור הם פערים מוסברים שנבדקו, "
        "ומחזורים חוצים אינם נכללים באחוז ההתאמה.</div>"
    )
)

<div dir="rtl"><h2>3. בניית הטבלה בבוקר המשחק</h2><p><b>מה התא עושה:</b> יוצר שתי שורות לכל משחק, עם מפתחות game_id ו-team_id. כל שורה משתמשת רק במשחקים מתאריך קודם באותה עונה. הפערים חתומים: נקודות קו היעד פחות נקודות הקבוצה; ערך חיובי משמעו שהקבוצה מפגרת בנקודות אחרי הקו.</p>
<p>בפלייאוף, המיקום הכללי הוא היסט הקבוצה בטבלה ועוד המיקום בתוכה, גם אם לקבוצה בתחתון יותר נקודות מקבוצה בעליון. לכן פער בנקודות בין קבוצות פלייאוף אינו אפשרות לעבור ביניהן. אחרי הפיצול, פער לקו שאינו נגיש הוא NA: פער לירידה בעליון ובאמצעי, ופער למובילה באמצעי ובתחתון; פער לקו העליון קיים רק בסדיר לפני פיצול. נקודות למשחק הן הנקודות המחושבות, כולל חצייה כשכבר חלה, חלקי המשחקים ששוחקו; לפני משחק ראשון הערך הוא אפס.</p></div>

In [ ]:
build_started = time.perf_counter()
features = TableStateBlock(rules).compute(tables)
build_seconds = time.perf_counter() - build_started
validate_feature_columns(features, "table_state")
assert len(features) == 2 * len(tables.match_table)
display(
    HTML(
        f'<div dir="rtl">נבנו {len(features):,} שורות ו-{len(features.columns)} עמודות '
        f"ב-{build_seconds:.2f} שניות. הקלט והטבלאות הרשמיות אינם משתנים.</div>"
    )
)

<div dir="rtl"><h2>משחק נדחה: מספר המחזור אינו ציר זמן</h2><p><b>השאלה:</b> למה לא לקחת את הטבלה הרשמית של המחזור?</p><p><b>מה התא עושה:</b> מוצא משחק אמיתי של קבוצה שמשחק ממחזור קודם שלה הושלם רק אחריו, ומשווה בין שורת הבוקר שלה לטבלה הרשמית.</p>
<p>הטבלה הרשמית היא אחרי המחזור, ולכן היא כוללת גם את תוצאת המשחק הנחזה ומשחקים נוספים מאותו מחזור. הדוגמה מבודדת במפורש גם משחק ממחזור קודם ששוחק בעתיד; זו המחשת דליפה, לא השוואה של שני מדדים לאותה נקודת זמן.</p></div>

In [ ]:
postponed_context, postponed_tables = postponed_example(tables, features, official_rounds)
display(review_table(postponed_context))
display(review_table(postponed_tables))

<div dir="rtl"><h2>4. רשמי מול ספורטיבי — הורדות נקודות</h2><p><b>מה התא עושה:</b> מציג עונות שבהן יש הורדות או פער טכני, ומפריד בין הנקודות הספורטיביות, התוספת הטכנית לצורך האימות והנקודות הרשמיות. לצורך המודלים משתמשים רק בספורטיבי.</p>
<p>ב-<bdi dir="ltr">2011/2012</bdi> הורדות נקודות השפיעו על קו הירידה: לכן ״ירדה בפועל״ היא תווית אבחון של הטבלה הרשמית, ולא יעד אימון ספורטיבי. גם המיקום הספורטיבי כאן משתמש בשיבוץ הפלייאוף הידוע בפועל.</p></div>

In [ ]:
official_comparison = sporting_official_comparison(tables, rules, official_finals)
display(
    review_table(
        official_comparison.loc[
            official_comparison.rank_changed | official_comparison.deduction.ne(0)
        ]
    )
)

<div dir="rtl"><h2>השיבוץ ההיסטורי נשמר — הנקודות נשארות ספורטיביות</h2><p><b>מה התא עושה:</b> מציג את כל ההבדלים בין שיבוץ ספורטיבי מחושב לבין השיבוץ מהלוח בפועל, עם הסיבה המאומתת.</p>
<p>ב-<bdi dir="ltr">2010/2011</bdi> הניצחון הטכני של ב״ש על אשדוד נתן לה שלוש נקודות נוספות והכניס אותה לאמצעי. בלעדיו בית״ר הייתה באמצעי וב״ש בתחתון. בפועל בית״ר שיחקה חמישה משחקי תחתון במאבק ירידה וב״ש שלושה משחקי אמצעי. ב-M0 משתמשים בשיבוץ מהלוח כמידע K רק מרגע הפיצול; ב-M4 הפיצול ייגזר מהמסלול המדומה. final_stage נשאר לאימות בלבד.</p></div>

In [ ]:
display(review_table(split_comparison(tables, validation)))

<div dir="rtl"><h2>5. כמה משחקים נשארים בלי יעד תחרותי פתוח?</h2><p><b>מה התא עושה:</b> סופר בכל עונה ושלב משחקים שבהם לאף קבוצה, לאחת או לשתיהן אין יעד פתוח. המכנה הוא משחקים, ולא שתי שורות הקבוצה של כל משחק.</p>
<p><b>״אין על מה לשחק״ כאן הוא דגל מתמטי שמרני, לא ציון מוטיבציה:</b> הוא בודק אליפות, קו עליון וירידה, ובעליון ובעונות בלי פלייאוף גם את קווי המקומות 2, 3 ו-4 (ה-ת10). שוויון נקודות נשאר פתוח. לפני הפיצול מוכיחים פסילת אליפות דרך פסילה מהעליון, ובטיחות דרך הבטחת שיבוץ שאינו תחתון. אחרי הפיצול משתמשים במעטפת נקודות בתוך הקבוצה. השיטה עשויה לזהות הכרעה מאוחר; היא אינה מניחה שקבוצה בטוחה תפסיק להתאמץ.</p>
<p>באמצעי הישן אין יעד רלוונטי, ולכן כל הקבוצות בו מסומנות בלי יעד. בעליון הירידה כבר מוכרעת; בתחתון האליפות והקו העליון כבר מוכרעים.</p><p><b>למה נוספו קווי המקום (ה-ת10)?</b> גם אחרי שהאליפות הוכרעה, קבוצות בעליון עוד נלחמות על הסגנות ועל אירופה. לכן ״מוכרע״ מחייב שהצד של הקבוצה ביחס לכל קו בין המקומות <bdi dir="ltr">2|3, 3|4, 4|5</bdi> יהיה נעול במעטפת הנקודות. המקום הרביעי נכלל תמיד מטעמי שמרנות, בלי לנסות לקבוע כאן מי תקבל כרטיס אירופי; אירופה אינה יעד חיזוי נפרד.</p><p>טבלת ההשוואה בעליון מחושבת כאן מהעמודות בזיכרון: ההגדרה הישנה היא AND של title_decided, cut_decided ו-relegation_decided, והחדשה דורשת גם places_decided. שיעור שורות קבוצה אינו שיעור משחקים; המכנים מוצגים בנפרד.</p><p><b>מגבלה מתועדת — מבחן הירידה:</b> בעונות <bdi dir="ltr">2009/2010–2010/2011</bdi> מקום 14 הולך למבחן ירידה, ומקומות <bdi dir="ltr">15–16</bdi> יורדים ישירות. הדגל בתחתון רואה את שלושתם כצד אחד של קו הירידה. לכן קבוצה שכבר לא יכולה לסיים מעל מקום 14 מסומנת ״מוכרע״, אף שהיא עוד נלחמת על מבחן הירידה ולא על ירידה ישירה. המגבלה נוגעת רק לתחתון בשתי העונות; היא מתועדת כאן ללא שינוי בקוד הדגל.</p></div>

In [ ]:
dead_summary, match_flags = dead_game_summary(features)
display(review_table(dead_summary, percent=("at_least_one_share", "both_share")))
stage_totals = (
    dead_summary.groupby("stage")[["games", "neither", "one", "both"]]
    .sum()
    .reindex(["regular", "upper", "middle", "lower"])
    .reset_index()
)
display(review_table(stage_totals))

legacy_flag = features[["title_decided", "cut_decided", "relegation_decided"]].all(axis=1)
upper_comparison_rows = []
for label, flags in [("לפני ה-ת10", legacy_flag), ("אחרי ה-ת10", features.nothing_to_play)]:
    upper_flags = flags.loc[features.stage.eq("upper")]
    upper_comparison_rows.append(
        {
            "definition": label,
            "team_games": len(upper_flags),
            "dead_team_rows": int(upper_flags.sum()),
            "dead_team_share": float(upper_flags.mean()),
            "both": int(upper_flags.groupby(level="game_id").sum().eq(2).sum()),
        }
    )
upper_comparison = pd.DataFrame(upper_comparison_rows)
display(HTML('<div dir="rtl"><b>בעליון — הדגל לפני ואחרי ה-ת10</b></div>'))
display(review_table(upper_comparison, percent=("dead_team_share",)))
dead_counts = match_flags.dead_teams.value_counts().reindex([0, 1, 2], fill_value=0)
display(
    HTML(
        f'<div dir="rtl">מתוך {len(match_flags):,} משחקים, '
        f"ב-{int(dead_counts[1] + dead_counts[2]):,} "
        f"({100 * (dead_counts[1] + dead_counts[2]) / len(match_flags):.2f}%) "
        "לפחות קבוצה אחת בלי יעד פתוח לפי ההגדרה החדשה.</div>"
    )
)

<div dir="rtl"><h2>התפלגות המשחקים בלי יעד לאורך העונות</h2><p><b>מה התא עושה:</b> מציג את שיעור המשחקים עם לפחות קבוצה אחת בלי יעד פתוח, בנפרד לכל שלב. בריחוף מופיעים גם המכנים ומספר המשחקים שבהם קבוצה אחת או שתיים בלי יעד. זהו תיאור של הדגל שהוגדר, לא אומדן להשפעת מוטיבציה על תוצאות.</p></div>

In [ ]:
dead_figure = dead_games_chart(dead_summary)
display(dead_figure)

<div dir="rtl"><h2>דוגמאות: אלופה שמגיעה למשחק עם תואר מוכרע</h2><p><b>מה התא עושה:</b> לכל עונה מציג את המשחק הראשון של המובילה שבו הדגל כבר מוכיח שהאליפות שלה, וממיין לפי משחקים שנותרו בשלב.</p>
<p><b>זה אינו תאריך ההכרעה המתמטית המדויק:</b> האליפות יכולה להיות מוכרעת בין משחקי האלופה, והדגל השמרני עשוי לזהות זאת מאוחר. אלו דוגמאות מתועדות לפני משחק, ולא דירוג מוחלט של ״האליפות המוקדמת בהיסטוריה״.</p></div>

In [ ]:
display(review_table(championship_examples(tables, features)))

<div dir="rtl"><h2>6. משחקים עד סוף השלב — אחרי המשחק הנחזה</h2><p><b>מה התא עושה:</b> מציג את התפלגות המונה ומוודא שאינו שלילי. games_remaining_stage כולל את המשחק הנחזה; rounds_to_stage_end מפחית אותו. הספירה מחושבת מהפורמט פחות משחקי העבר, ולא ממספר שורות עתידיות.</p>
<p>משחק טכני שהוחרג אינו תוצאה ספורטיבית שנצפתה. לכן אצל קבוצה כזו המונה עשוי להשאיר משחק אחד נוסף גם אחרי משחקה האחרון בפועל. זה מרחיב את מעטפת האפשרויות ומאחר הכרעות; לא ממלאים תוצאה או מעניקים נקודות למשחק שהוצא.</p></div>

In [ ]:
stage_distribution = stage_end_distribution(features)
stage_figure = stage_end_chart(stage_distribution)
display(stage_figure)

<div dir="rtl"><h2>רישום הפיצ׳רים ומעמדם</h2><p><b>מה התא עושה:</b> מציג את הרשימה המפורשת שמותרת בפלט. עמודה לא רשומה, לרבות שופט, מאמן, אצטדיון או אירוע מתוך המשחק הנחזה, מכשילה את הבדיקה. לדגל nothing_to_play יש מעמד ברירת מחדל עבור M1 והנחת Elo, אך הוא עדיין מועמד לאבלציה ב-M2. עצם הרישום אינו הוכחה לתרומה לחיזוי.</p></div>

In [ ]:
display(review_table(registry_overview()))

<div dir="rtl"><h2>טעינת גרסה 2 מהשכבה הנגזרת</h2><p><b>מה התא עושה:</b> מאמת את ה־checksum של המטא־דאטה מול העותק הקפוא שאושר בפעימה A, ואת קובץ הטבלה מול ה־hash שבמטא־דאטה. טוען את הטבלה ומשווה אותה לחישוב בזיכרון, תוך נרמול טיפוס מספרי המזהים באינדקס בלבד (Int64/int64 לאחר Parquet). הערכים וטיפוסי כל עמודות הנתונים נבדקים בדיוק. אין דריסה של גרסה 2 או של הגרפים הקיימים. רק אם קובץ הטבלה חסר, נוצרת גרסה ונרשמת הודעה מפורשת.</p><p><b>מה רואים:</b> שהעותק השמור מתאים בדיוק למצב המחושב. אין משמעות כדורגלנית לבדיקה הטכנית.</p></div>

In [ ]:
from ligat.analysis.prematch_b_review import assert_same_table_state, load_table_state_snapshot

artifact = Path("data/derived/prematch/table_state_v2.parquet")
metadata = artifact.with_suffix(".metadata.json")
created_table_state = not artifact.exists()
if created_table_state:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    artifact, metadata = save_table_state(
        features, input_sha256=input_fingerprint(tables, rules), commit=commit
    )
    display(HTML('<div dir="rtl"><b>הקובץ היה חסר ולכן נוצר מחדש — יש לדווח.</b></div>'))
else:
    saved_features = load_table_state_snapshot(
        artifact,
        metadata_sha256="d3cfe3f409490b094889953cef461ff3cbdc2fe775fa79ea89cd7f2370049ec0",
    )
    assert_same_table_state(features, saved_features)
    features = saved_features
    display(HTML('<div dir="rtl">גרסה 2 אומתה ונטענה ללא כתיבה; הערכים זהים לחישוב בזיכרון.</div>'))

<div dir="rtl"><h2>מה הושלם ומה ממשיך לפעימה B</h2><p>יש כעת חוזה ברור למצב הטבלה לפני משחק: תוצאות עבר בלבד, שיבוץ היסטורי ידוע, נקודות ספורטיביות, פיצ׳רים רשומים ודגלים עם הוכחה שמרנית. הטבלאות הרשמיות שימשו לבדיקת תקינות בלבד, עם סיווג מפורש של ארטיפקטים, משחקים טכניים, הורדות וסדר תצוגה.</p>
<p>זו תשובה ראשונה לשאלת המחקר 6א ב-VISION: כמה משחקים נשארים בלי אחד היעדים שהוגדרו. כדי לטעון להשפעה על הביצועים, צריך בהמשך קנה מידה של חוזק וציפייה לתוצאה; הדגל לבדו אינו מוכיח ירידת מוטיבציה.</p>
<p>עד כאן פעימה A: מצב טבלה ודגלים, בלי רכיב נלמד. בחלק B1 הבא נבחן את Elo שנבחר ואת ניסוי EXP-001 מתוך התוצרים השמורים, בלי לאמן מחדש. הכושר וההתקפה וההגנה יוסברו בחלק B2 במשימה נפרדת; פעימה C תוסיף הקשר עונתי ושכבת prematch מאוחדת.</p><p><b>ממצא תיאורי:</b> מתוך 4,315 משחקי פיתוח, ב-156 משחקים (3.62%) קבוצה אחת בלי יעד פתוח וב-89 (2.06%) שתיהן; יחד 245 משחקים (5.68%). לדוגמה, מכבי ת״א הגיעה למשחק ב-7.4.2019 עם אליפות מוכחת בדגל, כשנותרו עוד שבעה משחקי שלב אחרי אותו משחק. זהו מועד תצפית לפני משחק שלה, ולא טענה שזה יום ההכרעה בפועל.</p></div>

<div dir="rtl"><h2>B1 — Elo ו־EXP-001: קוראים את הניסוי השמור</h2>
<p><b>מה התא עושה:</b> מייבא כלי תצוגה ומאמת את כל ה־checksums לפני פתיחת הטבלאות והתחזיות.</p><p><b>מה רואים:</b> רק עונות הפיתוח 2006/2007 עד 2023/2024. אין אימון, חיפוש פרמטרים או שחזור Elo.</p>
<p><b>הסבר נתונים:</b> הניסוי כבר הסתיים. בדיקות ה־hash מונעות ערבוב תוצרים מהרצות שונות; גרסת הקוד והקלט נשמרו במטא־דאטה.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לבדיקת שלמות הקבצים.</p></div>

In [ ]:
from ligat.analysis.prematch_b_review import (
    WORDS,
    alignment_summary,
    calibration_bins,
    configurations,
    control_numbers,
    conversion_examples,
    elo_example,
    fold_plan,
    gap_explanations,
    grid_overview,
    load_elo_review,
    prediction_diagnostics,
    score_example,
    score_ruler,
    season_scores,
    selection_comparison,
    top_configuration_contrasts,
)
from ligat.viz.prematch_b import (
    alignment_chart,
    calibration_chart,
    confusion_chart,
    draw_histogram,
    marginal_chart,
    season_score_chart,
    time_weight_chart,
)
from ligat.viz.prematch_b import (
    review_table as b_table,
)

b1_started = time.perf_counter()
elo_review = load_elo_review(ROOT)
b1_figures = {}
display(
    HTML(
        f'<div dir="rtl">כל הקבצים אומתו. {len(elo_review.predictions):,} תחזיות ולידציה של E0050; '
        f"{len(elo_review.prematch):,} שורות Elo לפני משחק בעונות הפיתוח בלבד.</div>"
    )
)

<div dir="rtl"><h2>B1.1 — איך תוצאת משחק מזיזה את Elo</h2>
<p><b>מה התא עושה:</b> מציג חישוב ידני בעזרת הנוסחה הקבועה של E0050.</p><p><b>מה רואים:</b> שתי קבוצות בדירוג 1500; הבית מנצח 2:0. E הוא 0.543 בקירוב והעדכון 6.9 נקודות בקירוב.</p>
<pre dir="ltr">E = 1 / (1 + 10 ** (−(R_home + 30 − R_away) / 400))
S = 1 (home win), 0.5 (draw), 0 (away win)
Δ = 10 × G × 1 × (S − E)
R_home ← R_home + Δ; R_away ← R_away − Δ</pre><p><b>הסבר נתונים:</b> E הוא ניקוד צפוי בסולם 0, חצי, 1 — אינו הסתברות לניצחון. K=10 ממתן את התגובה למשחק יחיד. HFA=30 נכנס לעדכון, לא נוסף שוב בשכבת ההמרה.</p><p><b>הסבר כדורגלני:</b> ניצחון צפוי נותן פחות מניצחון מפתיע. כאן הבית עולה לכ־1506.9 והחוץ יורד לכ־1493.1, והסכום נשמר.</p></div>

In [ ]:
update_example, multiplier_example = elo_example()
display(b_table(update_example))

<div dir="rtl"><h2>B1.1א — הפרש שערים, קיץ ועולות</h2>
<p><b>מה התא עושה:</b> מציג את מכפיל הפרש השערים שבקוד הייצור.</p><p><b>מה רואים:</b> בהפרש 0 או 1 המכפיל 1; בהפרש 2 הוא 1.5; מ־3 ואילך הוא ‎(11+n)/8.</p>
<pre dir="ltr">R_next = 1500 + (1 − 0.33) × (R_end − 1500)</pre><p><b>הסבר נתונים:</b> E0050: נסיגה בקיץ r=0.33; לעולה 1450 קבוע; אין הנחה במשחק מת (d=1). כל משחקי אותו יום משתמשים בדירוגי הבוקר ורק אחר כך מעדכנים.</p><p><b>הסבר כדורגלני:</b> ניצחון בהפרש גדול משפיע יותר. בקיץ דירוג 1600 חוזר ל־1567; קבוצה שחוזרת אחרי ירידה מתחילה כעולה, בלי לשחזר את הדירוג הישן שלה.</p></div>

In [ ]:
display(b_table(multiplier_example))

<div dir="rtl"><h2>B1.2 — מדירוג אחד לשלוש הסתברויות</h2>
<p><b>מה התא עושה:</b> קורא את מקדמי ההמרה הסדורה שנלמדו בכל אחד מ־15 קיפולי האימון.</p><p><b>מה רואים:</b> β מתאר את רגישות ההסתברויות לפער Elo; שני הספים θ₁ ו־θ₂ יוצרים אזור של תיקו.</p>
<pre dir="ltr">x = (R_home − R_away) / 400; σ(z) = 1 / (1 + exp(−z))
P(A) = σ(θ₁ − βx)
P(D) = σ(θ₂ − βx) − σ(θ₁ − βx)
P(H) = 1 − σ(θ₂ − βx)</pre><p><b>הסבר נתונים:</b> בכל קיפול המקדמים נאמדים מעונות האימון בלבד. החוץ הוא הקטגוריה הנמוכה, אחריו תיקו, ואז הבית; הפלט מוצג בסדר בית/תיקו/חוץ.</p><p><b>הסבר כדורגלני:</b> פער Elo גדול יותר מעלה את סיכוי הבית. גם בפער אפס סיכויי הבית והחוץ שונים, כי הביתיות נלמדת בספים.</p></div>

In [ ]:
coefficient_table = elo_review.tables["proposal_converter_effect"].copy()
coefficient_table[WORDS] = (
    "תצורת E0050: פער הדירוג מחולק ב־400; המרה סדורה נלמדת רק מעונות האימון של הקיפול"
)
display(
    b_table(
        coefficient_table,
        columns=["season", "fold", "beta", "theta1", "theta2", "home_probability_change_pp", WORDS],
    )
)

<div dir="rtl"><h2>B1.2א — חיפה 1540 מול עולה 1450: דוגמת הסתברויות</h2>
<p><b>מה התא עושה:</b> מציב פערי דירוג בנוסחה, עם מקדמי קיפול 2023/2024 השמורים.</p><p><b>מה רואים:</b> זו דוגמה היפותטית, לא טענה שכך דורגו הקבוצות ביום מסוים: פער 90 נותן כ־58% / 25% / 17%.</p>
<p><b>הסבר נתונים:</b> ההפרש בין שורת אפס לשורת 100 מודד שינוי בנקודות אחוז, לא תוספת יחסית של אחוזים. בדוגמה האחרונה הוא כ־21.2 נקודות אחוז; על הקיפולים התיאור הוא כ־20.</p><p><b>הסבר כדורגלני:</b> לדירוג החזק יש יתרון, אבל גם לעולה נשאר סיכוי לנצח או לסיים בתיקו.</p></div>

In [ ]:
display(b_table(conversion_examples(elo_review), percent=("בית", "תיקו", "חוץ")))

<div dir="rtl"><h2>B1.3 — השופט: תמיד מאמנים בעבר ובודקים בעונה הבאה</h2>
<p><b>מה התא עושה:</b> קורא את גבולות האימון והוולידציה מהקיפולים השמורים.</p><p><b>מה רואים:</b> 15 קיפולים מתרחבים: תחילה 2006/2007–2008/2009 לאימון, 2009/2010 לוולידציה; לבסוף 2023/2024 לוולידציה.</p>
<p><b>הסבר נתונים:</b> 3721 משחקי ולידציה. הבחירה נעשית לפי log loss משוקלל במספר המשחקים; RPS הוא מדד משני. חמש האחרונות מדווחות גם בנפרד לפי ה־ת6.</p><p><b>הסבר כדורגלני:</b> כך השיטה נשפטת בכל פעם על עונה שעדיין לא נראתה באימון של אותו קיפול.</p></div>

In [ ]:
display(b_table(fold_plan(elo_review)))

<div dir="rtl"><h2>B1.3א — משחק אחד: מה log loss ו־RPS מודדים</h2>
<p><b>מה התא עושה:</b> מחשב ידנית את שני המדדים לתחזית קבועה 58% בית, 25% תיקו, 17% חוץ.</p><p><b>מה רואים:</b> שלוש שורות הן שלוש תוצאות אפשריות לאותו משחק, ולא שלושה משחקים שנצפו.</p>
<pre dir="ltr">LL = −ln(p_observed)
RPS = ((p_H − 1[y=H])² + (p_H+p_D − 1[y∈{H,D}])²) / 2</pre><p><b>הסבר נתונים:</b> log loss הוא מינוס הלוגריתם של ההסתברות לתוצאה שקרתה. RPS הוא ממוצע ריבועי השגיאה בשני הגבולות המצטברים, בסדר בית/תיקו/חוץ. בשניהם נמוך עדיף.</p><p><b>הסבר כדורגלני:</b> תחזית שמותירה סיכוי סביר להפתעה נענשת פחות אם ההפתעה אכן מתרחשת.</p></div>

In [ ]:
display(b_table(score_example()))

<div dir="rtl"><h2>B1.3ב — סרגל איכות: מה אומר log loss של 1.02</h2>
<p><b>מה התא עושה:</b> משווה ייחוס אחיד, B0, E0050 והסף שנקבע מראש.</p><p><b>מה רואים:</b> לכל שורה מוצג גם e בחזקת מינוס log loss: ממוצע גאומטרי של ההסתברות לתוצאה שקרתה.</p>
<p><b>הסבר נתונים:</b> זה ממוצע גאומטרי, ולא ממוצע רגיל. הוא נמוך מהממוצע הרגיל (ב־E0050: 36.0% מול 38.8%), כי כמו log loss הוא מעניש חזק הסתברות נמוכה שניתנה להפתעה. סף 0.98 הוא יעד, לא תוצאת ניסוי.</p><p><b>הסבר כדורגלני:</b> E0050 מבחינה בין משחקים לפי חוזק עבר ומשפרת את הייחוס שאינו יודע מי משחקת.</p></div>

In [ ]:
display(b_table(score_ruler(elo_review), percent=("ממוצע גאומטרי של ההסתברות לתוצאה שקרתה",)))

<div dir="rtl"><h2>B1.4 — מה בדיוק נבדק ב־EXP-001</h2>
<p><b>מה התא עושה:</b> מציג את הרשת שנרצה בפועל, מתוך טבלת הפרמטרים השמורה.</p><p><b>מה רואים:</b> 1280 צירופים בשלב הראשון; 40 בשלב משקלי הזמן, על 10 בסיסי Elo. אחרי איחוד כפילויות: 1310 צירופים שונים.</p>
<p>ניסיון 1 נעצר בבעיית precision loss; ב־B3a-fix אושרה קבלה מותנית רק עם גרדיאנט מנורמל מרבי ≤10⁻⁶, וניסיון 2 רץ מחדש בשלמותו.</p><p><b>הסבר נתונים:</b> כמות הצירופים נובעת מ־4×4×4×2×2×5. שלב משקלי הזמן אינו חיפוש חוזר על כל הרשת. K=10 בקצה הרשת לא הורחב, לפי החלטת עמית; המיטוב מוגבל לרשת שאושרה.</p><p><b>הסבר כדורגלני:</b> בודקים כמה מהר להגיב לתוצאות, מה לשכוח בקיץ ואיך להתחיל עולות; אין נתוני שווי, שחקנים או שופטים.</p></div>

In [ ]:
display(b_table(grid_overview(elo_review)))

<div dir="rtl"><h2>B1.4א — עשר התצורות המובילות</h2>
<p><b>מה התא עושה:</b> קורא את עשר השורות הראשונות של הדירוג המשולב השמור.</p><p><b>מה רואים:</b> הפרשים קטנים מאוד בין הציונים. לכל תצורה מופיע הסבר מילולי קצר, לצד המזהה.</p>
<p><b>הסבר נתונים:</b> E1283 מובילה בממוצע על 15 קיפולים; E0050 שלישית. דירוג מספרי בלבד אינו מספיק לבחירה כשיש תצורות קרובות לפי כלל שגיאת תקן אחת.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית להפרש זעיר בציון בפני עצמו; אין להסיק ממנו שמשקל זמן מסוים הוא חוק של הליגה.</p><p><b>משותף לכל העשר:</b> K=10, נסיגה בקיץ של 33%, מכפיל הפרש שערים ואתחול עולה ב־1450. עמודת ״מה מייחד אותה״ מציגה רק את ההבדלים; עמודת המילים מסבירה את משמעותם. בסיס ההשוואה הוא יתרון בית 30 בעדכון, ללא משקל זמן וללא הנחה במשחק מת. דירוג חמש האחרונות מפורט גם בסעיף ה־ת6.</p></div>

In [ ]:
display(
    b_table(
        top_configuration_contrasts(elo_review),
        columns=[
            "label",
            "rank_all",
            "log_loss",
            "rps",
            "מה מייחד אותה",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B1.4ב — אפקט שולי ברשת הגסה: תיאור, לא סיבתיות</h2>
<p><b>מה התא עושה:</b> מציג את ממוצע log loss בכל ערך פרמטר, על יתר צירופי הרשת.</p><p><b>מה רואים:</b> שישה פנלים, בסדר ערכי הרשת, עם אותו טווח בציר Y כדי להשוות את גודל הפערים. הריחוף כולל את מספר הצירופים ואת הציון הטוב ביותר באותה קבוצה.</p>
<p><b>הסבר נתונים:</b> זה ממוצע על תצורות, לא אומדן סיבתי. ערך עם הממוצע הנמוך ביותר אינו בהכרח הערך של הצירוף המוביל, כי יש אינטראקציות עם פרמטרים אחרים.</p><p><b>הסבר כדורגלני:</b> למשל, תגובה מהירה מדי למשחקים יכולה להכניס רעש; הגרף מתאר את הרשת שנבדקה ואינו מוכיח מה גורם להצלחה על המגרש.</p><p>בממוצע על כל הצירופים K=20 טוב מעט, אבל הצירוף הטוב ביותר יושב על K=10. הגרף מראה ממוצע, לא את המיטב, ולכן אין סתירה.</p></div>

In [ ]:
b1_figures["marginals"] = marginal_chart(elo_review.tables["stage1_marginals"])
b1_figures["marginals"].show()

<div dir="rtl"><h2>B1.4ג — משקלי זמן על אותם עשרה בסיסי Elo</h2>
<p><b>מה התא עושה:</b> משווה ממוצע ציונים לפי זמן המחצית בשלב השני בלבד.</p><p><b>מה רואים:</b> עשרה צירופי בסיס בכל נקודה: ללא דעיכה או מחצית משקל בכל 1, 2 או 4 עונות.</p>
<p><b>הסבר נתונים:</b> גם כאן הקשר תיאורי ולא סיבתי. שינוי משקל הזמן חל בהתאמת שכבת ההמרה; הוא אינו משנה את סדר עדכוני Elo או מחשב Elo מחדש.</p><p><b>הסבר כדורגלני:</b> כך בודקים אם הקשר בין פער דירוג לתוצאה השתנה לאורך השנים, מעל דירוג Elo שכבר קיים.</p></div>

In [ ]:
b1_figures["time_weights"] = time_weight_chart(elo_review.tables["stage2_table"])
b1_figures["time_weights"].show()

<div dir="rtl"><h2>B1.4ד — הקרובות והבחירה ב־E0050</h2>
<p><b>מה התא עושה:</b> קורא את 13 התצורות שסומנו קרובות ומציג את הפרש הציון ואת שגיאת התקן המזווגת.</p><p><b>מה רואים:</b> הבדיקה מזווגת על אותם משחקים בכל קיפול. קרובה אם הפער הממוצע המשוקלל אינו גדול מ־SE אחד; SE הוא סטיית התקן הלא משוקללת של 15 הפרשי הקיפולים חלקי √15.</p>
<p><b>הסבר נתונים:</b> הבחירה המאושרת היא E0050: בין הקרובות יש לה מספר התוספות המינימלי (אחת: מכפיל הפרש), ובתיקו בפשטות בוחרים log loss נמוך. אין משקל זמן, אתחול עולה דינמי או הנחה במשחק מת.</p><p><b>הסבר כדורגלני:</b> נשמר זיכרון יציב של כוח הקבוצה; ניצחונות בהפרש גדול מקבלים משקל נוסף, בלי להוסיף מנגנונים שלא נדרשו במדגם.</p></div>

In [ ]:
display(
    b_table(
        configurations(elo_review, "close_configurations"),
        columns=["label", "log_loss", "paired_mean_difference", "paired_se", "additions", WORDS],
    )
)

<div dir="rtl"><h2>B1.4ה — E0050 מול B0 בכל עונת ולידציה</h2>
<p><b>מה התא עושה:</b> מציב את ציוני שני הייחוסים על אותו ציר עונות.</p><p><b>מה רואים:</b> B0 משתמש רק בשכיחויות תוצאות האימון. E0050 משתמשת בפער Elo ובהמרה הסדורה; כל נקודה היא עונת ולידציה אחת.</p>
<p><b>הסבר נתונים:</b> הקו מאפשר לראות יציבות ופערים בין עונות; הסיכום הכולל משוקלל במספר המשחקים ואינו ממוצע פשוט של הנקודות בגרף.</p><p><b>הסבר כדורגלני:</b> חוזק הקבוצות מבדיל בין משחק של פייבוריטית למשחק שקול, גם כששכיחות התוצאות הכללית דומה.</p></div>

In [ ]:
b1_figures["season_scores"] = season_score_chart(season_scores(elo_review))
b1_figures["season_scores"].show()

<div dir="rtl"><h2>B1.4ו — ה־ת6: חמש האחרונות אינן בוחרות אותה מובילה</h2>
<p><b>מה התא עושה:</b> מציג את מובילת 15 הקיפולים, מובילת חמש האחרונות ו־E0050.</p><p><b>מה רואים:</b> E1283 ראשונה על כל הקיפולים; E1286 ראשונה בחמש האחרונות. E0050 שלישית בכלל ו־73 בחמש האחרונות.</p>
<p><b>הסבר נתונים:</b> דגל אי־ההסכמה דולק. חמש האחרונות הן דיווח יציבות, בלי סימון קרובות ובלי בחירה מחדש. עמית אישר E0050 לפי כלל הפשטות על 15 הקיפולים.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית ישירה להחלפת המובילה: הטבלה אינה מזהה מה השתנה בליגה.</p></div>

In [ ]:
display(
    b_table(
        selection_comparison(elo_review),
        columns=[
            "label",
            "rank_all",
            "log_loss",
            "rps",
            "rank_last5",
            "last5_log_loss",
            "additions",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B1.4ז — LOSO: רטרוספקטיבי, לא לבחירה</h2>
<p><b>מה התא עושה:</b> קורא את בדיקת היציבות של עשרת מובילי הרשת הגסה, ללא משקלי זמן.</p><p><b>מה רואים:</b> הכותרת היא אזהרה מהותית: באימון ההמרה נכללות גם עונות מאוחרות מעונת הבדיקה. זו אינה תחזית כרונולוגית.</p>
<p style="font-size:1.4em;color:#9c2c22"><b>⚠️ רטרוספקטיבי, לא לבחירה — ללא משקלי זמן</b></p><p><b>הסבר נתונים:</b> LOSO אינו בוחר או מחליף את walk-forward. הוא מכסה רק את בסיסי Elo ללא משקל זמן; אין להשוות את ציונו כאילו נמדד על אותם קיפולים ומשחקים. דירוג walk-forward כאן הוא ברשת הגסה בלבד: E0050 ראשונה בה, לעומת שלישית בדירוג המשולב עם משקלי הזמן.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לציון LOSO בפני עצמו; הוא אבחון רגישות של ההמרה לעונת האימון.</p></div>

In [ ]:
display(
    b_table(
        configurations(elo_review, "loso_vs_walk_forward"),
        columns=[
            "label",
            "rank_loso",
            "loso_log_loss",
            "rank_walk_forward",
            "walk_forward_log_loss",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B1.5 — כיול: האם הסתברות חזויה מתאימה לשכיחות בפועל</h2>
<p><b>מה התא עושה:</b> מקבץ את תחזיות E0050 השמורות בעשרה בינים שווי רוחב, בנפרד לבית, לתיקו ולחוץ.</p><p><b>מה רואים:</b> הקו המקווקו הוא כיול מושלם. כל נקודה היא בין לא ריק; בריחוף רואים גם כמה משחקים יש בו.</p>
<p><b>הסבר נתונים:</b> מרחק מהאלכסון מצביע על אי־התאמה מקומית. בינים בקצוות עם מעט משחקים רועשים יותר; הקווים בין הנקודות הם עזר תצוגה ולא התאמה חדשה.</p><p><b>הסבר כדורגלני:</b> כשכתוב 30% לתיקו, רצוי שכשלושה מתוך עשרה משחקים דומים אכן יסתיימו בתיקו; אין צורך שתיקו יהיה התוצאה הסבירה ביותר.</p><p><b>גודל הנקודה מייצג את מספר המשחקים בבין</b>, והמספר המדויק בריחוף. נקודה חלולה מסמנת פחות מ־30 משחקים. נקודה קיצונית בבין קטן, כמו 100% בחוץ או 0% בתיקו, רועשת ואינה מעידה כשלעצמה על חוסר כיול.</p></div>

In [ ]:
b1_figures["calibration"] = calibration_chart(calibration_bins(elo_review.predictions))
b1_figures["calibration"].show()

<div dir="rtl"><h2>B1.5א — עד כ־31% לתיקו: ההסתברות נשארת חיובית</h2>
<p><b>מה התא עושה:</b> מציג את התפלגות הסתברויות התיקו ומחשב סיכום מתוך התחזיות השמורות.</p><p><b>מה רואים:</b> הסתברות התיקו המרבית כ־30.9%; הממוצע כ־27.1%, מול כ־26.8% תיקו שנצפה בוולידציה.</p>
<p><b>הסבר נתונים:</b> ממוצע סביר אינו מוכיח כיול בכל רמת סיכון; לכן מוצג גם גרף הבינים. אין כאן מודל נוסף או התאמה לתוצאות הוולידציה.</p><p><b>הסבר כדורגלני:</b> גם במשחק שקול, סיכוי התיקו נמוך מסיכוי ניצחון של אחת הקבוצות — ועדיין תיקו צפוי להתרחש בהרבה משחקים.</p></div>

In [ ]:
b1_figures["draw_distribution"] = draw_histogram(elo_review.predictions)
b1_figures["draw_distribution"].show()
diagnostic_summary, confusion = prediction_diagnostics(elo_review.predictions)
display(b_table(diagnostic_summary, percent=("accuracy", "max_draw", "observed_draw", "mean_draw")))

<div dir="rtl"><h2>B1.5ב — 47.8% דיוק, ואפס תחזיות תיקו ב־argmax</h2>
<p><b>מה התא עושה:</b> מציג מטריצת בלבול בין התוצאה בפועל לקטגוריה בעלת ההסתברות המרבית.</p><p><b>מה רואים:</b> השורות הן בפועל והעמודות הן argmax. עמודת התיקו כולה אפס; אין פירוש הדבר שהשיטה נותנת הסתברות אפס לתיקו.</p>
<p><b>הסבר נתונים:</b> log loss שופט את כל ההתפלגות ולא רק את הקטגוריה המובילה. היעדר תיקו ב־argmax כשלעצמו אינו פגם במדד; הסתברות נמוכה מדי לתיקו כן תיענש. דיוק אינו השופט לבחירה.</p><p><b>הסבר כדורגלני:</b> בסימולציה מגרילים מכל ההסתברויות, ולכן יתקבלו גם תיקו והפתעות. לא בוחרים אוטומטית את התוצאה בעלת ההסתברות הגבוהה ביותר בכל משחק.</p></div>

In [ ]:
b1_figures["confusion"] = confusion_chart(confusion)
b1_figures["confusion"].show()

<div dir="rtl"><h2>B1.6 — כל הליגה: הדירוג בסיום מול הטבלה בסיום</h2>
<p><b>מה התא עושה:</b> מציג Spearman ומדדי התאמה לכל עונת פיתוח, ואת הסיכום על כל 252 שורות קבוצה־עונה.</p><p><b>מה רואים:</b> דירוג Elo אחרי המשחק האחרון מול מיקום ספורטיבי סופי: 38.5% זהים, 74.2% עד מקום אחד, 89.7% עד שניים.</p>
<p><b>הסבר נתונים:</b> זו בדיקה בדיעבד באותה נקודת סיום, לא חיזוי מקום סופי בתחילת העונה ולא ציון מחוץ למדגם. המיקום ספורטיבי, בלי הורדות נקודות; Spearman מחושב בנפרד בכל עונה.</p><p><b>הסבר כדורגלני:</b> Elo מזהה בדרך כלל את סדרי הכוח אך אינו מחקה במדויק את צבירת הנקודות בעונה. ביתיות, חוזק היריבה וזיכרון העבר משפיעים עליו.</p></div>

In [ ]:
display(
    b_table(
        elo_review.tables["proposal_sporting_alignment"],
        columns=[
            "season",
            "teams",
            "spearman",
            "exact_percent",
            "within_one_percent",
            "within_two_percent",
        ],
    )
)
display(
    b_table(
        alignment_summary(elo_review.tables["proposal_1_all_teams"]),
        percent=("מקום זהה", "עד מקום אחד", "עד שני מקומות"),
    )
)

<div dir="rtl"><h2>B1.6א — פיזור מיקומים: האלכסון הוא התאמה מלאה</h2>
<p><b>מה התא עושה:</b> מציג את כל 252 הקבוצות־עונות על צירי מיקום ספורטיבי ומיקום לפי Elo.</p><p><b>מה רואים:</b> נקודות על האלכסון הן אותו מיקום. ייתכן חפיפה בין עונות; בריחוף יש קבוצה, עונה, דירוג Elo ופער מיקומים.</p>
<p><b>הסבר נתונים:</b> הדירוג נעשה בתוך כל עונה. העונות עם 12, 14 ו־16 קבוצות נשמרות; אין לערבב דירוגים מוחלטים בין עונות.</p><p><b>הסבר כדורגלני:</b> סטייה מהאלכסון מזכירה שכוח לפי תוצאות עבר וטבלת נקודות עונתית הם שני תיאורים שונים של אותה קבוצה.</p></div>

In [ ]:
b1_figures["league_alignment"] = alignment_chart(elo_review.tables["proposal_1_all_teams"])
b1_figures["league_alignment"].show()

<div dir="rtl"><h2>B1.6ב — פער של ארבעה מקומות ומעלה: מה Elo זוכר</h2>
<p><b>מה התא עושה:</b> מציג את כל 14 המקרים שנשמרו, עם פירוש כיוון הפער.</p><p><b>מה רואים:</b> פער חיובי פירושו שהקבוצה סיימה במקום ספורטיבי טוב מהדירוג שנתן לה Elo; פער שלילי הוא הכיוון ההפוך.</p>
<p><b>הסבר נתונים:</b> זה פירוש תיאורי של מנגנון הזיכרון, לא זיהוי סיבתי של מקור ההפתעה. אין כאן מידע על סגל, מאמן או תקציב.</p><p><b>הסבר כדורגלני:</b> Elo סקפטי כלפי הפתעות ונאמן למוניטין שנוצר מתוצאות עבר, לא לשם המועדון. למשל חדרה ב־2018/2019 מסיימת שישית אך מדורגת 12 לפי Elo, אחרי פתיחה כעולה ב־1450; בני יהודה ב־2013/2014 מסיימת 14 אך Elo עדיין מדרג אותה שישית.</p></div>

In [ ]:
display(
    b_table(
        gap_explanations(elo_review),
        columns=[
            "season",
            "team_name",
            "elo_start",
            "elo_end",
            "position",
            "elo_rank",
            "position_difference",
            "הסבר כדורגלני",
        ],
    )
)

<div dir="rtl"><h2>B1 — מספרי בקרה ושמירת הגרפים</h2>
<p><b>מה התא עושה:</b> מציג מספרי בקרה מדויקים ושומר רק את גרפי B1 החדשים ל־HTML ול־PNG.</p><p><b>מה רואים:</b> E0050: ‏1.020424 ו־0.208413; B0: ‏1.084974. שלוש התאמות המקום מוצגות באחוזים.</p>
<p><b>הסבר נתונים:</b> המספרים מחושבים מתוצרי הניסוי המאומתים. בשלב זה לא נבנה חלק B2, ולא נבחר או אומן רכיב חדש.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לפעולת השמירה. ההמשך המאושר בנפרד יסביר את הכושר ואת ההתקפה וההגנה מעל הבסיס של Elo.</p></div>

In [ ]:
display(b_table(control_numbers(elo_review)))
b1_figure_dir = Path("reports/figures/prematch_b1")
b1_figure_dir.mkdir(parents=True, exist_ok=True)
for name, figure in b1_figures.items():
    figure.write_html(b1_figure_dir / f"{name}.html", include_plotlyjs=True)
    figure.write_image(
        b1_figure_dir / f"{name}.png", width=1700, height=figure.layout.height or 600, scale=1.5
    )
display(
    HTML(
        f'<div dir="rtl">נשמרו {len(b1_figures)} גרפים. זמן חלק B1: '
        f"{time.perf_counter() - b1_started:.2f} שניות.</div>"
    )
)

<div dir="rtl"><h2>B2 — כושר, התקפה והגנה: קוראים את EXP-002</h2><p><b>מה התא עושה:</b> מאמת את ה־checksums של הניסוי ושל form_v1, וטוען רק פיצ׳רים, מקדמים ותחזיות שמורים.</p><p><b>מה רואים:</b> אותן 18 עונות פיתוח: 2006/2007 עד 2023/2024. אין טעינה מחודשת של נתוני משחקים נקיים בחלק B2, ואין אימון.</p><p><b>הסבר נתונים:</b> כל הטבלאות נגזרות מקבצים שנשמרו ב־B4a וב־B4c. נתיב תחזיות EXP-002 מרוכז בקבוע אחד במודול, לקראת המעבר ב־B6. חריגת האימות ההנדסי של A נשארת כפי שאושרה.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לבדיקת ה־hash; היא מוודאת שההסבר נשען על אותו ניסוי שאושר.</p></div>

In [ ]:
from ligat.analysis.prematch_form_review import (
    SELECTED_P,
    candidate_table,
    coefficient_summary,
    decision_table,
    displayed_controls,
    draw_diagnostics,
    ewma_illustrations,
    haifa_season,
    load_form_review,
    opening_coefficients,
    points_coefficients,
    points_leaders,
    poisson_coefficients,
    reported_summer_check,
    route_overview,
    route_references,
)
from ligat.viz.prematch_form import (
    addition_chart,
    form_table,
    haifa_chart,
    recent_scatter,
    rho_chart,
)

b2_started = time.perf_counter()
form_review = load_form_review(ROOT)
b2_figures = {}
display(
    HTML(
        f'<div dir="rtl">הקבצים אומתו: {len(form_review.form):,} שורות × '
        f"{len(form_review.form.columns)} פיצ׳רים; {len(form_review.predictions):,} "
        "תחזיות ולידציה לתצורה שנבחרה.</div>"
    )
)

<div dir="rtl"><h2>B2.1 — ערכי פתיחה: כמה מהכוח נשמר בין עונות</h2><p><b>מה התא עושה:</b> מציג את תשעת ערכי ρ לכל עונה מתוך המטא־דאטה של form_v1.</p><p><b>מה רואים:</b> שלושה מדדים — הבקעה, ספיגה ונקודות — וכל אחד לכל המשחקים, לבית ולחוץ בנפרד. לקראת 2023/2024 ערכי all הם 0.677, 0.510 ו־0.762.</p><pre dir="ltr">prior = L_previous + ρ × (team_previous − L_previous)</pre><p><b>הסבר נתונים:</b> ρ נלמד רק מזוגות עונות קודמים לקראת העונה הנחזית, ב־OLS דרך הראשית על פערים מממוצע הליגה, עם חסימה ל־<bdi dir="ltr">[0,1]</bdi>. סימון חלול הוא גיבוי 0.5 בתחילת הדאטה; בריחוף רואים את גודל המדגם. אלו אינם המקדמים של בדיקת הקיץ שתופיע בהמשך.</p><p><b>הסבר כדורגלני:</b> קבוצה שהבקיעה מעל הממוצע מתחילה עם חלק מהיתרון שלה, ולא חוזרת מיד לקבוצה ממוצעת. לדוגמה, עם ρ התקפה 0.677, פער עבר של חצי שער מצטמצם לכ־0.34 שער בפתיחה.</p></div>

In [ ]:
rho_values = opening_coefficients(form_review)
b2_figures["rho"] = rho_chart(rho_values)
b2_figures["rho"].show()

<div dir="rtl"><h2>B2.1א — עולות: מתחילות ממידע על עולות קודמות</h2><p><b>מה התא עושה:</b> מציג את ערכי הליגה, ρ וערכי העולה שהיו זמינים לקראת 2023/2024.</p><p><b>מה רואים:</b> כל תשעת המדדים והגרסאות. ערך העולה נלמד מעונות שכבר הסתיימו; אין שימוש בעונה שבה היא עולה.</p><p><b>הסבר נתונים:</b> ממוצע הליגה בטבלה הוא של 2022/2023; ממוצע העולות מאגד את העונות הראשונות של עולות קודמות. לכן אלה אינם שני מדגמים מאותה תקופה. כשיש פחות מארבע עולות קודמות, משתמשים בשלוש האחרונות בטבלה הספורטיבית של העונה הקודמת. בעונת הדאטה הראשונה הפתיחה ניטרלית, כמוגדר ב־T-010.</p><p><b>הסבר כדורגלני:</b> בבדיקה התיאורית הכללית של Claude שצוטטה ב־T-010, עולה הבקיעה כ־1.03 מול 1.22 בליגה, ספגה כ־1.45 מול 1.23 וצברה כ־1.09 נקודות מול 1.36. כלומר כ־0.2 ספיגות נוספות למשחק. זה תיאור כללי של העולות; הוא אינו ההפרש בין שתי עמודות הפתיחה של 2023/2024 בטבלה.</p></div>

In [ ]:
display(
    form_table(
        rho_values.loc[rho_values.season_id.eq(25)],
        columns=[
            "season",
            "metric",
            "stratum",
            "rho",
            "league_mean",
            "promoted_value",
            "promoted_observations",
            "promoted_fallback",
        ],
    )
)

<div dir="rtl"><h2>B2.1ב — הקיץ: הכוח שורד, המומנטום כמעט לא מוסיף</h2><p><b>מה התא עושה:</b> מציג את הבדיקה המדווחת של Claude, עם הגדרת המשתנים שאישר עמית; אינו מתאים רגרסיה מחדש.</p><p><b>מה רואים:</b> 203 קבוצות־עונה שנשארו בליגה, בעונות <bdi dir="ltr">2007/2008–2023/2024</bdi>. המשתנה הנחזה: נקודות ספורטיביות למשחק בעשרת המשחקים הראשונים של העונה.</p><p><b>הסבר נתונים:</b> אותה רגרסיה ליניארית עם חותך וללא משקלים משתמשת בשני מנבאים: נקודות למשחק בכל העונה הקודמת ובעשרת האחרונים שלה, לפי תאריך וכולל פלייאוף. עשרת האחרונים כלולים גם בעונה המלאה: 0.11 מודד את התוספת שלהם אחרי שהעונה כולה כבר נלקחה בחשבון. המתאמים הפשוטים 0.54 ו־0.37 הם מדד אחר. המקור הוא T-010 §8.4 והבהרת עמית במשימה זו; אין כאן ציון חיזוי חדש.</p><p><b>הסבר כדורגלני:</b> קצב הנקודות לאורך עונה מלאה מספר על חוזק הקבוצה יותר מסיום העונה בלבד. לכן החלונות אינם חוצים את הקיץ: מתחילים מערך פתיחה שמתאר כוח, ואז לומדים מחדש את העונה הנוכחית.</p></div>

In [ ]:
display(form_table(reported_summer_check()))

<div dir="rtl"><h2>B2.2 — EWMA: זיכרון שדועך בהדרגה</h2><p><b>מה התא עושה:</b> מחשב משקלים מתמטיים בלבד לשלושה זמני מחצית ולחלון 5.</p><p><b>מה רואים:</b> משחק 1 הוא המשחק האחרון שהסתיים לפני יום המשחק; משחק 5 הוא החמישי לאחור. השורה האחרונה מרכזת את המשקל של חמשת האחרונים יחד.</p><pre dir="ltr">α = 1 − 0.5 ** (1/h)
value_new = α × result + (1−α) × value_old
weight(age=j) = α × (1−α) ** (j−1)</pre><p><b>הסבר נתונים:</b> EWMA הוא ממוצע מתעדכן: כל תוצאה חדשה מקבלת α, והעבר מוכפל ב־1−α. זמן מחצית 5 אינו חלון 5: חמשת האחרונים נותנים בו רק 50% מהמשקל, לעומת 100% בחלון מלא של 5. בגרסת venue סופרים משחקים באותו מגרש בלבד.</p><p><b>הסבר כדורגלני:</b> ניצחון או הפסד משפיעים מיד, ואז חשיבותם דועכת בלי להיעלם בבת אחת. בגרסת בית, משחק חוץ אינו משנה את הממוצע של משחקי הבית.</p></div>

In [ ]:
weight_table, prior_weight_table = ewma_illustrations()
display(form_table(weight_table, percent=("EWMA 5", "EWMA 10", "EWMA 20", "חלון 5")))

<div dir="rtl"><h2>B2.2א — כמה נשאר מהדעה של תחילת העונה</h2><p><b>מה התא עושה:</b> מחשב את חלקו של ערך הפתיחה אחרי 5, 10, 20 ו־30 משחקים.</p><p><b>מה רואים:</b> ב־EWMA 5 נשאר חצי מהפתיחה אחרי חמישה משחקים ורבע אחרי עשרה. בחלון 5 הפתיחה נעלמת לחלוטין אחרי חמישה.</p><pre dir="ltr">prior_mass(k) = (1−α) ** k = 0.5 ** (k/h)</pre><p><b>הסבר נתונים:</b> המשקל שנשאר לפתיחה אחרי k תוצאות הוא ‎(1−α)^k. המונה הוא מספר המשחקים באותה גרסה, לא מחזור הליגה; כל המועמדים מאופסים בקיץ לפי החוזה.</p><p><b>הסבר כדורגלני:</b> בתחילת העונה עדיין יודעים מעט על הקבוצה החדשה. כשמצטברים משחקים, התוצאות העדכניות מחליפות בהדרגה את ההערכה שהגיעה מהעונה הקודמת.</p></div>

In [ ]:
display(form_table(prior_weight_table, percent=tuple(f"אחרי {k} משחקים" for k in (5, 10, 20, 30))))

<div dir="rtl"><h2>B2.3 — מכבי חיפה 2023/2024: ההתקפה לפני כל משחק</h2><p><b>מה התא עושה:</b> מחבר את שורות form_v1 לדירוגי הזמן ולשערים בפועל שנשמרו בניסוי.</p><p><b>מה רואים:</b> EWMA 5 בבית ובחוץ כשני קווים, EWMA 20 על כל המשחקים כקו מקווקו, והתוצאות בפועל כסימני ×.</p><p><b>הסבר נתונים:</b> אלו פיצ׳רים של קצב הבקעה, לא λ של מודל השערים. קו בית מופיע רק במועדי משחקי הבית וקו חוץ רק בחוץ; אין השלמה או חישוב מחדש של ערכי venue בתאריכים אחרים. סימני השערים בפועל מוצגים להשוואה אחרי המשחק ואינם קלט לפיצ׳ר של אותו משחק.</p><p><b>הסבר כדורגלני:</b> ההפרדה מאפשרת לתאר קבוצה אחרת בבית ובחוץ. EWMA 20/all מגיב לאט יותר ומחליק את ההבדלים, כי הוא זוכר יותר משחקים ומערבב מגרשים.</p></div>

In [ ]:
haifa_example = haifa_season(form_review)
b2_figures["haifa_attack"] = haifa_chart(haifa_example, "gf")
b2_figures["haifa_attack"].show()

<div dir="rtl"><h2>B2.3א — מכבי חיפה: ההגנה בבית הגיעה לכ־0.41</h2><p><b>מה התא עושה:</b> מציג את אותה השוואה עבור שערי חובה.</p><p><b>מה רואים:</b> הספיגות בפועל הן נקודות, והקווים הם ההערכות שהיו זמינות לפני כל משחק. נמוך יותר הוא הגנה שסופגת פחות.</p><p><b>הסבר נתונים:</b> הערך הנמוך בבית הוא כ־0.406 לפני משחק, ולכן מעוגל ל־0.41. שינויי הקו נובעים מהמשחקים שכבר הסתיימו בלבד; זה אינו אומדן של השפעת אצטדיון או של סיבה אחרת.</p><p><b>הסבר כדורגלני:</b> במונחי הפיצ׳ר, חיפה מציגה ״מבצר בבית״: ממוצע ספיגות נמוך מאוד בגרסת הבית, שאינו נבלע בממוצע הכללי והאיטי.</p></div>

In [ ]:
b2_figures["haifa_defense"] = haifa_chart(haifa_example, "ga")
b2_figures["haifa_defense"].show()
display(
    HTML(
        f'<div dir="rtl">המינימום בפיצ׳ר הגנת הבית לפני משחק: '
        f"{haifa_example.loc[haifa_example.is_home, 'ga_venue_ewma5'].min():.6f} שערים למשחק.</div>"
    )
)

<div dir="rtl"><h2>B2.4 — שני מסלולים, שתי שאלות ושני סרגלים</h2><p><b>מה התא עושה:</b> מציג את מבנה EXP-002 ואת ציוני הייחוס של כל מסלול.</p><p><b>מה רואים:</b> P בודק התקפה והגנה לפי כמות השערים; O בודק אם קצב הנקודות מוסיף להסתברויות התוצאה מעבר ל־Elo.</p><p><b>הסבר נתונים:</b> ב־P מדד השיפוט הוא Poisson NLL לשורת קבוצה־משחק, כולל האיבר log(y!). ב־O זה log loss למשחק על בית/תיקו/חוץ. הערכים אינם באותן יחידות ואין להשוות ביניהם. שני המסלולים משתמשים באותם 15 קיפולי פיתוח; כאן רק קוראים את הציונים שנשמרו.</p><p><b>הסבר כדורגלני:</b> אפשר לשפר את ההערכה כמה שערים יובקעו בלי לשפר באותה מידה את תוצאת המשחק; כמות שערים ותוצאת בית/תיקו/חוץ הן שאלות שונות.</p></div>

In [ ]:
display(form_table(route_overview()))
form_references = route_references(form_review)
display(form_table(form_references))

<div dir="rtl"><h2>B2.4א — P: כל 22 המועמדים עם Elo</h2><p><b>מה התא עושה:</b> קורא את הדירוג השמור ומחשב הפרשים מזווגים מהציונים השמורים בכל קיפול.</p><p><b>מה רואים:</b> לכל מועמד: נוסחה במילים, ההבדל מהאחרים, NLL, תוספת מעל Elo ביחידות SE, מספר עונות עם שיפור ודירוג חמש האחרונות. קרבה נשמרת רק מול מוביל 15 הקיפולים.</p><pre dir="ltr">d_f = NLL(Elo reference, f) − NLL(candidate + Elo, f)
z = weighted_mean(d_f) / (std(d_f, ddof=1) / √15)</pre><p><b>הסבר נתונים:</b> לכל עונה d הוא NLL של ייחוס Elo פחות NLL של המועמד. הממוצע משוקלל במספר שורות הקבוצה; SE היא סטיית התקן הלא משוקללת של 15 ההפרשים חלקי √15. z הוא היחס ביניהם, לא p-value. המספרים משקפים פיתוח שבו נבדקו 22 אפשרויות, לא אישור על עונת מבחן חדשה.</p><p><b>הסבר כדורגלני:</b> החלונות בודקים אם כמה שהקבוצה מבקיעה וכמה שהיריבה סופגת נותנים חידוד מעל דירוג הכוח. בשני צדי ההשוואה יש Elo.</p></div>

In [ ]:
p_candidates = candidate_table(form_review, "P")
display(
    form_table(
        p_candidates,
        columns=[
            "method",
            "מה מייחד אותה",
            "poisson_nll",
            "z",
            "seasons_better",
            "last5_poisson_nll",
            "rank_last5",
            "within_one_se",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B2.4ב — רק מידע מתוך העונה מוסיף מעל Elo</h2><p><b>מה התא עושה:</b> מציג את יחס השיפור לשגיאת התקן לכל 22 המועמדים, בצבע לפי מקור המידע.</p><p><b>מה רואים:</b> כל 18 המועמדים שמשתמשים בתוצאות העונה נמצאים מעל שגיאת תקן אחת. ארבעת prev/hist אינם מוסיפים בממוצע מעל Elo.</p><p><b>הסבר נתונים:</b> הציר חיובי כשהמועמד מקטין NLL מעבר לייחוס Elo. זהו תיאור של ההפרשים בקיפולי הפיתוח, ולא בדיקת סיבתיות או z-test עם p-value. כלל הקרבה למוביל הוא מבחן אחר, שמופיע בטבלה.</p><p><b>הסבר כדורגלני:</b> Elo כבר זוכר היסטוריה ארוכה של תוצאות. תיאור ההבקעה והספיגה בעונה הנוכחית נותן מידע נוסף; חזרה על ממוצעי העונות הישנות אינה מוסיפה כאן.</p></div>

In [ ]:
b2_figures["addition_z"] = addition_chart(p_candidates)
b2_figures["addition_z"].show()

<div dir="rtl"><h2>B2.4ג — ה־ת6: 15 העונות מול חמש האחרונות</h2><p><b>מה התא עושה:</b> מציב את NLL של כל מועמד על שני צירים, עם ייחוס Elo בשניהם.</p><p><b>מה רואים:</b> ewma5/all מוביל לפי הכלל על 15 הקיפולים ומדורג 21 בחמש האחרונות; ewma5/venue מוביל בחמש האחרונות, ונבחר בידי עמית.</p><p><b>הסבר נתונים:</b> נמוך עדיף בשני הצירים. דגל ה־ת6 דולק משום שהדירוגים אינם מסכימים. בחמש האחרונות ההבדלים בין המועמדים קטנים מהרעש; אין כאן סימון קרבה חדש או הכרעה סטטיסטית לפי חמש עונות בלבד.</p><p><b>הסבר כדורגלני:</b> חוזק כללי והפרדה בין בית לחוץ נותנים תמונה שונה בתקופות שונות. הגרף מציג את אי־היציבות, ואינו מסביר לבדו מה השתנה בליגה.</p></div>

In [ ]:
b2_figures["recent_scatter"] = recent_scatter(p_candidates, form_references)
b2_figures["recent_scatter"].show()

<div dir="rtl"><h2>B2.4ד — הבחירה המאושרת: ewma5/venue</h2><p><b>מה התא עושה:</b> מעמיד את המוביל לפי כלל הפשטות מול התצורה שנבחרה, בלי לבחור מחדש.</p><p><b>מה רואים:</b> ewma5/venue: ‏NLL 1.409455 ותוספת של כ־2.95 שגיאות תקן מעל Elo. הוא אינו בין הקרובים למוביל על 15 הקיפולים.</p><p><b>הסבר נתונים:</b> זו חריגה מודעת מכלל הפשטות, שהכריע עמית אחרי QA: עדיפות לחיזוי העתיד ולעקביות התוספת. התצורה נבחרה למרות ש־ewma5/all מוביל בממוצע הארוך. הפערים הקטנים בחמש האחרונות נשארים הסתייגות מפורשת.</p><p><b>הסבר כדורגלני:</b> הבחירה שומרת קצב הבקעה וספיגה נפרד לבית ולחוץ, עם זיכרון קצר יחסית, כדי לייצג הבדלים עדכניים בין שתי סביבות המשחק.</p></div>

In [ ]:
display(
    form_table(
        p_candidates.loc[p_candidates.config_id.isin(["P_ewma5_all_elo", SELECTED_P])],
        columns=[
            "method",
            "rank_all",
            "poisson_nll",
            "z",
            "seasons_better",
            "rank_last5",
            "last5_poisson_nll",
            "within_one_se",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B2.4ה — מקדמי הנבחר בכל קיפול</h2><p><b>מה התא עושה:</b> קורא את a, d, b ו־γ מתוך fits.parquet של ewma5/venue.</p><p><b>מה רואים:</b> a מתייחס להתקפת הקבוצה, d להגנת היריבה, b לפער Elo ו־γ לביתיות. כל שורה נלמדה רק מעונות האימון של אותו קיפול.</p><pre dir="ltr">log λ_X = c + γ × home + a × log(gf_X+0.1)
          + d × log(ga_opponent+0.1) + b × Elo_difference_X/400</pre><p><b>הסבר נתונים:</b> בכל מועמדי P הוגדר מראש log(x+0.1), כדי לאפשר גם חלון עם אפס שערים. ההיסט אינו משנה את form_v1 ואינו פרמטר שנבחר בניסוי. המקדמים מתארים קשרים מותנים במודל השיפוט; הם אינם משקלים שמחליפים את הפיצ׳ר ב־M0.</p><p><b>הסבר כדורגלני:</b> התקפה טובה והגנה שסופגת הרבה אצל היריבה שתיהן מעלות את ציפיית השערים; Elo מוסיף את הפרש הכוחות מעבר להן.</p></div>

In [ ]:
selected_coefficients = poisson_coefficients(form_review)
display(form_table(selected_coefficients, columns=["season", "a", "d", "b", "gamma", WORDS]))

<div dir="rtl"><h2>B2.4ו — תרגום המקדמים לשערים ולהסתברות ניצחון</h2><p><b>מה התא עושה:</b> מציג חציון וטווח על 15 הקיפולים, עם נקודת הייחוס שנקבעה מראש.</p><p><b>מה רואים:</b> עלייה של gf מ־1.5 ל־1.65: בחציון כ־1.39% תוספת שערים וכ־0.50 נקודות אחוז לניצחון בית. עלייה מקבילה בספיגות היריבה נותנת כ־0.87% תוספת שערים.</p><pre dir="ltr">λ_after / λ_before = ((1.65+0.1)/(1.5+0.1)) ** a
ΔP(home win) = 100 × (P_after − P_before)  [percentage points]</pre><p><b>הסבר נתונים:</b> בדוגמה Elo=0; יתר משתני ההבקעה והספיגה בממוצע הליגה של עונות האימון של כל קיפול. ממוצע זה משוחזר מממוצעי העונות השמורים ומספר שורות Elo, בלי קריאת תוצאות נקיות. ההסתברויות מחושבות ברשת <bdi dir="ltr">0–10</bdi> המנורמלת הקיימת. אלה המחשות מותנות, לא השפעות סיבתיות.</p><p><b>הסבר כדורגלני:</b> בגרסת venue כל ממוצע נשען על חצי מהמשחקים ולכן רועש יותר; המודל נותן לו מקדם קטן יותר ומשאיר יותר משקל ל־Elo: b בחציון כ־1.15 מול כ־1.0 ב־all. למשל, שיפור של 10% בקצב ההבקעה אינו הופך במודל מיד לשיפור של 10% בשערים הצפויים.</p></div>

In [ ]:
display(form_table(coefficient_summary(selected_coefficients)))

<div dir="rtl"><h2>B2.5 — כושר בנקודות: 0 מתוך 22 מוסיפים בממוצע</h2><p><b>מה התא עושה:</b> מציג את ייחוס Elo, שלושת המובילים במסלול O ואת הצעת הפשטות.</p><p><b>מה רואים:</b> גם המוביל בין מועמדי הכושר גרוע מעט מ־E0050 בממוצע 15 הקיפולים. EWMA 20/all הוא הצעת הפשטות של O, ולא המוביל ולא פיצ׳ר שנבחר.</p><p><b>הסבר נתונים:</b> זו תוצאה על תוספת ליניארית של הפרש נקודות למשחק להמרה הסדורה של פער Elo. מתאם עם התוצאה לבדו אינו מספיק: כאן בודקים מה נשאר אחרי ש־Elo כבר במודל.</p><p><b>הסבר כדורגלני:</b> צבירת נקודות חוזרת במידה רבה על המידע מתוצאות ש־Elo כבר מסכם. בניסוי הזה היא לא סיפקה תוספת ממוצעת מעליו.</p></div>

In [ ]:
o_candidates = candidate_table(form_review, "O")
display(form_table(points_leaders(form_review)))
display(
    HTML(
        f'<div dir="rtl">מועמדים עם שיפור ממוצע מעל Elo: '
        f"{int(o_candidates.improvement.gt(0).sum())} מתוך {len(o_candidates)}.</div>"
    )
)

<div dir="rtl"><h2>B2.5א — מדוע בדוגמת הכושר מתקבל מקדם שלילי</h2><p><b>מה התא עושה:</b> קורא את שני המקדמים של הצעת הפשטות O: ‏ewma20/all, לכל קיפול.</p><p><b>מה רואים:</b> בקיפול הראשון β של Elo מגיע לכ־6.1, ולכושר מקדם שלילי. הטבלה מתייחסת לדוגמה זו במפורש, ולא לכל 22 המועמדים.</p><p><b>הסבר נתונים:</b> קולינאריות פירושה ששני משתנים חולקים מידע דומה. Elo וקצב הנקודות מסכמים שניהם תוצאות עבר, ולכן כל מקדם מתאר את המשתנה לאחר שליטה באחר. מקדמים גדולים או שליליים אינם פירוש עצמאי של איכות קבוצה.</p><p><b>הסבר כדורגלני:</b> השערה אפשרית: נקודות ש־Elo אינו מאשר הן מזל שחוזר לממוצע. <b>זו השערה שלא נבדקה ישירות</b>, ולא סיבה שהוכחה בטבלה. ההחלטה המאושרת: כושר בנקודות אינו נכנס ל־M0; נשאר 🔬 רק לבנצ׳מרק XGBoost וייצא אם לא יתרום.</p></div>

In [ ]:
display(form_table(points_coefficients(form_review)))

<div dir="rtl"><h2>B2.6 — התיקו ב־Poisson: חזוי מול בפועל</h2><p><b>מה התא עושה:</b> מחשב שיעורי בית/תיקו/חוץ מהתחזיות השמורות של P_ewma5_venue_elo.</p><p><b>מה רואים:</b> לתיקו: 25.49% חזוי מול 26.85% בפועל, על אותם 3721 משחקים מחוץ למדגם.</p><p><b>הסבר נתונים:</b> שיעור חזוי הוא ממוצע ההסתברויות, לא ספירת argmax. Poisson בלתי תלוי מניח, בהינתן λ, שאין תלות בין שערי הקבוצות. המודל יכול לתאר היטב ממוצעי שערים ובכל זאת לפספס חלק ממבנה התיקו.</p><p><b>הסבר כדורגלני:</b> משחק כדורגל מגיב למצב שלו: התנהגות הקבוצות כשהן שוות או בפיגור עשויה לשנות את קצב ההבקעה. ההסבר מתאים לממצא, אך אינו נבדק כאן סיבתית.</p></div>

In [ ]:
draw_rates, draw_strata, draw_scores, draw_totals = draw_diagnostics(form_review.predictions)
display(form_table(draw_rates, percent=("predicted", "observed")))

<div dir="rtl"><h2>B2.6א — באילו סוגי משחקים חסר תיקו</h2><p><b>מה התא עושה:</b> מחלק לשלישים לפי הערך המוחלט של ההפרש בין λ הבית ל־λ החוץ.</p><p><b>מה רואים:</b> שלוש קבוצות דומות בגודל: שקול, בינוני ולא שקול. בבינוני כ־26.8% תיקו חזוי מול כ־30.2% שנצפה.</p><p><b>הסבר נתונים:</b> החלוקה נעשית רק לפי ציפיות השערים לפני המשחק, ולא לפי התוצאה שהתגלתה אחריו. בטבלה מופיעים גבולות ההפרש ומספר המשחקים. שמות השלישים יחסיים למדגם, אינם ספי כדורגל שנקבעו למודל.</p><p><b>הסבר כדורגלני:</b> החסר אינו אחיד: במשחקים עם פער בינוני בכוח השערים מתקבלים יותר תיקו מכפי שההנחה הבלתי תלויה נותנת.</p></div>

In [ ]:
display(form_table(draw_strata, percent=("predicted", "observed")))

<div dir="rtl"><h2>B2.6ב — תיקו מדויק וממוצע שערים: בדיקות שונות</h2><p><b>מה התא עושה:</b> גוזר את הסתברות 0:0, 1:1, 2:2 ו־3:3 מה־λ השמור ומשווה לתוצאות בפועל.</p><p><b>מה רואים:</b> יחס גדול מ־1 פירושו שנצפה יותר מהצפוי. לצד זה: כ־2.47 שערים חזויים למשחק מול כ־2.49 בפועל.</p><p><b>הסבר נתונים:</b> התוצאות המדויקות משתמשות באותה רשת מנורמלת <bdi dir="ltr">0–10</bdi> ששימשה ל־H/D/A בניסוי. ממוצע סך השערים הוא λ הבית ועוד λ החוץ. התאמת הממוצע אינה מבטיחה התאמה של כל תוצאה מדויקת.</p><p><b>הסבר כדורגלני:</b> זה הגשר ל־M3 לפי ה־ס13: לבדוק תיקון לתלות בתוצאות נמוכות, למשל Dixon-Coles, או ניפוח תיקו. לא מוסיפים תיקון כזה עכשיו, כדי לשמור על הניסוי שנשפט.</p></div>

In [ ]:
display(form_table(draw_scores, percent=("predicted", "observed")))
display(form_table(draw_totals))

<div dir="rtl"><h2>B2.7 — סיכום פעימה B וגשר ל־C</h2><p><b>מה התא עושה:</b> מרכז את ההחלטות שכבר אושרו ואת שני סרגלי ההמשך.</p><p><b>מה רואים:</b> E0050, התקפה והגנה ewma5/venue, וכושר בנקודות רק כמועמד XGBoost. סרגל התוצאות נשאר 1.0204; סרגל השערים הוא NLL ‏1.4095.</p><p><b>הסבר נתונים:</b> הבחירות התקבלו בפיתוח וב־QA; המחברת מציגה אותן ואינה בוחרת מחדש. פעימה C תוסיף הקשר t−1 — שווי, מיקום ספורטיבי, נקודות והפרש — השלמות לעולות, שכבת פיצ׳רים מאוחדת ופקודת ligat run prematch. אלה תכנון המשך, לא קוד שממומש כאן.</p><p><b>הסבר כדורגלני:</b> יש כעת זיכרון של כוח מתוצאות, ולצדו קצב הבקעה וספיגה עדכני לבית ולחוץ. ההקשר מהעונה הקודמת ישלים בהמשך את התמונה, עם מיקום ספורטיבי ונקודות על המגרש.</p></div>

In [ ]:
display(form_table(decision_table()))

<div dir="rtl"><h2>B2 — מספרי בקרה ושמירת גרפים</h2><p><b>מה התא עושה:</b> מציג את עוגני הבקרה ושומר את גרפי B2 בלבד כ־PNG ו־HTML.</p><p><b>מה רואים:</b> שני ייחוסי P, הנבחר ו־z, ייחוס O ו־0 מתוך 22, שיעורי התיקו ושלושת ערכי ρ של all לקראת 2023/2024.</p><p><b>הסבר נתונים:</b> כל המספרים מחושבים מהתוצרים שאומתו בתחילת B2. B5c הוא הסקירה המשותפת הבאה; העברת התחזיות ופעולות Git נשארות ל־B6.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לפעולת השמירה; הדוח והתמונות מאפשרים לבדוק את ההסבר לפני ההמשך.</p></div>

In [ ]:
display(form_table(displayed_controls(form_review)))
b2_figure_dir = Path("reports/figures/prematch_b2")
b2_figure_dir.mkdir(parents=True, exist_ok=True)
for name, figure in b2_figures.items():
    figure.write_html(b2_figure_dir / f"{name}.html", include_plotlyjs=True)
    figure.write_image(
        b2_figure_dir / f"{name}.png", width=1800, height=figure.layout.height or 650, scale=1.5
    )
display(
    HTML(
        f'<div dir="rtl">נשמרו {len(b2_figures)} גרפים. זמן חלק B2: '
        f"{time.perf_counter() - b2_started:.2f} שניות.</div>"
    )
)

<div dir="rtl"><h2>חלק C — מהעונה הקודמת אל המידע שלפני השריקה</h2><p><b>מה התא עושה:</b> טוען את season_context ואת תוצאות EXP-003 השמורות אחרי אימות checksums; אין הרצה של ניסוי.</p><p><b>מה רואים:</b> אותם 8,630 מפתחות קבוצה־משחק ו-487 התאמות שכבר עברו QA.</p><p><b>הסבר נתונים:</b> t−1 פירושו העונה הקודמת. הערכים נקבעים לפני העונה ונשארים קבועים בה; אין כאן מידע מהמשחק שמנסים לחזות. כל הנתונים החדשים בחלק C מוגבלים לעונות הפיתוח.</p><p><b>הסבר כדורגלני:</b> תמונת אשתקד היא נקודת מוצא של המועדון, לצד Elo וכושר שמתעדכנים ממשחק למשחק.</p></div>

In [ ]:
from ligat.analysis.prematch_context_review import (
    WORDS,
    added_results,
    alone_results,
    context_associations,
    context_correlations,
    context_schema,
    control_numbers,
    direct_decisions,
    experiment_design,
    gd_translation,
    load_context_review,
    probe_reference,
    promotion_check,
    reference_ruler,
    selected_coefficients,
    sensitivity_overview,
    team_season_context,
    training_depth_table,
    validate_context_controls,
    value_fold_gaps,
    value_methodology_tables,
)
from ligat.viz.prematch_context import (
    added_chart,
    alone_chart,
    context_heatmap,
    context_matrix_table,
    context_table,
    depth_display,
    distribution_chart,
    save_context_figures,
    value_comparison_chart,
    value_fold_chart,
)

c_started = time.perf_counter()
c_review = load_context_review(ROOT)
c_figures = {}
display(
    HTML(
        f'<div dir="rtl">{len(c_review.context):,} שורות ו-{len(c_review.fits)} התאמות שמורות. '
        "אין חישוב בלוק מחדש.</div>"
    )
)

<div dir="rtl"><h2>C1 — תשעת השדות של season_context</h2><p><b>מה התא עושה:</b> מציג את עמודות הבלוק הקפוא ואת כללי ההשלמה המאושרים.</p><p><b>מה רואים:</b> שני נתחים לשווי, מיקום, נקודות והפרש שערים, שלושה דגלים וגודל ליגה.</p><p><b>הסבר נתונים:</b> שווי לעולה הוא אחוזון 30 בחצי התחתון של ליגת t−1; נקודות והפרש הם ממוצע עולות קודמות בלבד. בעונה הראשונה אין עבר, ולכן חמשת הערכים המספריים חסרים והדגל כבוי.</p><p><b>הסבר כדורגלני:</b> אין ״כוח אפס״ לעולה. ערך המיקום 0.75 הוא נקודת מוצא פשוטה: 37 העולות בעונות הפיתוח סיימו בממוצע סביב 0.74, לפי ה־ת13. זה אותו היגיון של התחלה ב-1450 ב-Elo.</p><p><b>שתי החרגות:</b> בעונת 2006/2007 אין t−1. שווי המקור של הפועל פ״ת ב-2008/2009 חסר; לכן ההשלמה שלו מופיעה בפיצ׳ר t−1 של 2009/2010 עם squad_value_imputed. שלושת הדגלים וגודל הליגה אינם עמודות ערך חסרות. שווי כולל נשאר בבלוק הקפוא גם לאחר שהוחלט שלא להכניסו ל-prematch.</p></div>

In [ ]:
display(context_table(context_schema()))

<div dir="rtl"><h2>C2.1 — דף העונה קרוב יותר לסוף העונה מאשר לאחרי הקיץ</h2><p><b>מה התא עושה:</b> מצטט את תוצאת בדיקת C1-A מדוח הביקורת; דפי העונה 26 אינם נטענים כאן.</p><p><b>מה רואים:</b> ספירמן 0.945 מול 15.5.2025 ו-0.773 מול 1.9.2025, על אותם 11 מועדונים.</p><p><b>הסבר נתונים:</b> אלה קבועים עם מקור מפורש: data/reports/nb03_C1_value_check.md, סעיף A. החריג שאושר לקריאת דפי עונה 26 חל על הבדיקה המקורית בלבד.</p><p><b>הסבר כדורגלני:</b> הממצא תומך בהחלטת ה־ת13: הדף מתאר את סוף העונה שלו, ולכן משמש תמונת t−1 לקראת העונה הבאה.</p></div>

In [ ]:
display(context_table(probe_reference()))

<div dir="rtl"><h2>C2.2 — כיוון המידע: עבר, הווה ועתיד</h2><p><b>מה התא עושה:</b> מחשב מחדש את בדיקות הכיוון מתוך עונות development בלבד, בעזרת value_methodology.</p><p><b>מה רואים:</b> נתח השווי הכולל בעונה s מול נקודות למשחק ב-s−1, ב-s וב-s+1; וכן התמדה של נקודות בין עונות.</p><p><b>הסבר נתונים:</b> כל מתאם מחושב בתוך עונה, על מועדונים משותפים. בסיכום כל עונה מקבלת משקל שווה. המספרים נבדקים מול דוח C1, ופער עוצר את התא.</p><p><b>הסבר כדורגלני:</b> השווי קשור לעבר יותר מאשר לעתיד: השוק נושא מידע על מה שהמועדון כבר הראה, ולא את תוצאת העונה הבאה.</p></div>

In [ ]:
c_values = value_methodology_tables(tables)
display(
    context_table(
        c_values["direction"],
        columns=[WORDS, "mean_spearman", "min_spearman", "max_spearman", "seasons"],
    )
)

<div dir="rtl"><h2>C2.3 — ממוצע מול סכום: השוואה מזווגת</h2><p><b>מה התא עושה:</b> משווה על אותם מועדונים את שתי הגדרות השווי מול דירוג הנקודות למשחק בעונה הבאה.</p><p><b>מה רואים:</b> לממוצע יתרון ספירמן של 0.058, עם SE של 0.017, ב-15 מתוך 17 עונות.</p><p><b>הסבר נתונים:</b> ההפרש בתצוגה הוא ממוצע פחות סכום; בדוח המקור ההפרש הוגדר הפוך. SE הוא סטיית התקן של 17 ההפרשים חלקי שורש 17, ולא רווח של 95%.</p><p><b>הסבר כדורגלני:</b> הסכום גדל עם מספר השחקנים ברשימת הסגל, שכוללת תחלופה. הממוצע משקף יותר את הערך לשחקן ופחות את גודל הרשימה.</p></div>

In [ ]:
display(context_table(c_values["paired_summary"]))

<div dir="rtl"><h2>C2.4 — יתרון הממוצע בכל מעבר עונה</h2><p><b>מה התא עושה:</b> מציג את 17 ההפרשים המזווגים ולא רק את הממוצע שלהם.</p><p><b>מה רואים:</b> ערך חיובי לטובת ממוצע השווי; יש גם שתי עונות שבהן הסכום גבוה ממנו.</p><p><b>הסבר נתונים:</b> נקודת השווי נבדקת רק כשהעונה הבאה קיימת בפיתוח; אין קריאה של עונת החזקה. העונות על הציר הן עונות השווי.</p><p><b>הסבר כדורגלני:</b> היתרון אינו כלל שעובד בכל שנה; זו השוואה של שתי דרכים לתאר סגל לאורך זמן.</p></div>

In [ ]:
c_figures["value_average_vs_total"] = value_comparison_chart(c_values["paired"])
c_figures["value_average_vs_total"].show()

<div dir="rtl"><h2>C2.5 — דף 2008/2009 נשאר כפי שהתקבל במקור</h2><p><b>מה התא עושה:</b> מציג את הפועל ת״א ומכבי חיפה מול העונה הקודמת והעונה שאחריה.</p><p><b>מה רואים:</b> סכומים של כ-3.2 ו-3.8 מיליון אירו ב-2008/2009, נמוכים בהרבה מהעונות הסמוכות.</p><p><b>הסבר נתונים:</b> זו בדיקת מקור. לא מתקנים את הערכים ולא משנים את הבלוק. הדירוג בטבלה הוא לפי השווי הממוצע של כל המועדונים באותה עונה.</p><p><b>הסבר כדורגלני:</b> הפועל ת״א שתזכה באליפות בעונה הבאה מדורגת רק שביעית בשווי הממוצע בדף הזה; מכבי חיפה שלישית. לכן נשמרה שורת רגישות בלי 2009/2010.</p></div>

In [ ]:
display(context_table(c_values["anomaly"]))

<div dir="rtl"><h2>C3.1 — מדדי אשתקד וחוזק העונה הנוכחית</h2><p><b>מה התא עושה:</b> מצמצם את הקלט לשורה אחת לכל קבוצה־עונה, ומחשב ספירמן מול נקודות 3/1/0 למשחק בעונה הנוכחית.</p><p><b>מה רואים:</b> 240 קבוצות־עונה עם עבר: 0.589 להפרש, −0.587 למיקום, 0.581 לשווי הממוצע, 0.559 לשווי הכולל ו-0.549 לנקודות.</p><p><b>הסבר נתונים:</b> אין משקל יתר לקבוצה משום ששיחקה יותר משחקים. כל מספר נבדק מול בקרת C2 בטולרנס 0.001; עונה 8 חסרה במדדי t−1.</p><p><b>הסבר כדורגלני:</b> מיקום קטן יותר הוא טוב יותר, ולכן הסימן שלו שלילי. היתר הם מדדים שבהם ערך גבוה יותר קשור לקצב נקודות גבוה יותר.</p></div>

In [ ]:
c_seasons = team_season_context(c_review, tables)
c_context_controls = validate_context_controls(c_seasons)
display(context_table(context_associations(c_seasons)))

<div dir="rtl"><h2>C3.2 — התפלגות: 37 עולות מול 203 ותיקות</h2><p><b>מה התא עושה:</b> משווה את חמשת מדדי t−1 בין קבוצות שעלו לבין קבוצות שהמשיכו בליגה.</p><p><b>מה רואים:</b> כל מדד בסולם משלו, וכל תצפית היא קבוצה־עונה; אין שכפול לפי מספר משחקים.</p><p><b>הסבר נתונים:</b> העולות מקבלות את ערכי ההשלמה שנקבעו לפני העונה. התפלגות זו אינה מבחן ולא בחירת פיצ׳ר.</p><p><b>הסבר כדורגלני:</b> אין לעולה היסטוריה באותה ליגה, אך ההשלמות נותנות לה נקודת מוצא שנשענת על עולות קודמות, ולא אפס שרירותי.</p></div>

In [ ]:
c_figures["context_distributions"] = distribution_chart(c_seasons)
c_figures["context_distributions"].show()

<div dir="rtl"><h2>C3.3 — המדדים חולקים מידע</h2><p><b>מה התא עושה:</b> מחשב מטריצת ספירמן בין חמשת המדדים ברמת קבוצה־עונה.</p><p><b>מה רואים:</b> קשרים בגודל מוחלט של כ-0.86–0.92 בין נקודות, הפרש ומיקום. לשווי הממוצע כ-0.66–0.69 מול מדדי התוצאות; לשווי הכולל כ-0.61–0.63.</p><p><b>הסבר נתונים:</b> הסימנים נשמרים: מיקום מספרי נע בכיוון ההפוך מחוזק. מתאם גבוה אינו הוראה להוציא פיצ׳ר.</p><p><b>הסבר כדורגלני:</b> שלושה מדדים מתארים את אותן תוצאות. השווי מגיע מהערכת השוק ולכן הוא מקור מידע שונה, שתרומתו תיבדק בשרשרת.</p></div>

In [ ]:
c_context_matrix = context_correlations(c_seasons)
display(context_matrix_table(c_context_matrix))

<div dir="rtl"><h2>C3.4 — אותה מטריצה בצבע</h2><p><b>מה התא עושה:</b> מציג את המתאמים החתומים על סולם משותף מ-1− עד 1.</p><p><b>מה רואים:</b> הגוש של מדדי התוצאות, והכיוון ההפוך של המיקום.</p><p><b>הסבר נתונים:</b> זה אותו חישוב מהטבלה הקודמת, ללא חישוב חדש או סינון.</p><p><b>הסבר כדורגלני:</b> ההפרש והנקודות מספרים סיפור דומה; הקשר לביצועי עבר אינו מוכיח תוספת מעל Elo.</p></div>

In [ ]:
c_figures["context_correlations"] = context_heatmap(c_context_matrix)
c_figures["context_correlations"].show()

<div dir="rtl"><h2>C3.5 — בדיקת העולות: 1.12 לפני העונה מול 1.09 בפועל</h2><p><b>מה התא עושה:</b> משווה את ממוצע ההשלמה ל-prev_ppm של 37 העולות לקצב הנקודות שלהן בעונה הנוכחית.</p><p><b>מה רואים:</b> 1.1225 נקודות למשחק כערך פתיחה, מול 1.0895 בפועל.</p><p><b>הסבר נתונים:</b> ממוצע שווה לכל אירוע עלייה. ערך ההשלמה משתמש רק בעולות מעונות קודמות; העונה הנוכחית משמשת לבדיקה בלבד.</p><p><b>הסבר כדורגלני:</b> הערך נותן לעולה קצב סביר להתחיל ממנו, אך אינו מבטיח מה תעשה עולה מסוימת.</p></div>

In [ ]:
display(context_table(promotion_check(c_seasons)))

<div dir="rtl"><h2>C4.1 — EXP-003: שלוש שאלות שונות</h2><p><b>מה התא עושה:</b> מציג את שלושת המסלולים כפי שהוגדרו לפני הריצה.</p><p><b>מה רואים:</b> A בודק מדד לבדו; O בודק תוספת מעל Elo; P בודק תוספת מעל מודל השערים.</p><p><b>הסבר נתונים:</b> A ו-O נשפטים ב-log loss של בית/תיקו/חוץ; P נשפט ב-Poisson NLL. אין להשוות בין שני המדדים המספריים כאילו הם אותו סרגל.</p><p><b>הסבר כדורגלני:</b> שווי יכול לתאר היטב מועדון לבדו ובכל זאת לא להוסיף מעל מה שהדירוג והכושר כבר מספרים.</p></div>

In [ ]:
c_routes, c_candidates = experiment_design()
display(context_table(c_routes))

<div dir="rtl"><h2>C4.2 — שבעת המועמדים, במילים ובנוסחה</h2><p><b>מה התא עושה:</b> מציג את ההגדרות המדויקות ששימשו בניסוי הקפוא.</p><p><b>מה רואים:</b> שני סוגי שווי, שתי השלמות למיקום עולה, נקודות, הפרש, ושווי־מיקום יחד.</p><p><b>הסבר נתונים:</b> הנוסחאות הן מנקודת המבט של הבית. בשורת החוץ ההפרש או log היחס מתהפכים. מיקום קטן יותר טוב יותר.</p><p><b>הסבר כדורגלני:</b> ההשוואה בין 0.75 לממוצע עולות שואלת אם עולה צריכה נקודת מוצא פשוטה או ממוצע שנלמד מהעבר.</p></div>

In [ ]:
display(context_table(c_candidates))

<div dir="rtl"><h2>C4.3 — אותם משחקים, סרגל הוגן</h2><p><b>מה התא עושה:</b> מציג את הייחוסים שאומנו מחדש מעונה 9 ואת הייחוסים הרשמיים הישנים שהתחילו בעונה 8.</p><p><b>מה רואים:</b> Elo הוא 1.02076 לעומת 1.02042; Poisson הוא 1.41010 לעומת 1.40946.</p><p><b>הסבר נתונים:</b> לעונה 8 אין t−1, ולכן גם הייחוס מתחיל מעונה 9. הקיפול הראשון מתאמן על שתי עונות בלבד. d̄ הוא ייחוס פחות מועמד; SE הוא סטיית התקן של ההפרשים העונתיים חלקי שורש מספר הקיפולים.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לשינוי הטכני בסרגל. ההשוואה ההוגנת מוודאת שהמדד והייחוס נשפטים על אותם משחקים ואותו עבר.</p><p><b>הכללים המקוריים:</b> ״מוסיף״ אם d̄ &gt; SE על 15 הקיפולים. סכום צריך לשפר יותר מ-SE אחד מול הממוצע ב-O או ב-P; ממוצע עולות צריך לעשות זאת מול 0.75. ה־ת6 משווה את ההכרעה הזו לכיוון לפי הציון בלבד בחמש האחרונות, ללא תנאי SE בחמש. כלל עומק האימון יופיע בהמשך כהחלטה שנוספה אחרי התוצאות.</p></div>

In [ ]:
c_ruler = reference_ruler(c_review)
display(context_table(c_ruler, columns=["method", "training_start", "metric", "score", WORDS]))

<div dir="rtl"><h2>C5.1 — מסלול A: מה מדד אשתקד אומר לבדו</h2><p><b>מה התא עושה:</b> קורא את ציוני שבעת המועמדים מהקיפולים השמורים, ב-15 וב-5 האחרונות.</p><p><b>מה רואים:</b> הפרש שערים הוא המדורג הראשון בין המועמדים לבדם; כל המועמדים טובים מ-B0 אך אינם מגיעים ל-Elo.</p><p><b>הסבר נתונים:</b> הממוצעים משוקללים במספר המשחקים. הדירוג הוא בין שבעת המועמדים בלבד; אינו בוחר הגדרה.</p><p><b>הסבר כדורגלני:</b> תוצאות העונה הקודמת אומרות הרבה על מועדון, אך דירוג שמתעדכן ממשחק למשחק נושא יותר מידע.</p></div>

In [ ]:
c_alone = alone_results(c_review)
display(
    context_table(
        c_alone, columns=["method", "score", "rank_all", "recent_score", "rank_last5", WORDS]
    )
)

<div dir="rtl"><h2>C5.2 — מסלול A, עם קווי B0 ו-Elo</h2><p><b>מה התא עושה:</b> מציג את אותה טבלה כגרף בשני חלונות הזמן.</p><p><b>מה רואים:</b> לכל חלון קווי הייחוס שלו; קטן יותר הוא טוב יותר.</p><p><b>הסבר נתונים:</b> אין להשוות את עמודות חמש האחרונות לקו ייחוס של 15. שני הפאנלים משתמשים בייחוסים התואמים שלהם.</p><p><b>הסבר כדורגלני:</b> גם כשהמדד מתאר נכון את ההבדלים בין מועדונים, הוא נשאר רחוק מהדירוג המתעדכן.</p></div>

In [ ]:
c_figures["experiment_alone"] = alone_chart(c_alone, c_ruler)
c_figures["experiment_alone"].show()

<div dir="rtl"><h2>C5.3 — האם המדד מוסיף מעל Elo ומעל מודל השערים</h2><p><b>מה התא עושה:</b> מחשב הפרשים מזווגים מתוך הציונים השמורים, כולל SE ומניין הקיפולים הטובים.</p><p><b>מה רואים:</b> אף מועמד אינו עומד בכלל המקורי d̄ &gt; SE על כל 15 הקיפולים. הכיוון בחמש האחרונות מוצג בנפרד.</p><p><b>הסבר נתונים:</b> ערך חיובי הוא שיפור. בטבלה מוצג הגודל המקורי; בגרפים הבאים כל הערכים מוכפלים ב-1000. אין סימון ״מוסיף״ לפי חמש האחרונות.</p><p><b>הסבר כדורגלני:</b> רוב המדדים נגזרים מאותן תוצאות שכבר נמצאות ב-Elo. השווי עשוי להיות שונה, אך גם את תרומתו צריך לבדוק מעל הייחוס.</p></div>

In [ ]:
c_added = added_results(c_review)
display(
    context_table(
        c_added,
        columns=[
            "route",
            "method",
            "improvement",
            "se",
            "better",
            "folds",
            "recent_improvement",
            "recent_better",
            "adds",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>C5.4 — מסלול O: תוספת מעל Elo</h2><p><b>מה התא עושה:</b> מציג d̄ עם טווח של SE אחד לכל צד, באלפיות log loss.</p><p><b>מה רואים:</b> קו אפס ברור; ערך שלילי פירושו שהמועמד נתן ציון גרוע מהייחוס.</p><p><b>הסבר נתונים:</b> הטווח הוא ±SE, ולא רווח סמך של 95%. זה אותו כלל שנקבע מראש בניסוי.</p><p><b>הסבר כדורגלני:</b> מדד חזק לבדו לא בהכרח אומר משהו חדש אחרי שמכירים את Elo.</p></div>

In [ ]:
c_figures["experiment_added_O"] = added_chart(c_added, "O")
c_figures["experiment_added_O"].show()

<div dir="rtl"><h2>C5.5 — מסלול P: תוספת מעל מודל השערים</h2><p><b>מה התא עושה:</b> מציג d̄ עם ±SE באלפיות Poisson NLL.</p><p><b>מה רואים:</b> ההשוואה היא מול EWMA5/venue עם Elo, לא מול B0.</p><p><b>הסבר נתונים:</b> ערכי ציר P אינם log loss. הכפלה ב-1000 היא לצורך קריאה בלבד.</p><p><b>הסבר כדורגלני:</b> כאן כבר ידועים גם הדירוג וגם מצב ההתקפה וההגנה של שתי הקבוצות.</p></div>

In [ ]:
c_figures["experiment_added_P"] = added_chart(c_added, "P")
c_figures["experiment_added_P"].show()

<div dir="rtl"><h2>C5.6 — תרגום כדורגלני: מה נשאר מהפרש השערים</h2><p><b>מה התא עושה:</b> מציב את רבעוני פער הפרש השערים במקדמים החציוניים השמורים, ללא התאמה מחדש.</p><p><b>מה רואים:</b> לבד: סיכוי הבית עולה מ-33.7% ל-48.5%; מעל Elo, מ-39.7% ל-41.1%.</p><p><b>הסבר נתונים:</b> הרבעונים נקבעו על שורות הבית בוולידציה; ב-O מחזיקים Elo שווה. זו המחשה בנקודת ייחוס, לא ממוצע של כל השינויים במשחקים.</p><p><b>הסבר כדורגלני:</b> הפרש השערים של אשתקד הוא מדד חוזק טוב, אבל רוב הסיפור שהוא מספר כבר נמצא ב-Elo.</p></div>

In [ ]:
display(context_table(gd_translation(c_review)))

<div dir="rtl"><h2>C5.7 — ההכרעות שהוגדרו מראש</h2><p><b>מה התא עושה:</b> מציג את ארבע ההשוואות הישירות ואת דגל ה־ת6.</p><p><b>מה רואים:</b> הממוצע נשאר מול הסכום, 0.75 נשאר מול ממוצע העולות, ואין דגל אי־הסכמה בארבע השורות.</p><p><b>הסבר נתונים:</b> התנאי ב-15 הוא d̄ &gt; SE לטובת החלופה; בחמש האחרונות משווים ציונים בלבד. ההכרעה המוצגת היא זו שאושרה ב-EXP-003.</p><p><b>הסבר כדורגלני:</b> הנתונים אינם מצדיקים סיבוך של נקודת הפתיחה לעולה מעבר ל-0.75.</p></div>

In [ ]:
display(
    context_table(
        direct_decisions(c_review),
        columns=[
            "route",
            "method",
            "improvement",
            "se",
            "condition",
            "recent_improvement",
            "disagreement",
            "approved",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>C6.1 — כל ההפסד של השווי מרוכז בקיפול הדל הראשון</h2><p><b>מה התא עושה:</b> מציג את תוספת value_avg לכל עונת ולידציה, ב-O וב-P.</p><p><b>מה רואים:</b> 2009/2010 מודגשת: קיפול עם שתי עונות אימון והפסד גדול; האחרים קרובים הרבה יותר לאפס.</p><p><b>הסבר נתונים:</b> הערכים הם ייחוס פחות מועמד, כפול 1000. הפער בקיפול 11 הוא כ-14.43− ב-O וכ-24.17− ב-P.</p><p><b>הסבר כדורגלני:</b> באותו מעבר יש גם עבר אימון קצר וגם דף שווי 2008/2009 חריג, שבו האלופה הבאה דורגה רק שביעית בשווי הממוצע.</p></div>

In [ ]:
c_value_folds = value_fold_gaps(c_review)
c_figures["value_by_validation_season"] = value_fold_chart(c_value_folds)
c_figures["value_by_validation_season"].show()

<div dir="rtl"><h2>C6.2 — המקדמים מראים איך המשקל התחלק</h2><p><b>מה התא עושה:</b> קורא את מקדמי קיפולים 11, 12, 18 ו-25, עם שווי ובלעדיו.</p><p><b>מה רואים:</b> ב-P של הקיפול הראשון מקדם Elo יורד מכ-1.20 לכ-0.39; מקדם השווי כ-0.158 מול כ-0.06 בקיפולים מאוחרים.</p><p><b>הסבר נתונים:</b> אלה המקדמים שנאמדו בריצה המקורית. אין התאמה חדשה, ואין פרשנות שלפיה כל מקדם הוא השפעה סיבתית.</p><p><b>הסבר כדורגלני:</b> כששווי ו-Elo מתארים חלק מאותו חוזק, המודל מחלק ביניהם משקל. עם מעט עונות החלוקה פחות יציבה.</p></div>

In [ ]:
display(context_table(selected_coefficients(c_review)))

<div dir="rtl"><h2>C6.3 — שורת הרגישות: בלי עונה 11</h2><p><b>מה התא עושה:</b> מציג את ההשוואה שבה עונה 11 הוסרה גם מהאימון וגם מהולידציה.</p><p><b>מה רואים:</b> הקיפול הראשון הוא עכשיו 2010/2011: שוב שתי עונות אימון, ושוב הפסד לשווי.</p><p><b>הסבר נתונים:</b> גם הייחוס אומן באותו תנאי. השוואה זו מכסה 14 קיפולים ולא את המדגם המקורי של 15.</p><p><b>הסבר כדורגלני:</b> הבעיה אינה נפתרת רק בהוצאת הדף החריג: אימון קצר הוא גורם בפני עצמו, גם כשהדף הבא תקין.</p></div>

In [ ]:
display(context_table(sensitivity_overview(c_review)))

<div dir="rtl"><h2>C7 — ה־ת14: מה קורה כשמוציאים קיפולים דלים מהסיכום</h2><p><b>מה התא עושה:</b> מסכם את אותם קיפולים שמורים לפי מינימום של 2, 4, 6 או 8 עונות אימון.</p><p><b>מה רואים:</b> הטבלה של EXPERIMENTS: d̄ (SE) כפול 1000, ולצדה הכיוון בחמש האחרונות.</p><p><b>הסבר נתונים:</b> כל הקיפולים עדיין מדווחים. זו בדיקה בדיעבד שהדגימה את העיקרון, ולא ניסוי חדש שהוכרז מראש.</p><p><b>הסבר כדורגלני:</b> המודל הסופי יתאמן על כל העבר, ולכן קבוצה לא אמורה לאבד מדד שימושי רק בגלל קיפול שהתאמן על שתי עונות.</p><p style="font-size:1.25em;border:3px solid #6b7280;padding:12px"><b>בדיקה בדיעבד, אחרי התוצאות. הדגמה של העיקרון, לא תוצאה מוכרזת מראש.</b></p><p><b>הסתייגות מהממצא:</b> הרווח אינו גדל עם עומק האימון; בסף של 8 עונות התוספת של value_avg ב-P היא כ-0.19 אלפיות בלבד.</p></div>

In [ ]:
c_depth = training_depth_table(c_review)
display(depth_display(c_depth))

<div dir="rtl"><h2>C7 — החלטת עמית ה־ת14</h2>
<ol><li>המודל הסופי מתאמן על כל העונות; קיפול דל אינו סיבה מספקת לפסול מדד.</li>
<li>ניסוי חדש מכריז מראש על מינימום עונות אימון — ברירת מחדל 4. ההכרעה דורשת d̄ &gt; SE בקיפולים העמוקים ובנוסף כיוון חיובי בלפחות 4 מתוך 5 האחרונים. לא משנים בדיעבד את EXP-001 או EXP-002.</li>
<li>value_avg נכנס ל-M2 כברירת מחדל, <b>בחריגה מתועדת מכלל הניסוי המקורי</b>. ה-ablation של M2 עדיין יכול להוציא אותו; עונה 26 נשארת בדיקה חד־פעמית.</li>
<li>value_total ו-position_mean יוצאים; 0.75 נשאר לעולה.</li>
<li>gd, ppm ו-position נשארים 🔬 ל-ablation של M2, ו-ppm גם מועמד ל-M5.</li></ol>
<p><b>הסבר נתונים:</b> זו החלטת עמית אחרי התוצאות, כפי שתועדה ב-VISION §2.4 וב-EXPERIMENTS כלל 8; לא כלל שהפעלנו מחדש לבחירה.</p>
<p><b>הסבר כדורגלני:</b> השווי הוא מקור מידע שאינו רק סיכום של אותן תוצאות, ויקבל הזדמנות מסודרת ב-M2. מעמדו אינו מבטיח שיתרום.</p></div>

<div dir="rtl"><h2>C8 — מספרי בקרה ושמירת גרפי חלק C</h2><p><b>מה התא עושה:</b> מציג מספרי בקרה מהקבצים המאומתים ושומר את שבעת גרפי C6a כ-PNG ו-HTML.</p><p><b>מה רואים:</b> B0, Elo, Poisson, תוספת השווי ב-P ו-487 ההתאמות; לצד בדיקת מתאמי C2.</p><p><b>הסבר נתונים:</b> כל מספר מחושב מהתוצרים השמורים ונבדק מול המקור. המקורות נשמרים ללא שינוי; תאים 0–119 שומרים על אותו קוד וטקסט.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לפעולת השמירה; אלה עוגנים שיאפשרו לעמית ול-Claude לבדוק את הסיפור מול המספרים.</p></div>

In [ ]:
display(context_table(control_numbers(c_review)))
display(
    context_table(
        c_context_controls,
        columns=["metric", "spearman", "expected", "difference", "passed", WORDS],
    )
)
c_elapsed = time.perf_counter() - c_started
c_saved_figures = save_context_figures(c_figures, Path("reports/figures/prematch_c"))
display(
    HTML(
        f'<div dir="rtl">נשמרו {len(c_saved_figures)} גרפים. זמן חלק C6a: {c_elapsed:.2f} שניות. '
        "המשך C6b יעסוק בטבלת prematch ובממצאי C5.</div>"
    )
)

<div dir="rtl"><h2>C9 — מה M0 מספק לכל השרשרת</h2><p><b>מה התא עושה:</b> קורא את prematch_v1 השמור אחרי אימות SHA-256 מול ה-sidecar וארבעת המקורות הקפואים, ואז מחשב את סיכומי C5 באמצעות המודול הקיים.</p><p><b>מה רואים:</b> שורה לכל קבוצה לפני משחק: אותה טבלת מידע תשמש את M1, M2 ואת המשך השרשרת, לפי ההקצאה לכל מודל.</p><p><b>הסבר נתונים:</b> אין בנייה מחדש של בלוק, אין אימון ואין יעד בתוך prematch. תוצאות אמת מתחברות בזיכרון רק לצורך הסריקה התיאורית ומושוות לכל תוצאות C5 השמורות. כל הנתונים החדשים הם מעונות הפיתוח בלבד.</p><p><b>הסבר כדורגלני:</b> זו תמונת המידע שלפני השריקה: מצב הטבלה, חוזק הקבוצות, הכושר והעונה הקודמת. היא מאפשרת לשאול בהמשך מה מוסיפה המוטיבציה מעל החוזק הידוע.</p></div>

In [ ]:
from ligat.analysis.prematch_final_review import (
    analyze_final_prematch,
    check_c5_reference,
    column_groups,
    cross_family_pairs,
    dead_game_summary,
    final_controls,
    load_final_prematch,
    m0_deliverables,
    match_example,
    missingness,
    outcome_statistics,
    overlap_checks,
    planning_decisions,
    target_leaders,
    target_trends,
)
from ligat.viz.prematch_final import (
    contrast_heatmap,
    cramers_chart,
    example_table,
    final_table,
    outcome_chart,
    save_final_figures,
    target_comparison_chart,
    trend_description,
)

final_started = time.perf_counter()
final_frame, final_sha = load_final_prematch(ROOT)
final_review = analyze_final_prematch(final_frame, tables, final_sha)
check_c5_reference(final_review, ROOT / "data/reports/nb03_C5_correlations.json")
final_figures = {}
display(
    HTML(
        '<div dir="rtl">prematch_v1 אומת; ה-SHA-256 הוא '
        f'<bdi dir="ltr">{final_sha}</bdi>. תוצאות C5 תואמות לדוח.</div>'
    )
)

<div dir="rtl"><h2>C9.1 — שש קבוצות העמודות</h2><p><b>מה התא עושה:</b> מציג את קבוצות העמודות לפי DATA_DICTIONARY, ואת מקור כל קבוצה.</p><p><b>מה רואים:</b> 30 עמודות טבלה, 4 Elo, שלוש כושר קבוצה, שתיים כושר יריבה, 8 הקשר עונתי ו-4 הפרשים; בסך הכול 51.</p><p><b>הסבר נתונים:</b> 51 עמודות אינן 51 פיצ׳רים מספריים: שדות תאריך, שלב ומפתחות אינם כולם חלק מסריקת המתאמים. ב-C5 יש 45 פיצ׳רים מספריים או בוליאניים.</p><p><b>הסבר כדורגלני:</b> ההתקפה של קבוצה מוצלבת עם הגנת היריבה. ההפרשים מתארים מי מגיעה חזקה יותר ביחס לקבוצה שמולה.</p><div style="border:3px solid #6b7280;padding:12px;font-size:1.15em"><b>שתי מלכודות שחשוב לזכור:</b><ul><li>goals_for, goals_against, points, wins, draws ו-losses <b>מצטברים לפני המשחק</b>. אלה אינם יעדי המשחק הנוכחי.</li><li>ב-position_log_ratio <b>נמוך טוב יותר</b>: מקום 1 טוב ממקום 14. יחס חיובי פירושו שמיקום הקבוצה היה גרוע ממיקום היריבה.</li></ul></div></div>

In [ ]:
display(final_table(column_groups()))

<div dir="rtl"><h2>C9.2 — אותו משחק, שתי שורות זו לצד זו</h2><p><b>מה התא עושה:</b> בוחר משחק מ-2023/2024 אחרי לפחות עשרה משחקים של הבית, מציג את שתי נקודות המבט ומאמת את הקשרים ביניהן.</p><p><b>מה רואים:</b> ההפרשים מחליפים סימן, וכושר ההתקפה של היריבה בצד אחד הוא כושר ההתקפה של הקבוצה בצד השני. שמות הקבוצות נלקחים מ-config/teams.yaml.</p><p><b>הסבר נתונים:</b> הטבלה מסובבת לתצוגה בלבד: כל עמודה היא שורת קבוצה־משחק מקורית. הבחירה באמצע העונה נועדה להראות שערים ונקודות מצטברים, ולא משנה שום מדגם מחקר.</p><p><b>הסבר כדורגלני:</b> אפשר לראות איך אותו מפגש מתואר פעם עבור המארחת ופעם עבור האורחת; המודל ינסה לחזות את מספר השערים של כל צד בנפרד.</p></div>

In [ ]:
final_example = match_example(final_frame, ROOT / "config/teams.yaml")
display(example_table(final_example))

<div dir="rtl"><h2>C9.3 — חסר אמיתי נשאר חסר</h2><p><b>מה התא עושה:</b> סופר חסרים בנפרד ב-2006/2007 ובעונות המאוחרות, ללא מילוי חדש.</p><p><b>מה רואים:</b> מדדי t−1 וארבעת ההפרשים חסרים רק ב-396 שורות העונה הראשונה. לפערי הטבלה יש חסרים נוספים התלויים בשלב.</p><p><b>הסבר נתונים:</b> ה-loader אימת שערכי gap_to_* וחסריהם זהים ל-table_state_v2. חוסר הגדרה של פער אחרי פיצול אינו חור בהיסטוריה ואינו מוחלף באפס.</p><p><b>הסבר כדורגלני:</b> בתחילת הדאטה אין עונה קודמת. אחרי פיצול הפלייאוף קווים מסוימים כבר אינם יעד משותף; אפס היה אומר בטעות שהקבוצה בדיוק על הקו.</p></div>

In [ ]:
display(final_table(missingness(final_frame)))

<div dir="rtl"><h2>C10 — תשעה זוגות חופפים בין משפחות</h2><p><b>מה התא עושה:</b> מציג את הזוגות המסומנים בין משפחות מתוך feature_pair_correlations של C5.</p><p><b>מה רואים:</b> למדדי t−1 יש חפיפה גבוהה עם Elo. הטבלה מציגה גם את הכיוון בחמש האחרונות ואת הטווח בין העונות.</p><p><b>הסבר נתונים:</b> הסימון המקורי הוא ערך מוחלט של Pearson גדול מ-0.8. בתוך משפחה יש 39 זוגות מסומנים נוספים; פירוטם בדוח C5. זו הגבלת תצוגה בלבד.</p><p><b>הסבר כדורגלני:</b> נקודות, הפרש שערים ומיקום מספרים חלק מאותו סיפור שכבר נכנס לדירוג. שווי מגיע מהשוק ולכן עשוי לשאת גם מידע אחר.</p><p><b>החלטת עמית:</b> C5 הוא דיווח בלבד. משאירים את כל העמודות; XGBoost מתמודד עם פיצ׳רים חופפים, ואת התרומה נבדוק ב-ablation של M2 ובדיון התכנון.</p><p>מקור מלא: data/reports/nb03_C5_correlations.md.</p></div>

In [ ]:
display(
    final_table(
        cross_family_pairs(final_review),
        columns=[
            "feature_a",
            "feature_b",
            "pearson",
            "pearson_last5",
            "pearson_min_season",
            "pearson_max_season",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>C10.1 — אותם חמישה מדדים בשתי תקופות</h2><p><b>מה התא עושה:</b> מציג את ארבעת הפרשי t−1 ו-elo_diff בשתי מטריצות Pearson שחושבו ב-C5.</p><p><b>מה רואים:</b> חפיפה חיובית בכחול ושלילית באדום; לשתי התקופות אותו סולם מ-1− עד 1, עם מרכז באפס.</p><p><b>הסבר נתונים:</b> הפאנל של חמש האחרונות משתמש רק ב-2019/2020–2023/2024. אין שינוי בגבולות הסולם בין הפאנלים, ואין p-value.</p><p><b>הסבר כדורגלני:</b> מיקום טוב הוא מספר קטן, ולכן כיוון המתאם שלו הפוך. החפיפה עם חוזק העבר נשארת גם בליגה העכשווית.</p></div>

In [ ]:
final_figures["c6b_contrast_pearson"] = contrast_heatmap(final_review.matrices)
final_figures["c6b_contrast_pearson"].show()

<div dir="rtl"><h2>C10.2 — כפילות שאפשר לבדוק ישירות</h2><p><b>מה התא עושה:</b> מאמת בקוד את הפרש ספירות המשחקים, את זהות השלבים ואת החריג בין שני דגלי ההשלמה.</p><p><b>מה רואים:</b> הפרש קבוע 1; אפס משחקים עם stage שונה; 35 שורות חריגות ב-2009/2010. ציפיית Elo אינה מדד כוח עצמאי נוסף.</p><p><b>הסבר נתונים:</b> שלוש התוצאות הראשונות נבדקות מתוך הטבלה. ציפיית Elo היא פונקציה לוגיסטית של הפער בתוספת יתרון הבית וכיוון השורה; היא אינה פונקציה של הפער לבדו כשמערבבים בית וחוץ.</p><p><b>הסבר כדורגלני:</b> עולה אינה בהכרח אותו דבר כמו קבוצה ששווי הסגל שלה הושלם: להפועל פ״ת הושלם גם שווי מקור חסר. אלה שאלות על ייצוג הנתונים, ולא סיבה אוטומטית למחיקת עמודה.</p></div>

In [ ]:
display(final_table(overlap_checks(final_frame)))

<div dir="rtl"><h2>C11 — אילו מדדים קשורים להפרש השערים במשחק</h2><p><b>מה התא עושה:</b> מציג את עשרת הפיצ׳רים עם Pearson המוחלט החזק ביותר מול הפרש השערים במשחק עצמו.</p><p><b>מה רואים:</b> הקשר הכללי, הקשר בחמש האחרונות, חציון עונתי, קצוות בין העונות ומספר העונות עם אותו סימן.</p><p><b>הסבר נתונים:</b> המיון הוא לצורך הצגה בלבד. יציבות נבדקת לפי השונות בין העונות: הטווח, החציון ומניין הסימנים. לא מסיקים תרומה מעל Elo ממתאם מול יעד.</p><p><b>הסבר כדורגלני:</b> חוזק קודם קשור בדרך כלל לשליטה בתוצאה על המגרש. כדי לדעת אם מידע מוסיף אחרי שידוע הדירוג, צריך את ההשוואה המזווגת בניסוי.</p><p style="border:3px solid #6b7280;padding:12px;font-size:1.15em"><b>כל משחק מופיע פעמיים. אין כאן p-value למתאמים, ואין 8,630 משחקים עצמאיים.</b> היציבות מתוארת בין העונות.</p></div>

In [ ]:
display(
    final_table(
        target_leaders(final_review, "goal_difference_game", 10),
        columns=[
            "feature",
            "pearson",
            "pearson_last5",
            "median_season_pearson",
            "pearson_min_season",
            "pearson_max_season",
            "seasons_same_sign",
            "seasons_counted",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>C11.1 — השווי מתחזק יותר משני מדדי Elo</h2><p><b>מה התא עושה:</b> מציג גרף נקודות של חמישה מדדים בשני חלונות הזמן, ומחשב את שינויי המתאם ואת היחסים ביניהם מהערכים המלאים.</p><p><b>מה רואים:</b> לשווי העלייה הגדולה ביותר מבין חמשת המדדים. השינויים והיחס לשני מדדי Elo מוצגים בפלט המחושב, ללא מספרים מוקלדים.</p><p><b>הסבר נתונים:</b> השווי, ציפיית Elo ופער Elo אינם אותו משתנה. התרגום שמתחת לגרף מבדיל ביניהם, בהתאם להבהרת עמית; זו התאמה תיאורית ל-EXP-003 ולא הוכחה.</p><p><b>הסבר כדורגלני:</b> ייתכן שהערכת השוק מתארת טוב יותר את פערי הכוחות בליגה העכשווית. זה עדיין אינו מוכיח שהיא מוסיפה מעל Elo והכושר במודל.</p></div>

In [ ]:
final_trends = target_trends(final_review)
final_figures["c6b_target_periods"] = target_comparison_chart(final_trends)
final_figures["c6b_target_periods"].show()
display(HTML(trend_description(final_trends)))

<div dir="rtl"><h2>C11.2 — גם שלושת היעדים האחרים נשארים נפרדים</h2><p><b>מה התא עושה:</b> מציג חמישה פיצ׳רים חזקים לכל אחד משלושת היעדים האחרים.</p><p><b>מה רואים:</b> שערי הקבוצה, שערי היריבה ונקודות 3/1/0 במשחק; אלה יעדים זמניים לניתוח בלבד.</p><p><b>הסבר נתונים:</b> כל יעד מקבל דירוג תצוגה נפרד לפי Pearson מוחלט. אין יעד שנשמר ב-prematch, ואין כאן רשימת פיצ׳רים שנבחרה למודל.</p><p><b>הסבר כדורגלני:</b> להבקיע, לא לספוג ולצבור נקודות הן שאלות קרובות אך שונות. M2 יפריד התקפה מהגנה באמצעות שתי נקודות המבט של המשחק.</p></div>

In [ ]:
final_other_targets = pd.concat(
    [
        target_leaders(final_review, target, 5)
        for target in ("goals_scored", "goals_conceded", "points_game")
    ],
    ignore_index=True,
)
display(
    final_table(
        final_other_targets,
        columns=["target", "feature", "pearson", "pearson_last5", "median_season_pearson", WORDS],
    )
)

<div dir="rtl"><h2>C12 — χ²: קשר לבית, תיקו או חוץ</h2><p><b>מה התא עושה:</b> מציג את Cramér’s V לכל פיצ׳ר בר־בדיקה, מתוך מבחני C5 על שורות הבית בלבד.</p><p><b>מה רואים:</b> 4,315 משחקים; גודל קשר מרבי של כ-0.227. מסגרת וכוכבית מציינות תא עם ספירה צפויה מתחת ל-5.</p><p><b>הסבר נתונים:</b> Cramér’s V מתאר את גודל הקשר. ה-p-value תיאורי בלבד: במדגם גדול גם קשר זניח עשוי להיראות מובהק. אין כלל בחירה או סימון מובהקות. פיצ׳ר קבוע אינו מקבל V של אפס.</p><p><b>הסבר כדורגלני:</b> קבוצות חזקות מנצחות לעיתים קרובות יותר, אך אף פיצ׳ר בודד אינו קובע תוצאת משחק. גם דירוג טוב משאיר אי־ודאות גדולה.</p></div>

In [ ]:
final_chi = outcome_statistics(final_review)
final_figures["c6b_cramers_v"] = cramers_chart(final_chi)
final_figures["c6b_cramers_v"].show()

<div dir="rtl"><h2>C12.1 — פיצ׳רים שלא מאפשרים מבחן</h2><p><b>מה התא עושה:</b> מציג את שורות ההחרגה שנותרו בדוח C5.</p><p><b>מה רואים:</b> is_home קבוע בשורות הבית ולכן אין לו מבחן χ²; n והסיבה נשארים, וערכי המבחן חסרים.</p><p><b>הסבר נתונים:</b> לא ממלאים ערך אפס עבור מבחן שלא ניתן לחשב. stage של הבית כן נכלל במבחנים, אף שאינו מתאים למתאם Pearson.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית להחרגה: זאת מגבלה של מדגם שמכיל רק מארחות, ולא מסקנה על יתרון הבית.</p></div>

In [ ]:
display(
    final_table(
        outcome_statistics(final_review, excluded=True),
        columns=[
            "feature",
            "n",
            "categories",
            "chi2",
            "dof",
            "p_value",
            "cramers_v",
            "min_expected",
            "cramers_v_last5",
            "excluded_reason",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>C12.2 — כשפער Elo גדל, חלקן של המארחות המנצחות עולה</h2><p><b>מה התא עושה:</b> מציג שיעורי בית/תיקו/חוץ בחמישוני elo_diff לפי גבולות C5, ללא חישוב חלוקה חדשה.</p><p><b>מה רואים:</b> החמישונים מסודרים מפער נמוך לגבוה; המספר בכל תווית הוא מספר המשחקים.</p><p><b>הסבר נתונים:</b> העמודות מתחילות באפס ומסתכמות ל-100%. הקווים המקווקווים מסמנים את גבולות שכבות הבית והתיקו במדגם הכללי; שיעורי הבסיס מוצגים מעל הגרף.</p><p><b>הסבר כדורגלני:</b> כשמארחת מגיעה חזקה יותר לפי הדירוג, היא מנצחת יותר. התיקו אינו נעלם גם במשחקים שבהם יש פייבוריטית.</p></div>

In [ ]:
final_figures["c6b_outcome_elo"] = outcome_chart(
    final_review.outcomes, ("elo_diff",), "פער Elo והסיכוי לתוצאת המשחק"
)
final_figures["c6b_outcome_elo"].show()

<div dir="rtl"><h2>C12.3 — שווי אשתקד ותוצאת המשחק</h2><p><b>מה התא עושה:</b> מציג שיעורי תוצאות לפי חמישוני יחס השווי הממוצע.</p><p><b>מה רואים:</b> חלקן של המארחות המנצחות גדל כשיחס השווי לטובתן.</p><p><b>הסבר נתונים:</b> לשווי אין t−1 ב-2006/2007, ולכן הבסיס כאן מחושב על המשחקים שיש בהם שווי, ולא על כל 4,315 המשחקים. אותם גבולות חמישונים נשמרים מ-C5.</p><p><b>הסבר כדורגלני:</b> סגל יקר יותר קשור לפער איכות, אבל זה מתאם בלי בקרה על Elo, כושר או זהות הקבוצות; הוא אינו אומד השפעה סיבתית של כסף.</p></div>

In [ ]:
final_figures["c6b_outcome_value"] = outcome_chart(
    final_review.outcomes, ("value_avg_log_ratio",), "שווי אשתקד: מהקבוצה הזולה ליקרה יותר"
)
final_figures["c6b_outcome_value"].show()

<div dir="rtl"><h2>C12.4 — העולה מארחת או מתארחת</h2><p><b>מה התא עושה:</b> מציג בנפרד את דגל העולה של הבית ושל האורחת, בשני פאנלים.</p><p><b>מה רואים:</b> עולה בבית מנצחת כ-30.2%, לעומת כ-42.2% לשאר המארחות. כשאורחת עולה, הבית מנצחת כ-48.1%.</p><p><b>הסבר נתונים:</b> H/D/A תמיד מנקודת המבט של הבית, גם בפאנל האורחת. דגל העלייה מתאר קבוצה שנכנסה לעונה כעולה, ולא את חוזקה אחרי שכבר שיחקה.</p><p><b>הסבר כדורגלני:</b> ההתאקלמות בליגה קשורה לקושי לצבור ניצחונות גם בבית. עדיין יש הבדלים גדולים בין עולות, והשערת השווי כמבדיל ביניהן טרם נבדקה.</p></div>

In [ ]:
final_figures["c6b_outcome_promoted"] = outcome_chart(
    final_review.outcomes,
    ("promoted_in", "away_promoted_in"),
    "עולה בבית ובעת ביקור של עולה: שתי נקודות מבט",
)
final_figures["c6b_outcome_promoted"].show()

<div dir="rtl"><h2>C12.5 — משחקים בלי יעד מתמטי: פחות תיקו</h2><p><b>מה התא עושה:</b> מציג שיעורי תוצאות כשהדגל המתמטי דולק אצל הבית או האורחת.</p><p><b>מה רואים:</b> תיקו יורד לכ-18.4% בבית מתה ולכ-18.1% באורחת מתה, לעומת כ-27.1% בכלל. אלה 174 ו-160 משחקים בהתאמה; חלקם חופפים.</p><p><b>הסבר נתונים:</b> זה קשר חלש: V של כ-0.056 וכ-0.041, ורק 245 משחקים עם לפחות קבוצה אחת מתה. הקבוצות אינן מוקצות באקראי לדגל; שלב העונה וחוזק הקבוצות עשויים להסביר חלק מהקשר.</p><p><b>הסבר כדורגלני:</b> השערה בלבד: הקבוצה שעוד נלחמת על יעד לוחצת לניצחון ולא מסתפקת בנקודה. זו שאלה ל-M1 במחברת 04 ורמז לבדיקה של התיקו ב-M3. מתאם, לא סיבתיות.</p></div>

In [ ]:
final_figures["c6b_outcome_dead"] = outcome_chart(
    final_review.outcomes,
    ("nothing_to_play", "away_nothing_to_play"),
    "אין על מה לשחק: פחות תיקו, קשר תיאורי בלבד",
)
final_figures["c6b_outcome_dead"].show()

<div dir="rtl"><h2>C12.6 — האם הקבוצות המתות פשוט חזקות או חלשות יותר</h2><p><b>מה התא עושה:</b> מחשב את ממוצע elo_diff בשורות הבית עבור בית מתה ועבור אורחת מתה, לצד שיעור התיקו ומספר המשחקים.</p><p><b>מה רואים:</b> פערי Elo ממוצעים קטנים יחסית, בכיוונים שונים. אין כאן קבוצה של פייבוריטיות מובהקות רק לפי הממוצע.</p><p><b>הסבר נתונים:</b> שתי ההשוואות מנקודת המבט של הבית. ממוצע קרוב לאפס אינו מבטל הבדלים בהתפלגות ואינו מחליף בקרה על חוזק, שלב וזהות הקבוצות; אין טענה סיבתית.</p><p><b>הסבר כדורגלני:</b> הבדיקה מחזקת את הצורך לחקור מוטיבציה מעבר לחוזק, אבל אינה מאשרת שהיעדר יעד הוא הסיבה לירידה בתיקו.</p></div>

In [ ]:
display(final_table(dead_game_summary(final_review)))

<div dir="rtl"><h2>C13 — מה מחברת 03 משאירה לשרשרת</h2><p><b>מה התא עושה:</b> מרכז את ארבעת הבלוקים, את prematch ושני סרגלי הבנצ׳מרק, ואת המעמדים שכבר אושרו.</p><p><b>מה רואים:</b> M0 מספק מידע סיבתי לפני המשחק; E0050 ו-Poisson נותנים נקודת ייחוס להמשך. value_avg ו-ewma5/venue הם ברירות המחדל המתועדות ל-M2.</p><p><b>הסבר נתונים:</b> סרגלי EXP-001 ו-EXP-002 הרשמיים מתחילים אימון ב-2006/2007, ולכן שונים מעט מייחוסי EXP-003. המעמד של מועמד אינו בחירה חדשה מתוך C5; לא הוצאה שום עמודה.</p><p><b>הסבר כדורגלני:</b> יש כעת בסיס שמפריד בין חוזק, התקפה והגנה, כושר והקשר אשתקד. M1 יבדוק מה מספרת חשיבות המשחק בנוסף לכל אלה.</p></div>

In [ ]:
display(final_table(m0_deliverables()))

<div dir="rtl"><h2>C13.1 — ההחלטות ה־ת1 עד ה־ת14</h2><p><b>מה התא עושה:</b> מציג תמצית של שורת החלטה אחת לכל החלטת תכנון, עם מקור ל-VISION §2.4.</p><p><b>מה רואים:</b> ההחלטות על Elo, דגלים, חלונות כושר, הקשר עונתי ועומק אימון; פירוט מחייב נשאר במסמך החזון.</p><p><b>הסבר נתונים:</b> זו תמצית קריאה בלבד, לא שינוי החלטות. ה־ת11 וה־ת12 מחדדות חלק מהניסוחים המוקדמים; ה־ת14 היא חריגה מתועדת מכלל EXP-003 המקורי.</p><p><b>הסבר כדורגלני:</b> המבנה מכוון לאותה שאלה: מה ידוע לפני המשחק, ומה באמת מוסיף מידע לתוצאה מעבר לחוזק שכבר נמדד.</p></div>

In [ ]:
display(final_table(planning_decisions()))

<div dir="rtl"><h2>C13.2 — מה נשאר לדיון, והגשר למחברת 04</h2>
<p><b>הסבר נתונים:</b> ממצאי C5 הם דיווח בלבד. לפני שינוי בהקצאת פיצ׳רים או באימון נדרש דיון התכנון של עמית ו-Claude, לפי T-010 §9:</p>
<ul><li><b>משחקים מתים:</b> פחות תיקו, מדגם קטן וקשר חלש; לבדוק מעבר לחוזק ולשלב.</li>
<li><b>עמודות כפולות:</b> ספירות המשחקים, שלב הבית והחוץ, דגלי העלייה/השלמה ושני מדדי Elo. הן נשארות בטבלה עד הכרעה.</li>
<li><b>מדדי t−1 מול Elo:</b> חפיפה גבוהה, ושווי שמתחזק בתקופה העכשווית. התרומה נבדקת ב-ablation של M2.</li>
<li><b>שתי השערות שלא נבדקו:</b> שווי כמבדיל בין עולות, ותרומת מדדי t−1 במחזורים הראשונים. אלה רעיונות להמשך, לא ממצאים.</li></ul>
<p><b>הסבר כדורגלני:</b> מחברת 04, M1 — מוטיבציה, מתחילה מהדגל המתמטי ״אין על מה לשחק״ ומהקשר החלש שנמצא לתיקו. היא תבדוק חשיבות משחק מעבר לציפיית החוזק, בלי להסיק שמתאם הוא סיבה.</p>
<p><b>מצב העבודה:</b> אין כאן בחירה חדשה, מחיקת פיצ׳ר או אימון מודל. ההמשך הוא דיון התכנון, סקירה משותפת של כל המחברת, ותיקונים מרוכזים לפי ההחלטות.</p></div>

<div dir="rtl"><h2>C14 — מספרי בקרה ושמירת גרפים</h2><p><b>מה התא עושה:</b> משווה את הספירות והמדדים לעוגני C5 ו-QA ושומר את שבעת הגרפים בשמות c6b_ חדשים.</p><p><b>מה רואים:</b> ה-SHA של prematch, ספירות הפיצ׳רים והזוגות, הקשרים המרביים ושיעורי התיקו במשחקים המתים. פלט הבקרות מחושב, ולא מועתק מהטקסט.</p><p><b>הסבר נתונים:</b> הקבצים הקפואים לא נכתבים מחדש. תאים 0–174 שומרים על אותו מקור. PNG ו-HTML של C6b נשמרים בנפרד, באותה תיקיית גרפים.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לפעולת השמירה; הבקרות מאפשרות לבדוק שכל מסקנה נשענת על המספרים שאומתו.</p></div>

In [ ]:
final_control_table = final_controls(final_review)
display(final_table(final_control_table))
display(HTML(f'<div dir="rtl">prematch_v1 SHA-256: <bdi dir="ltr">{final_sha}</bdi></div>'))
final_paths = save_final_figures(final_figures, Path("reports/figures/prematch_c"))
final_seconds = time.perf_counter() - final_started
display(
    HTML(
        f'<div dir="rtl">נשמרו {len(final_paths)} זוגות גרפים חדשים. '
        f"משך חלק C6b: {final_seconds:.2f} שניות.</div>"
    )
)

<div dir="rtl" style="text-align:right;line-height:1.7"><h2><bdi dir="ltr">S0</bdi> — השורה התחתונה</h2><p>כן. אנחנו משתמשים בלקחים ממחברת <bdi dir="ltr">02,</bdi> אבל שימוש במודל, בקרה הנדסית ותכנון עתידי הם דברים שונים.</p><p>מתוך <bdi dir="ltr">77</bdi> משפחות ממצאים: <bdi dir="ltr">14</bdi> בשימוש מודלי (<bdi dir="ltr">8</bdi> ישיר, <bdi dir="ltr">6</bdi> עקיף), <bdi dir="ltr">6</bdi> כלל הנדסי/בקרה, <bdi dir="ltr">25</bdi> נדחו, <bdi dir="ltr">29</bdi> מתוכננות ו־<bdi dir="ltr">3</bdi> ללא תוכנית. מקור: ספירת שורות <bdi dir="ltr">S2</bdi> כאן, ללא כפל סיכומים.</p><p>ממוצע נקודות לעונה — <bdi dir="ltr">A04</bdi> ו־<bdi dir="ltr">B01:</bdi> סרגל כיול מתוכנן ל־<bdi dir="ltr">M4</bdi> ול־<bdi dir="ltr">M5.</bdi></p><p>שערי זכות וחובה — <bdi dir="ltr">A05:</bdi> כבר נכנסים דרך <bdi dir="ltr">ewma5/venue.</bdi></p><p>יתרון בית — <bdi dir="ltr">B03</bdi> ו־<bdi dir="ltr">B05:</bdi> מתוכנן ל־<bdi dir="ltr">M2.</bdi></p><p>תיקו — <bdi dir="ltr">D05–D07</bdi> ו־<bdi dir="ltr">C10:</bdi> תיקון וכיול מתוכננים ל־<bdi dir="ltr">M3</bdi>; מונה סוף השלב של <bdi dir="ltr">C10</bdi> כבר קיים ב־<bdi dir="ltr">M0.</bdi></p><p><bdi dir="ltr">data</bdi> <bdi dir="ltr">scientist:</bdi> רוב הלקחים אינם פיצ׳רים למשחק, אלא סרגלים לרכיבים שעוד לא נבנו. קבוע שזהה בכל השורות אינו מבדיל בין משחקים.</p><p>כדורגל: ״אלופה צוברת כ־<bdi dir="ltr">2.28</bdi> נקודות למשחק״ בודק אם תחזית העונה סבירה, ולא מי ינצח במשחק הבא (<bdi dir="ltr">02,</bdi> <bdi dir="ltr">11–14,</bdi> <bdi dir="ltr">51–58).</bdi></p><p><bdi dir="ltr">S1</bdi> — מה מחברת <bdi dir="ltr">03</bdi> עשתה.</p><p><bdi dir="ltr">S2</bdi> — כל הממצאים, ספירות ורשימת המשך לפי מחברת; הפירוט מקופל.</p><p><bdi dir="ltr">S3</bdi> — רכיבים, קלטים, אימון ומשפחות פיצ׳רים.</p><p><bdi dir="ltr">S4</bdi> — פערים בין התוכנית למימוש.</p><p><bdi dir="ltr">S5</bdi> — שאלות לפי מועד ההכרעה.</p><p><bdi dir="ltr">S6</bdi> — הצעות של <bdi dir="ltr">Codex</bdi> לדיון, לא החלטות.</p><p><bdi dir="ltr">S7</bdi> — הקלט והשאלות הראשונות של מחברת <bdi dir="ltr">04.</bdi></p></div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S1</bdi> — מה מחברת <bdi dir="ltr">03</bdi> עשתה, בעמוד אחד</h2>
<p>זהו בסיס לדיון של עמית ו־<bdi dir="ltr">Claude,</bdi> ולא החלטות חדשות. המספור בכל הסיכום הוא אינדקס תא המתחיל באפס, כמו בפרומפט. מקור מספרי הסיכום הוא הפלט השמור של התאים המצוינים; אין כאן ניסוי או חישוב מחקרי חדש.</p>
<p>השרשרת עדיין בבנייה: <bdi dir="ltr">M0</bdi> והבנצ׳מרק המכויל קיימים; מודלי המוטיבציה, השערים הסופי, התוצאות, העונה, המאקרו והשילוב טרם נבנו. <bdi dir="ltr">Poisson</bdi> של <bdi dir="ltr">EXP-002</bdi> הוא סרגל ניסויי, ולא <bdi dir="ltr">M2</bdi> הסופי.</p>
<div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">חלק</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">השאלה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">התשובה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">המספר המרכזי ומקורו</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">ההחלטה</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תמונת טבלה לפני השריקה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מה באמת היה ידוע ביום המשחק, והאם נותר יעד פתוח?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">טבלה ספורטיבית לפי תאריך, חוקים ודגל שמרני; משחקי אותו יום אינם רואים זה את זה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">245</bdi> מתוך <bdi dir="ltr">4,315</bdi> משחקים (<bdi dir="ltr">5.68%)</bdi> עם לפחות קבוצה אחת בלי יעד; בעליון <bdi dir="ltr">11.6%</bdi> משורות הקבוצה אחרי הוספת קווי המקום. מחברת <bdi dir="ltr">03,</bdi> <bdi dir="ltr">18,</bdi> <bdi dir="ltr">29.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ת<bdi dir="ltr">3</bdi> וה-ת<bdi dir="ltr">10:</bdi> הוכחה מתמטית, בלי <bdi dir="ltr">Elo.</bdi> אין כאן הוכחה להשפעה על הביצועים.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חוזק מתוצאות והבקעה/ספיגה עדכניות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">איזה זיכרון של התוצאות וכושר מוסיפים מידע?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E0050</bdi> נבחר; התקפה והגנה <bdi dir="ltr">ewma5/venue</bdi> נבחרו בידי עמית בחריגה מתועדת; כושר בנקודות לא הוסיף בממוצע במסלול הסדור.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">Elo:</bdi> <bdi dir="ltr">log</bdi> <bdi dir="ltr">loss</bdi> <bdi dir="ltr">1.020424</bdi> מול <bdi dir="ltr">B0</bdi> <bdi dir="ltr">1.084974.</bdi> שערים: <bdi dir="ltr">NLL</bdi> <bdi dir="ltr">1.409455</bdi> מול <bdi dir="ltr">1.410383</bdi> של <bdi dir="ltr">Elo</bdi> בלבד; <bdi dir="ltr">0</bdi> מ־<bdi dir="ltr">22</bdi> חלונות נקודות מוסיפים בממוצע. מחברת <bdi dir="ltr">03,</bdi> <bdi dir="ltr">75,</bdi> <bdi dir="ltr">93,</bdi> <bdi dir="ltr">101,</bdi> <bdi dir="ltr">107.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ת<bdi dir="ltr">1,</bdi> ה-ת<bdi dir="ltr">2,</bdi> ה-ת<bdi dir="ltr">8,</bdi> ה-ת<bdi dir="ltr">11,</bdi> ה-ת<bdi dir="ltr">12</bdi>; הכרעות <bdi dir="ltr">EXP-001/002.</bdi> <bdi dir="ltr">NLL</bdi> ו־<bdi dir="ltr">log</bdi> <bdi dir="ltr">loss</bdi> אינם אותו מדד.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הקשר אשתקד וחיבור הבלוקים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מה <bdi dir="ltr">t−1</bdi> מוסיף מעל <bdi dir="ltr">Elo</bdi> והחלונות, ומה מקבל ההמשך?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שווי ממוצע הוא ברירת מחדל ל־<bdi dir="ltr">M2</bdi> בחריגה מתועדת; מיקום/נקודות/הפרש מועמדים. הבלוקים חוברו בלי לשנותם.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">prematch:</bdi> <bdi dir="ltr">8,630</bdi> שורות × <bdi dir="ltr">51</bdi> עמודות. בשווי, מעל מודל השערים ובקיפולים עם לפחות <bdi dir="ltr">4</bdi> עונות אימון: שיפור <bdi dir="ltr">0.00061</bdi> (<bdi dir="ltr">SE</bdi> <bdi dir="ltr">0.00027),</bdi> בדיקה בדיעבד. מחברת <bdi dir="ltr">03,</bdi> <bdi dir="ltr">171–172,</bdi> <bdi dir="ltr">178.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ת<bdi dir="ltr">13</bdi> וה-ת<bdi dir="ltr">14.</bdi> נשמר דיווח על כל הקיפולים; לא מסתירים את הכישלון לפי הכלל המקורי.</td></tr></tbody></table></div>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — ממצאי מחברת <bdi dir="ltr">02:</bdi> איפה כל אחד נמצא בשרשרת</h2>
<p>לא כל מה שלמדנו אמור להפוך לעמודת קלט. שערי זכות וחובה כבר נכנסים דרך חלונות ההתקפה וההגנה; נקודות והפרש נכנסים גם לטבלה ולזיכרון התוצאות. לעומתם, התפלגות תוצאות, ספי נקודות ופיזור טבלה הם בעיקר סרגלים לרכיבים שעוד לא נבנו. הממוצעים המסוימים של מחברת <bdi dir="ltr">02</bdi> אינם מוזרקים אוטומטית לפיצ׳רים ההיסטוריים.</p>
<p>יחידת המיפוי היא ממצא מובחן או משפחת סטטיסטיקות מאותו ניתוח. טבלה וגרף של אותו חישוב נספרים פעם אחת; פירוט מועדון־עונה וערכי ריחוף אינם מאות ממצאים נפרדים. המספרים להלן מתועדים במקור, כולל הסתייגויות; לא הוספנו חישובים. תאי הסיכום החוזרים והסינתזה מכוסים דרך מקורותיהם, ומפת הכיסוי המלאה נמצאת בדוח <bdi dir="ltr">C6c.</bdi></p>
<p>המעמד ״בשימוש ישיר: פיצ'ר או סרגל״ פירושו שימוש בפיצ׳ר, סרגל או ערך מודל שכבר קיימים. ״בשימוש ישיר: כלל הנדסי או בקרה״ פירושו חוקים, אימות, מפתחות או אוכלוסייה, והוא נספר בנפרד משימוש במודלים. ״בשימוש עקיף״ מציין רעיון/מידע שהוטמע בערך פתיחה, השלמה או ניסוי שכבר נערך; אינו טענה שהקבוע המספרי הועתק. ״מתוכנן״ דורש הפניה לתוכנית קיימת, ולא הצעה חדשה שלי. קיום פיצ׳ר אינו הוכחה שהמודל העתידי כבר נבנה.</p>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — כללי</h2>
<details><summary style="cursor:pointer;font-weight:bold;padding:12px;background:#edf3fa;border:1px solid #ccd7e4">כללי: <bdi dir="ltr">1</bdi> ממצאים (<bdi dir="ltr">0</bdi> ישיר: פיצ׳ר/סרגל, <bdi dir="ltr">1</bdi> הנדסי, <bdi dir="ltr">0</bdi> עקיף, <bdi dir="ltr">0</bdi> נדחו, <bdi dir="ltr">0</bdi> מתוכננים, <bdi dir="ltr">0</bdi> ללא תוכנית)</summary><div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הממצא — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור במחברת <bdi dir="ltr">02</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מעמד</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">איפה בשרשרת</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הערה / החלטה</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">G01</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מדגם המחקר: <bdi dir="ltr">18</bdi> עונות פיתוח, <bdi dir="ltr">4,315</bdi> משחקים, <bdi dir="ltr">8,630</bdi> שורות קבוצה־משחק ו־<bdi dir="ltr">252</bdi> קבוצות־עונות. בעידן <bdi dir="ltr">14:</bdi> <bdi dir="ltr">12</bdi> עונות ו־<bdi dir="ltr">168</bdi> קבוצות־עונות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">פתיחה, <bdi dir="ltr">0,</bdi> <bdi dir="ltr">5–6</bdi>; פרק <bdi dir="ltr">A,</bdi> <bdi dir="ltr">25–26</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: כלל הנדסי או בקרה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0:</bdi> מפתחות, תחום פיתוח ובקרות נפח</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין כאן <bdi dir="ltr">8,630</bdi> משחקים עצמאיים; רוב <bdi dir="ltr">A</bdi> ו־<bdi dir="ltr">C</bdi> בעידן <bdi dir="ltr">14,</bdi> ואילו <bdi dir="ltr">B,</bdi> <bdi dir="ltr">D</bdi> ו־<bdi dir="ltr">E</bdi> משתמשים גם במדגמים אחרים המפורטים במקור.</td></tr></tbody></table></div></details>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — <bdi dir="ltr">A</bdi></h2>
<details><summary style="cursor:pointer;font-weight:bold;padding:12px;background:#edf3fa;border:1px solid #ccd7e4">פרק <bdi dir="ltr">A:</bdi> <bdi dir="ltr">19</bdi> ממצאים (<bdi dir="ltr">4</bdi> ישיר: פיצ׳ר/סרגל, <bdi dir="ltr">1</bdi> הנדסי, <bdi dir="ltr">4</bdi> עקיף, <bdi dir="ltr">7</bdi> נדחו, <bdi dir="ltr">2</bdi> מתוכננים, <bdi dir="ltr">1</bdi> ללא תוכנית)</summary><div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הממצא — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור במחברת <bdi dir="ltr">02</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מעמד</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">איפה בשרשרת</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הערה / החלטה</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A01</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">דוגמת <bdi dir="ltr">2023/2024:</bdi> מכבי ת״א <bdi dir="ltr">85</bdi> נקודות; חיפה <bdi dir="ltr">76</bdi> ספורטיביות מול <bdi dir="ltr">74</bdi> רשמיות; בית״ר <bdi dir="ltr">41</bdi> פחות <bdi dir="ltr">5</bdi> = <bdi dir="ltr">36</bdi> רשמיות. ארבע קבוצות עם הורדות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">7–8</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: כלל הנדסי או בקרה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0/rules.py:</bdi> בסיס ספורטיבי ואימות רשמי</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">3.</bdi> זהות האלופה והמיקום הרשמי במחברת <bdi dir="ltr">02</bdi> אינם יעד ספורטיבי אוטומטי ל־<bdi dir="ltr">M5/M6.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בדף העונה מכבי ת״א: משחק ללא ספיגה <bdi dir="ltr">44.4%,</bdi> ללא הבקעה <bdi dir="ltr">11.1%,</bdi> <bdi dir="ltr">BTTS</bdi> <bdi dir="ltr">47.2%,</bdi> שלושה שערים ומעלה <bdi dir="ltr">50%</bdi>; רצף ללא הפסד <bdi dir="ltr">14</bdi> וללא ניצחון <bdi dir="ltr">3.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">7–8</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא בשימוש ואין תוכנית</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שיעורי דוגמה ורצפים, לא פיצ׳רים מוגדרים בשרשרת</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שיעורי האפסים והרצפים בדוגמה אינם פיצ׳רים שנבחרו, ואין תוכנית ייעודית להם. <bdi dir="ltr">gf/ga</bdi> נרשמו בנפרד ב־<bdi dir="ltr">A05</bdi>; מסקנת הסגנון הכללית ב־<bdi dir="ltr">A11.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A03</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מכבי חיפה עלתה מ־<bdi dir="ltr">1.15</bdi> ל־<bdi dir="ltr">1.61</bdi> נקודות למשחק בין <bdi dir="ltr">2017/2018</bdi> ל־<bdi dir="ltr">2018/2019.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">9–10</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר ״מכבי חיפה״</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בלוק ⛔ נשמר לתיעוד בלבד; אין הסקת תכונת מועדון מדוגמה יחידה. <bdi dir="ltr">FEATURES:</bdi> מזהה קבוצה כקטגוריה נדחה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אלופה: כ־<bdi dir="ltr">2.28</bdi> נקודות למשחק; מקום <bdi dir="ltr">9</bdi> כ־<bdi dir="ltr">1.19</bdi> ומקום <bdi dir="ltr">11</bdi> כ־<bdi dir="ltr">1.12,</bdi> פער של כ־<bdi dir="ltr">2–3</bdi> נקודות בעונה. מקום <bdi dir="ltr">7</bdi> עשוי לצבור יותר ממקום <bdi dir="ltr">6</bdi> בגלל לוח הפלייאוף.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">11–14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M4</bdi> בדיקות כיול, מחברת <bdi dir="ltr">07</bdi>; <bdi dir="ltr">M5</bdi> הקשר טבלה, <bdi dir="ltr">08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא מכניסים ״ממוצע אלופה״ לכל משחק ולא מנרמלים ידנית את טבלת הסימולציה אליו. פירוט טווחי היעדים ב־<bdi dir="ltr">B01.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שערי זכות וחובה אחרי מרכוז עונתי: מתאם <bdi dir="ltr">−0.642</bdi>; שלוש הראשונות מבקיעות <bdi dir="ltr">1.77</bdi> לעומת <bdi dir="ltr">0.90</bdi> בשלוש האחרונות. בדוגמת מכבי ת״א <bdi dir="ltr">2023/2024:</bdi> <bdi dir="ltr">2.08</bdi> זכות ו־<bdi dir="ltr">0.69</bdi> חובה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">7–8,</bdi> <bdi dir="ltr">15–16</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: פיצ'ר או סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0</bdi> <bdi dir="ltr">form_v1</bdi> ו־<bdi dir="ltr">prematch:</bdi> <bdi dir="ltr">gf/ga</bdi> נפרדים של הקבוצה והיריבה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ת<bdi dir="ltr">8</bdi> וה-ת<bdi dir="ltr">12</bdi>; נבחר <bdi dir="ltr">ewma5/venue.</bdi> המספרים התיאוריים עצמם אינם מקדמי המודל ולא λ.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חוזק אשתקד קשור לעונה הבאה: הפרש שערים למשחק <bdi dir="ltr">r=0.760</bdi> לעומת נקודות <bdi dir="ltr">r=0.750,</bdi> על <bdi dir="ltr">132</bdi> זוגות עונות סמוכות בעידן <bdi dir="ltr">14.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">15–16,</bdi> <bdi dir="ltr">25–30</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: פיצ'ר או סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0</bdi> <bdi dir="ltr">season_context:</bdi> <bdi dir="ltr">prev_gd_per_game</bdi> ו־<bdi dir="ltr">prev_ppm</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הם קיימים אך נשארים 🔬 ב־<bdi dir="ltr">M2</bdi>; <bdi dir="ltr">EXP-003</bdi> הראה שרוב המידע חופף ל־<bdi dir="ltr">Elo.</bdi> קיום הפיצ׳ר אינו הוכחת תוספת.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ביתיות לקבוצה־עונה: פער ממוצע כ־<bdi dir="ltr">0.19</bdi> נקודות למשחק, התמדה <bdi dir="ltr">−0.03</bdi>; סטיית תקן נצפית <bdi dir="ltr">0.443</bdi> לעומת <bdi dir="ltr">0.445</bdi> בקירוב מזל. בקבוצות עם לפחות שש עונות: <bdi dir="ltr">0.17</bdi> לעומת <bdi dir="ltr">0.14.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">17–18</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר ביתיות אישי למועדון</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES:</bdi> ביתיות לכל קבוצה נדחתה; ביתיות אחת לליגה מתוכננת ל־<bdi dir="ltr">M2</bdi> לפי ה-ס<bdi dir="ltr">2.</bdi> ממוצע באר שבע <bdi dir="ltr">0.51</bdi> אינו כוח בית אישי שאומץ.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">פער הביתיות לפי מיקום: כ־<bdi dir="ltr">0.38</bdi> בשלוש הראשונות, <bdi dir="ltr">0.22</bdi> במקומות <bdi dir="ltr">4–8</bdi> ו־<bdi dir="ltr">0.05</bdi> במקומות <bdi dir="ltr">9–11.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">17–18</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M2,</bdi> מחברת <bdi dir="ltr">05:</bdi> ביתיות כפלית אחת; אינטראקציה רק ב־<bdi dir="ltr">ablation</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בחירה לפי מיקום סופי היא בדיעבד. אין שלוש ביתיויות שונות בייצור; ראו בדיקת הפער ההוגנת יותר בפרק <bdi dir="ltr">E.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ניצחונות בהפרש לפחות שלושה: <bdi dir="ltr">15.8%</bdi> בשלוש הראשונות מול <bdi dir="ltr">2.3%</bdi> בשלוש האחרונות. התמדה של חלק הניצחונות הגדולים מכל הניצחונות <bdi dir="ltr">0.212,</bdi> מול <bdi dir="ltr">0.720</bdi> לשיעור הניצחונות; התמדה של ניצחונות צרים <bdi dir="ltr">0.006.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">19–20</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר ״מנצחת בגדול/יודעת לנצח בקטן״</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13:</bdi> סגנון נדחה. מכפיל הפרש השערים ב־<bdi dir="ltr">Elo</bdi> הוא מנגנון אחר שנבדק ב־<bdi dir="ltr">EXP-001,</bdi> ולא שימוש בשיעור עונתי זה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A10</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ניצחונות צרים קשורים לשארית נקודות באותה עונה (<bdi dir="ltr">r</bdi>≈<bdi dir="ltr">0.70),</bdi> אך <bdi dir="ltr">R</bdi>² לעונה הבאה נשאר כ־<bdi dir="ltr">0.578</bdi> גם כשמוסיפים אותם להפרש השערים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">19–20</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר ״ווינריות״</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שארית באותה עונה אינה הוכחה ליכולת מתמידה; ה-ס<bdi dir="ltr">6</bdi> וה-ס<bdi dir="ltr">13.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A11</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">BTTS</bdi> כ־<bdi dir="ltr">46.4%</bdi> ושלושה שערים ומעלה כ־<bdi dir="ltr">44%</bdi>; התמדה <bdi dir="ltr">0.01</bdi> ו־<bdi dir="ltr">0.163.</bdi> אחרי שליטה ב־<bdi dir="ltr">gf/ga:</bdi> <bdi dir="ltr">0.02</bdi> ו־<bdi dir="ltr">0.09.</bdi> <bdi dir="ltr">R</bdi>² מההתקפה וההגנה: <bdi dir="ltr">0.440</bdi> ו־<bdi dir="ltr">0.758</bdi>; פיזור <bdi dir="ltr">BTTS</bdi> <bdi dir="ltr">0.098</bdi> מול מזל <bdi dir="ltr">0.083.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">21–24</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר ״קבוצה פתוחה״ נפרד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13.</bdi> גם ממוצעי מועדונים <bdi dir="ltr">42.3%–50%</bdi> אינם ראיה לסגנון עצמאי. <bdi dir="ltr">M3</bdi> יוכל לגזור הסתברויות כאלה בעתיד; טרם מומש.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A12</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">132</bdi> זוגות השורדות: מתאם נקודות <bdi dir="ltr">0.75,</bdi> ספירמן <bdi dir="ltr">0.60</bdi> ושיפוע <bdi dir="ltr">0.84</bdi> (<bdi dir="ltr">CI</bdi> <bdi dir="ltr">0.75–0.95)</bdi>; פיזור סביב הקו <bdi dir="ltr">0.28</bdi> נקודות למשחק. מתאם מיקום <bdi dir="ltr">0.56.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">25–30</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש עקיף</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נסיגה בקיץ ב־<bdi dir="ltr">Elo</bdi> וערכי פתיחה ב־<bdi dir="ltr">form</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">השיפוע נתן ערך רשת של נסיגה <bdi dir="ltr">0.16.</bdi> <bdi dir="ltr">EXP-001</bdi> בחר <bdi dir="ltr">0.33</bdi>; אין שימוש מכני ב־<bdi dir="ltr">84%</bdi> בכל רכיב. מתאם אינו דיוק חיזוי.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A13</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">המדגם הרציף חזק מכלל הליגה: ממוצע אשתקד <bdi dir="ltr">1.43</bdi> לעומת <bdi dir="ltr">1.35</bdi> בכלל הליגה; בעונה הבאה <bdi dir="ltr">1.40.</bdi> באר שבע בדוגמה ירדה מ־<bdi dir="ltr">2.14</bdi> ל־<bdi dir="ltr">1.69.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">25–30</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש עקיף</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0:</bdi> פתיחה והשלמה נפרדות לעולות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">המספרים מלמדים למה לא לאתחל עולה כאחת השורדות. הקבועים המלאים של מחברת <bdi dir="ltr">02</bdi> אינם מוזרקים לערכי הפתיחה של עונות מוקדמות.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נתח שווי קודם מול ביצועים: <bdi dir="ltr">r=0.673,</bdi> לעומת <bdi dir="ltr">0.735</bdi> לשווי באותה עונה; דוגמת חיפה <bdi dir="ltr">2017/2018:</bdi> <bdi dir="ltr">1.15</bdi> מול <bdi dir="ltr">1.62</bdi> בקו התיאורי, שארית <bdi dir="ltr">−0.47.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">31–36</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: פיצ'ר או סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0:</bdi> נתח שווי <bdi dir="ltr">t−1</bdi>; <bdi dir="ltr">M2</bdi> ברירת מחדל <bdi dir="ltr">value_avg</bdi> לפי ה-ת<bdi dir="ltr">14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">02</bdi> נבדק שווי כולל, וב־<bdi dir="ltr">03</bdi> נבחר ממוצע: ההגדרות אינן זהות. שווי העונה עצמה דולף; השארית אינה הוכחה לאימון או ניהול חלשים.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A15</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בחיזוי <bdi dir="ltr">EDA</bdi> רטרוספקטיבי: <bdi dir="ltr">RMSE</bdi> שווי קודם <bdi dir="ltr">0.314</bdi> (לוג <bdi dir="ltr">0.319),</bdi> הפרש קודם <bdi dir="ltr">0.274,</bdi> הפרש+שווי <bdi dir="ltr">0.267</bdi> ונקודות <bdi dir="ltr">0.279.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">31–36</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש עקיף</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הרקע ל־<bdi dir="ltr">EXP-003:</bdi> לבד ומעל חוזק</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">LOSO</bdi> מחזיק עונה בחוץ אך עשוי להתאמן על עתידה. אלה אינם ציוני <bdi dir="ltr">walk-forward</bdi> ולא סרגלי המערכת הסופיים.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A16</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">גודל סגל מול נקודות: <bdi dir="ltr">r=−0.123,</bdi> <bdi dir="ltr">p=0.1125.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">31–36</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר גודל סגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> וה-ס<bdi dir="ltr">13:</bdi> אין תוספת מבוססת; דף השווי כולל תחלופה ואינו הרכב ידוע למשחק.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A17</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כלל העולות ב־<bdi dir="ltr">36</bdi> ערכי שווי ידועים: חציון החצי התחתון גבוה מדי ב־<bdi dir="ltr">72.2%</bdi>; אחוזון <bdi dir="ltr">30</bdi> ב־<bdi dir="ltr">44.4%.</bdi> הטיה חציונית <bdi dir="ltr">−0.39→+0.03</bdi> נ״א ו־<bdi dir="ltr">MAE</bdi> <bdi dir="ltr">0.89→0.79</bdi> נ״א. מוקדם: <bdi dir="ltr">75%→40%</bdi>; מאוחר: <bdi dir="ltr">68.8%→50%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">37–42</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: פיצ'ר או סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0</bdi> <bdi dir="ltr">season_context:</bdi> השלמת שווי מאשתקד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">1</bdi> וה-ת<bdi dir="ltr">13.</bdi> מכבי פ״ת <bdi dir="ltr">2020/2021</bdi> עם נתח <bdi dir="ltr">7.8%</bdi> מזכירה שעולה אינה תמיד זולה. מקור חסר אחד אינו אפס; הכלל משתמש רק ב־<bdi dir="ltr">t−1.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A18</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בעידן <bdi dir="ltr">14:</bdi> <bdi dir="ltr">23</bdi> עולות, <bdi dir="ltr">7</bdi> ירדו מיד; <bdi dir="ltr">1.07</bdi> נקודות למשחק מול <bdi dir="ltr">1.40</bdi> לאחרות, הפרש שערים <bdi dir="ltr">−0.44</bdi> וסטיית תקן נקודות <bdi dir="ltr">0.22</bdi>; מיקום שיא רשמי <bdi dir="ltr">5.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">43–44</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש עקיף</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0:</bdi> דגל עולה וערכי פתיחה/השלמות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">5.</bdi> אלה <bdi dir="ltr">23</bdi> אירועים בעידן <bdi dir="ltr">14,</bdi> לעומת <bdi dir="ltr">37</bdi> בפיתוח כולו ב־<bdi dir="ltr">03.</bdi> לא כל עולה מקבלת בדיעבד את ממוצע כל האירועים.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A19</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">היסטוריית מועדון מעבר לשנה: ב־<bdi dir="ltr">125</bdi> מקרים <bdi dir="ltr">RMSE</bdi> הפרש <bdi dir="ltr">0.275→0.274</bdi>; עם שווי <bdi dir="ltr">0.268→0.270.</bdi> יש <bdi dir="ltr">17</bdi> מועדונים עם לפחות שלוש עונות, ומתוכם שלושה שעלו לפחות פעמיים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">45–48</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר היסטוריה ארוכה/״יו־יו״ נפרד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES:</bdi> נדחה מחוסר תוספת. זיכרון <bdi dir="ltr">Elo</bdi> מתוצאות אינו זהה לתוספת הממוצע הרב־שנתי שנבדקה כאן.</td></tr></tbody></table></div></details>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — <bdi dir="ltr">B</bdi></h2>
<details><summary style="cursor:pointer;font-weight:bold;padding:12px;background:#edf3fa;border:1px solid #ccd7e4">פרק <bdi dir="ltr">B:</bdi> <bdi dir="ltr">13</bdi> ממצאים (<bdi dir="ltr">1</bdi> ישיר: פיצ׳ר/סרגל, <bdi dir="ltr">1</bdi> הנדסי, <bdi dir="ltr">0</bdi> עקיף, <bdi dir="ltr">4</bdi> נדחו, <bdi dir="ltr">6</bdi> מתוכננים, <bdi dir="ltr">1</bdi> ללא תוכנית)</summary><div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הממצא — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור במחברת <bdi dir="ltr">02</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מעמד</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">איפה בשרשרת</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הערה / החלטה</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B01</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בעידן <bdi dir="ltr">14:</bdi> אלופה <bdi dir="ltr">2.283</bdi>±<bdi dir="ltr">0.125</bdi> נקודות למשחק; קו עליון <bdi dir="ltr">1.415</bdi>±<bdi dir="ltr">0.073,</bdi> שורדת <bdi dir="ltr">1.023</bdi>±<bdi dir="ltr">0.074,</bdi> היורדת עם הכי הרבה נקודות <bdi dir="ltr">0.982</bdi>±<bdi dir="ltr">0.123.</bdi> חציוני נקודות וטווחים: <bdi dir="ltr">83.5</bdi> (<bdi dir="ltr">71–89),</bdi> <bdi dir="ltr">37</bdi> (<bdi dir="ltr">33–40),</bdi> <bdi dir="ltr">34</bdi> (<bdi dir="ltr">28–37),</bdi> <bdi dir="ltr">33</bdi> (<bdi dir="ltr">22–38).</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">51–58</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M4</bdi> בדיקות כיול, <bdi dir="ltr">07</bdi>; <bdi dir="ltr">M1</bdi> מעטפת יעדים, <bdi dir="ltr">04</bdi>; <bdi dir="ltr">M5,</bdi> <bdi dir="ltr">08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">זהות היעדים רשמית והנקודות ספורטיביות. הטווחים אינם קווי הכרעה מתמטיים. מבחני מגמה <bdi dir="ltr">p=0.489/0.259/0.909/0.531</bdi> לא זיהו מגמה ברורה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שוויון בנקודות על קו: <bdi dir="ltr">0</bdi> אליפויות, <bdi dir="ltr">1</bdi> בקו העליון מתוך <bdi dir="ltr">15</bdi> עונות רלוונטיות, <bdi dir="ltr">5</bdi> בקו הירידה מתוך <bdi dir="ltr">18.</bdi> ב־<bdi dir="ltr">2016/2017</bdi> הפועל ת״א צברה <bdi dir="ltr">38</bdi> על המגרש מול <bdi dir="ltr">32</bdi> לשורדת, וירדה עקב הורדה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">59–62</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: כלל הנדסי או בקרה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">rules.py</bdi> ואימות ספורטיבי/רשמי ב־<bdi dir="ltr">M0</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר ״כמה שוויונות היו״; השימוש הישיר הוא כלל שוברי השוויון ובסיס הנקודות. אין להעתיק זהות יורדת רשמית ליעד ספורטיבי.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B03</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ביתיות ליגה: פער נקודות בית–חוץ <bdi dir="ltr">0.378</bdi> ב־<bdi dir="ltr">2009/2010–2014/2015</bdi> מול <bdi dir="ltr">0.135</bdi> מאז <bdi dir="ltr">2015/2016</bdi>; הפרש <bdi dir="ltr">0.243</bdi> (<bdi dir="ltr">SE</bdi> <bdi dir="ltr">0.046).</bdi> סטיות בין עונות <bdi dir="ltr">0.073/0.104</bdi> מול רעש עונתי כ־<bdi dir="ltr">0.165.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">63–66</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M2,</bdi> מחברת <bdi dir="ltr">05:</bdi> ביתיות ליגה כפלית ודעיכת זמן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">2</bdi> וה-ת<bdi dir="ltr">7</bdi>; תקופות נבחרו בדיעבד. <bdi dir="ltr">0.19</bdi> בפרק <bdi dir="ltr">A</bdi> הוא מדגם אחר, לא אותה ביתיות. <bdi dir="ltr">HFA=30</bdi> של <bdi dir="ltr">Elo</bdi> אינו אומדן זהה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אחרי <bdi dir="ltr">Elo</bdi> ובתוך עונות: <bdi dir="ltr">GHOST</bdi> <bdi dir="ltr">n=252</bdi> אפקט <bdi dir="ltr">+0.144</bdi>; <bdi dir="ltr">LIMITED</bdi> <bdi dir="ltr">n=125</bdi> <bdi dir="ltr">−0.009</bdi>; <bdi dir="ltr">SHARED</bdi> <bdi dir="ltr">n=206</bdi> <bdi dir="ltr">−0.277</bdi>; <bdi dir="ltr">RELOCATED</bdi> <bdi dir="ltr">n=172</bdi> <bdi dir="ltr">−0.047.</bdi> אף השוואה לא עברה תיקון <bdi dir="ltr">Bonferroni</bdi>; <bdi dir="ltr">SHARED</bdi> <bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.11.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">67–71</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין דגל קהל/אצטדיון משותף או שאול במודלים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">2</bdi> וה-ס<bdi dir="ltr">13.</bdi> <bdi dir="ltr">NORMAL</bdi> <bdi dir="ltr">n=3,409</bdi> עם פער גולמי <bdi dir="ltr">0.271</bdi>; רווחים רחבים, ולא הוכחה שהאפקט בדיוק אפס. בתא <bdi dir="ltr">68</bdi> נשאר ניסוח מועמדות מיושן; <bdi dir="ltr">S4.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">גם ב־<bdi dir="ltr">NORMAL</bdi> בלבד הביתיות ירדה: <bdi dir="ltr">0.434→0.106,</bdi> הפרש <bdi dir="ltr">0.328</bdi> (<bdi dir="ltr">SE</bdi> <bdi dir="ltr">0.070).</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">67–71</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M2:</bdi> אותו ניסוי ביתיות ודעיכה, <bdi dir="ltr">05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מחזק בדיקה של שינוי ליגה ולא הוספת דגלי קהל. אין הסבר סיבתי מוכח לשינוי.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בעידני <bdi dir="ltr">12/16/14:</bdi> שערים למשחק <bdi dir="ltr">2.33/2.58/2.46</bdi>; תיקו <bdi dir="ltr">28.8%/26.6%/26.9%</bdi>; הפרש לפחות שלושה <bdi dir="ltr">11.1%/15.0%/14.0%</bdi>; פיזור נקודות למשחק <bdi dir="ltr">0.39/0.40/0.42.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">72–75</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: פיצ'ר או סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0:</bdi> פורמט וגודל ליגה; הכנה לאימון יחסי של <bdi dir="ltr">M2/M5</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">7</bdi>; <bdi dir="ltr">all-development</bdi> אינו עידן <bdi dir="ltr">14</bdi> בלבד. אין כלל שמחייב כל λ להשתוות ל־<bdi dir="ltr">2.46</bdi> או כל עידן לקבל מקדם מהטבלה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא נמצאה מגמת תיקו כוללת (<bdi dir="ltr">p=0.312)</bdi>; <bdi dir="ltr">0:0</bdi> ירד בכ־<bdi dir="ltr">0.251</bdi> נ״א לעונה (<bdi dir="ltr">p=0.0053,</bdi> מתוקן <bdi dir="ltr">0.0422).</bdi> בעידן <bdi dir="ltr">14</bdi> לבדו <bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.0977.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">76–79</bdi>; <bdi dir="ltr">D,</bdi> <bdi dir="ltr">158–159</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3,</bdi> מחברת <bdi dir="ltr">06:</bdi> כיול תיקו וניסוי דעיכה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין להציג את הירידה כמובהקת בכל תת־תקופה; הצורה והקצב ייבדקו כרונולוגית, לא מקדם קבוע מה־<bdi dir="ltr">EDA.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מגמות אירועים: שערי פנדל עלו מכ־<bdi dir="ltr">0.16</bdi> לכ־<bdi dir="ltr">0.25</bdi> למשחק; מגמת אדומים עם <bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.04</bdi> לא שרדה תיקון ריבוי השוואות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">76–79</bdi>; <bdi dir="ltr">D,</bdi> <bdi dir="ltr">196–211</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳רי פנדל/אדום צפוי</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; אלה אירועי תוך־משחק. יש בדיקה חוזרת בתוך בלוק <bdi dir="ltr">D</bdi> שנשמר לתיעוד.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סטיית תקן בין קבוצות בסוף העונה: <bdi dir="ltr">0.419</bdi> נקודות למשחק, בטווח עונתי <bdi dir="ltr">0.31–0.49.</bdi> קירוב מזל <bdi dir="ltr">0.223,</bdi> יתרה <bdi dir="ltr">0.353</bdi> וכ־<bdi dir="ltr">29.6%</bdi> מהשונות מיוחסים למזל בקירוב; מגמה <bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.19.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">80–81</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M4</bdi> בדיקת פיזור תחזית, מחברת <bdi dir="ltr">07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">זה פירוק בקירוב, לא מדידה סיבתית של יכולת. לא מכפילים סימולציה ב״גורם כאוס״ ולא כופים פיזור <bdi dir="ltr">0.42</bdi> בכל עונה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B10</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מכבי ת״א וחיפה זכו יחד ב־<bdi dir="ltr">11</bdi> מ־<bdi dir="ltr">18</bdi> אליפויות (<bdi dir="ltr">6+5)</bdi>; בסך הכול שש אלופות שונות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">82–83</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא בשימוש ואין תוכנית</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תיעוד ההיסטוריה בלבד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר מספר אליפויות/שם מועדון ואין תוכנית לנצל ספירה זו במודל. זיכרון תוצאות <bdi dir="ltr">Elo</bdi> אינו שקול להזרקת מספר תארים.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B11</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הטובה ספורטיבית אשתקד הייתה האלופה הרשמית ב־<bdi dir="ltr">9/17</bdi> עונות; <bdi dir="ltr">14/17</bdi> אלופות היו בשלוש הראשונות אשתקד. בעידן <bdi dir="ltr">14:</bdi> <bdi dir="ltr">7/12</bdi> ו־<bdi dir="ltr">11/12.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">84–85</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B1</bdi> בתחילת עונה, מחברות <bdi dir="ltr">07–09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">8</bdi>; נדרש בנצ׳מרק על יעד ספורטיבי מחושב. ה־<bdi dir="ltr">9/17</bdi> הוא תיאור <bdi dir="ltr">EDA</bdi> עם זהות אלופה רשמית, לא פלט תחזית ספורטיבי שכבר נשמר.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B12</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שערים בשבת <bdi dir="ltr">2.45</bdi> (<bdi dir="ltr">n=2,735)</bdi> מול שני <bdi dir="ltr">2.48</bdi> (<bdi dir="ltr">n=528)</bdi>; בשעה <bdi dir="ltr">20+</bdi> כ־<bdi dir="ltr">2.55</bdi> מול <bdi dir="ltr">2.41</bdi> בשעות <bdi dir="ltr">18–20.</bdi> ינואר <bdi dir="ltr">2.39</bdi> (<bdi dir="ltr">n=555),</bdi> מאי <bdi dir="ltr">2.72</bdi> (<bdi dir="ltr">n=380),</bdi> ובמאי תיקו <bdi dir="ltr">20.8%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">86–91</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳רי יום/שעה/חודש</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES:</bdi> מחזור ושלב עדיפים. שיבוץ יריבות ותקופת העונה מעורבים; יוני בארבע עונות ויולי באחת אינם ראיה עונתית יציבה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B13</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">49</bdi> משחקים נדחים: <bdi dir="ltr">53.1%</bdi> ניצחונות בית ו־<bdi dir="ltr">2.65</bdi> שערים מול <bdi dir="ltr">40.3%</bdi> ו־<bdi dir="ltr">2.46</bdi> בשאר <bdi dir="ltr">4,266.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">92–93</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר דגל דחייה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">המדגם קטן ולא תוקנן לחוזק וליריבות. ההנדסה כן מטפלת בדחיות דרך <bdi dir="ltr">as_of</bdi> ו־<bdi dir="ltr">games_behind</bdi>; זו אינה הצדקה לאפקט חיזוי של הדגל.</td></tr></tbody></table></div></details>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — <bdi dir="ltr">C</bdi></h2>
<details><summary style="cursor:pointer;font-weight:bold;padding:12px;background:#edf3fa;border:1px solid #ccd7e4">פרק <bdi dir="ltr">C:</bdi> <bdi dir="ltr">14</bdi> ממצאים (<bdi dir="ltr">2</bdi> ישיר: פיצ׳ר/סרגל, <bdi dir="ltr">1</bdi> הנדסי, <bdi dir="ltr">0</bdi> עקיף, <bdi dir="ltr">0</bdi> נדחו, <bdi dir="ltr">11</bdi> מתוכננים, <bdi dir="ltr">0</bdi> ללא תוכנית)</summary><div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הממצא — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור במחברת <bdi dir="ltr">02</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מעמד</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">איפה בשרשרת</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הערה / החלטה</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C01</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כ־<bdi dir="ltr">3.4%</bdi> משורות הדירוג כוללות שוויון חלקי; רגישות לסילוק שוויונות שינתה שיעורי הצלחה לכל היותר בכ־<bdi dir="ltr">0.02.</bdi> הפלייאוף האמצעי כולל רק <bdi dir="ltr">12</bdi> משחקים בפיתוח.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">95–97,</bdi> <bdi dir="ltr">102–103</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: כלל הנדסי או בקרה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0/rules.py:</bdi> שוויונות ופורמט; בקרה על הגדרת האוכלוסייה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">02</bdi> דירוג ממוצע לצורכי <bdi dir="ltr">EDA</bdi> אינו סדר שוברי השוויון של הייצור. אין ללמוד אפקט אמצעי ממדגם זה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אחרי עשרה משחקים: מתאם דירוג לסיום <bdi dir="ltr">0.683</bdi> (<bdi dir="ltr">CI</bdi> <bdi dir="ltr">0.613–0.752)</bdi>; <bdi dir="ltr">26/36</bdi> בשלוש הראשונות נשארו שם. ממוצע תזוזה כ־<bdi dir="ltr">0.83</bdi> למקומות <bdi dir="ltr">1–2</bdi> מול <bdi dir="ltr">3.92</bdi> למקום <bdi dir="ltr">9</bdi> ו־<bdi dir="ltr">3.25</bdi> למקום <bdi dir="ltr">14.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">98–101</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M5</bdi> פיזור תלוי מקום ו־<bdi dir="ltr">k</bdi>; <bdi dir="ltr">M4</bdi> בדיקת כיול, <bdi dir="ltr">07–08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">9.</bdi> <bdi dir="ltr">k</bdi> ב־<bdi dir="ltr">02</bdi> הוא מספר משחקים לקבוצה ולא תאריך משותף; מודלי ההמשך יקבלו טבלה לפי <bdi dir="ltr">as_of.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C03</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתאם הדירוג גדל מכ־<bdi dir="ltr">0.68</bdi> אחרי עשרה משחקים לכ־<bdi dir="ltr">0.95</bdi> בפיצול. הצמרת עוברת לראשונה <bdi dir="ltr">80%</bdi> שימור אזור אחרי <bdi dir="ltr">14</bdi> משחקים; אזורי הקו והתחתית אחרי <bdi dir="ltr">23,</bdi> והאמצע בפיצול. תזוזות ב־<bdi dir="ltr">k=10:</bdi> <bdi dir="ltr">1.22/2.56/2.62/2.54</bdi> לפי אזור מול אקראי <bdi dir="ltr">4.64.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">104–107</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M5/M6:</bdi> אי־ודאות לפי אזור ומועד; <bdi dir="ltr">M4</bdi> בדיקת כיול</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">9.</bdi> חלק מהעלייה בשימור ליד הפיצול מכני: משתתפת עליון לא יכולה לסיים מתחת למקום <bdi dir="ltr">6.</bdi> המועדים אינם חוקים קשיחים או פיצ׳רי עתיד.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מובילת <bdi dir="ltr">k=10</bdi> הייתה אלופה ב־<bdi dir="ltr">9/12</bdi> (<bdi dir="ltr">75%),</bdi> וב־<bdi dir="ltr">k=20</bdi> ב־<bdi dir="ltr">11/12</bdi> (<bdi dir="ltr">91.7%)</bdi>; האחרונה ירדה ב־<bdi dir="ltr">4/12</bdi> וב־<bdi dir="ltr">5/12.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">104–107</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B1</bdi> באמצע עונה, מחברות <bdi dir="ltr">07–09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">טבלה נוכחית כטבלת סיום היא סרגל רציני; אין להפוך <bdi dir="ltr">75%</bdi> להסתברות מכוילת לכל מובילה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מהפיצול לסיום: בעליון <bdi dir="ltr">68.1%</bdi> ללא תזוזה, בתחתון <bdi dir="ltr">33.3%</bdi>; תזוזה ממוצעת <bdi dir="ltr">0.36</bdi> מול <bdi dir="ltr">1.10</bdi> וספירמן <bdi dir="ltr">0.924</bdi> מול <bdi dir="ltr">0.774.</bdi> סדר אקראי נותן <bdi dir="ltr">1.94/2.625.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">108–113</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M5:</bdi> קבוצת פלייאוף ופיזור; <bdi dir="ltr">M4</bdi> בדיקת תזוזה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">4</bdi> וה-ס<bdi dir="ltr">9.</bdi> אלה <bdi dir="ltr">72/96</bdi> קבוצות־עונות בעידן <bdi dir="ltr">14.</bdi> אין להפוך זאת לתוספת רעש ידנית.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מובילת הפיצול זכתה ב־<bdi dir="ltr">12/12</bdi> בעידן <bdi dir="ltr">14</bdi>; <bdi dir="ltr">17/24</bdi> משתי האחרונות ירדו מול <bdi dir="ltr">6/24</bdi> ממקומות <bdi dir="ltr">11–12.</bdi> בכל הפיתוח <bdi dir="ltr">14/15</bdi> מובילות זכו, עם חריגת חצייה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">108–113</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M6:</bdi> להוכיח ערך גם בירידה; <bdi dir="ltr">M4/B1</bdi> כיול</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">האליפות בסוף הסדיר קלה יחסית לחיזוי במדגם זה. אסור להסתפק בספירמן כולל שמסתיר טעויות בקו הירידה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שערים למשחק: סדיר <bdi dir="ltr">2.411,</bdi> סדיר מאוחר <bdi dir="ltr">2.310,</bdi> עליון <bdi dir="ltr">2.669</bdi> ותחתון <bdi dir="ltr">2.539.</bdi> לפי כוח הסדיר צפויים בעליון <bdi dir="ltr">2.463</bdi> ובתחתון <bdi dir="ltr">2.201</bdi>; העודף העונתי <bdi dir="ltr">+0.204</bdi> (<bdi dir="ltr">SE</bdi> <bdi dir="ltr">0.085)</bdi> ו-<bdi dir="ltr">+0.338</bdi> (<bdi dir="ltr">SE</bdi> <bdi dir="ltr">0.106).</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">114–117</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: פיצ'ר או סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0:</bdi> <bdi dir="ltr">stage</bdi>; ברירת מחדל ל־<bdi dir="ltr">M2</bdi> לפי ה-ס<bdi dir="ltr">10</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הכוח נאמד מסדיר מלא, בדיעבד. השלב נבנה; <bdi dir="ltr">M2</bdi> שיבחן את תרומתו טרם נבנה. אין הוכחת מוטיבציה או תיקון קבוע של <bdi dir="ltr">+0.338.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מול הסדיר: בעליון <bdi dir="ltr">+0.257</bdi> ובתחתון <bdi dir="ltr">+0.128</bdi> שערים; בסדיר המאוחר <bdi dir="ltr">−0.139</bdi> (<bdi dir="ltr">SE</bdi> <bdi dir="ltr">0.066).</bdi> שינוי קצב נקודות: עליון <bdi dir="ltr">−0.425</bdi> ותחתון <bdi dir="ltr">+0.336</bdi> בעידן <bdi dir="ltr">14,</bdi> מול כ-<bdi dir="ltr">−0.41/+0.35</bdi> בכלל הפיתוח.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">114–123</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M4</bdi> שחזור שינוי יריבות טבעי; <bdi dir="ltr">M1</bdi> חקר השארית, <bdi dir="ltr">04/07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">4/FEATURES</bdi> <bdi dir="ltr">M4:</bdi> אין תיקון מכני לקצב. דוגמאות באר שבע <bdi dir="ltr">1.88→1.20</bdi> ואשדוד <bdi dir="ltr">0.85→2.14</bdi> אינן סיבתיות.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שימור הסטייה היחסית בתוך הפלייאוף: בעידן <bdi dir="ltr">14</bdi> עליון שיפוע <bdi dir="ltr">0.99</bdi> ומתאם <bdi dir="ltr">0.71,</bdi> תחתון <bdi dir="ltr">0.31/0.12.</bdi> בכלל הפיתוח <bdi dir="ltr">0.93/0.62</bdi> ו־<bdi dir="ltr">0.33/0.12.</bdi> פיזור כוח בסדיר <bdi dir="ltr">0.389/0.198</bdi> מול רעש כ־<bdi dir="ltr">0.255.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">124–127</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M5</bdi> הקשר קבוצה; <bdi dir="ltr">M1</bdi> שאלת בטוחות; <bdi dir="ltr">M4</bdi> בדיקת שפיות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ההבדל מתאים לפערי כוח ולמדגם הקצר בתחתון, ואינו הוכחה לכאוס או לחוסר רצון. ה-ס<bdi dir="ltr">4</bdi>; אין גורם כאוס.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C10</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בכל הפיתוח, ללא הפלייאוף האמצעי: בשלושת מחזורי הסיום של שלב <bdi dir="ltr">20.8%</bdi> תיקו (<bdi dir="ltr">n=689)</bdi> מול <bdi dir="ltr">28.4%</bdi> (<bdi dir="ltr">n=3,614),</bdi> <bdi dir="ltr">OR=0.671,</bdi> <bdi dir="ltr">CI</bdi> <bdi dir="ltr">0.546–0.824,</bdi> <bdi dir="ltr">p=0.000132.</bdi> בעידן <bdi dir="ltr">14:</bdi> <bdi dir="ltr">20.3%</bdi> מול <bdi dir="ltr">28.3%,</bdi> <bdi dir="ltr">OR=0.644</bdi>; הכיוון ב־<bdi dir="ltr">11/12</bdi> עונות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">128–133</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: פיצ'ר או סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0:</bdi> <bdi dir="ltr">rounds_to_stage_end</bdi>; <bdi dir="ltr">M2</bdi> ברירת מחדל, ה-ס<bdi dir="ltr">11</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הגרף המרכזי הוא כל הפיתוח, חריג מפורש לברירת המחדל של <bdi dir="ltr">C.</bdi> המונה שנבנה הוא משחקים לקבוצה, בעוד ה־<bdi dir="ltr">EDA</bdi> השתמש במחזור ממוספר: <bdi dir="ltr">S4.</bdi> המבחן נבחר בדיעבד.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C11</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפי שלב <bdi dir="ltr">OR</bdi> סוף/יתר: <bdi dir="ltr">0.710</bdi> סדיר, <bdi dir="ltr">0.494</bdi> עליון ו־<bdi dir="ltr">0.704</bdi> תחתון. בעליון בעידן <bdi dir="ltr">14:</bdi> <bdi dir="ltr">3.06</bdi> שערים מול <bdi dir="ltr">2.50</bdi> ותיקו <bdi dir="ltr">15.0%</bdi> מול <bdi dir="ltr">28.6%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">128–133</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M1,</bdi> מחברת <bdi dir="ltr">04:</bdi> חשיבות מול חוזק; <bdi dir="ltr">M3,</bdi> <bdi dir="ltr">06:</bdi> כיול התיקו</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא מסיקים שהגורם הוא מוטיבציה; אין עוד פיצ׳ר חדש מכוח הפירוק. ה-ת<bdi dir="ltr">5</bdi> מגביל קלט <bdi dir="ltr">M3,</bdi> ולכן תלות סוף שלב דורשת חוזה מפורש.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C12</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בעידן <bdi dir="ltr">14,</bdi> תיקו בסדיר לפי מרחק <bdi dir="ltr">0–2/3–5/6–8/9–11/12–14/15+:</bdi> <bdi dir="ltr">22.2%/29.0%/26.6%/25.9%/31.3%/28.0%.</bdi> בעליון בארבעת הטווחים הקיימים: <bdi dir="ltr">15.0%/30.6%/30.6%/16.7%</bdi>; בתחתון: <bdi dir="ltr">20.8%/31.2%/27.1%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">130–131</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M1/M2:</bdi> בדיקת צורת הקשר, <bdi dir="ltr">04–05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">זו סדרת שיעורים תיאורית עם <bdi dir="ltr">Wilson,</bdi> ולא עקומת מוטיבציה שנבחרה. המקור הוא עידן <bdi dir="ltr">14,</bdi> ולא כל הפיתוח.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C13</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מרוץ <bdi dir="ltr">2023/2024:</bdi> מכבי ת״א <bdi dir="ltr">85</bdi> וחיפה <bdi dir="ltr">76</bdi> נקודות ספורטיביות ב־<bdi dir="ltr">36</bdi> משחקים; הפועל ירושלים <bdi dir="ltr">43</bdi> ב־<bdi dir="ltr">33.</bdi> בדוגמה מופיע רצף תשעה ניצחונות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">134–137</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M4:</bdi> צבירת נקודות ולוחות באורכים שונים, מחברת <bdi dir="ltr">07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין להשלים משחקים שלא התקיימו או לפרש קו ישר כחסר. תכונת ״רצף״ עצמה לא נבחרה; הדוגמה משמשת לבדיקת מנגנון הסימולציה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ניצחון החלשה לפי קצב נקודות עבר בסוף שלב מול יתרו, בעידן <bdi dir="ltr">14:</bdi> תחתון <bdi dir="ltr">43.1%/32.4%,</bdi> עליון <bdi dir="ltr">30.2%/21.9%,</bdi> סדיר <bdi dir="ltr">21.4%/27.3%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">C,</bdi> <bdi dir="ltr">129,</bdi> <bdi dir="ltr">132</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M1:</bdi> מחקר חשיבות מול ציפיית חוזק, מחברת <bdi dir="ltr">04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הפתעות אינן ממצא שהוכרע; הגדרת כוח היא קצב נקודות פשוט, בלי תקנון מלא ללוח. אין מסקנה שיש להוסיף הסתברות הפתעה קבועה.</td></tr></tbody></table></div></details>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — <bdi dir="ltr">D</bdi></h2>
<details><summary style="cursor:pointer;font-weight:bold;padding:12px;background:#edf3fa;border:1px solid #ccd7e4">פרק <bdi dir="ltr">D:</bdi> <bdi dir="ltr">11</bdi> ממצאים (<bdi dir="ltr">0</bdi> ישיר: פיצ׳ר/סרגל, <bdi dir="ltr">1</bdi> הנדסי, <bdi dir="ltr">0</bdi> עקיף, <bdi dir="ltr">0</bdi> נדחו, <bdi dir="ltr">9</bdi> מתוכננים, <bdi dir="ltr">1</bdi> ללא תוכנית)</summary><div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הממצא — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור במחברת <bdi dir="ltr">02</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מעמד</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">איפה בשרשרת</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הערה / החלטה</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D01</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לניתוחי ציר הזמן מוחרגים <bdi dir="ltr">14</bdi> מ־<bdi dir="ltr">4,315</bdi> משחקים (<bdi dir="ltr">13</bdi> סתירות מחצית ואחד עם מבקיעים לא אמינים); נשארים <bdi dir="ltr">4,301.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">140–141</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: כלל הנדסי או בקרה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בקרת אוכלוסייה ל־<bdi dir="ltr">EDA,</bdi> לא סינון אימון <bdi dir="ltr">M0/M2</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">DATA_DICTIONARY</bdi> <bdi dir="ltr">D:</bdi> תוצאת משחק מלא נשארת תקינה גם כשלא ניתן להשתמש בתזמון. אין להעביר את ההחרגה למודל השערים.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כיסוי תוצאות: <bdi dir="ltr">0–3</bdi> מכסה <bdi dir="ltr">91.56%</bdi>; <bdi dir="ltr">0–4</bdi> <bdi dir="ltr">97.91%</bdi> (<bdi dir="ltr">90</bdi> בחוץ); <bdi dir="ltr">0–5</bdi> <bdi dir="ltr">99.56%</bdi> (<bdi dir="ltr">19)</bdi>; <bdi dir="ltr">0–6</bdi> <bdi dir="ltr">99.91%</bdi> (<bdi dir="ltr">4)</bdi>; השיא <bdi dir="ltr">7.</bdi> לקבוצה עם λ=<bdi dir="ltr">3,</bdi> הסתברות <bdi dir="ltr">5+</bdi> כ־<bdi dir="ltr">18.5%</bdi> ו־<bdi dir="ltr">6+</bdi> כ־<bdi dir="ltr">8.4%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">142–143</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3</bdi> מטריצת <bdi dir="ltr">6×6</bdi> עם <bdi dir="ltr">5+</bdi>; <bdi dir="ltr">M4</bdi> רושם <bdi dir="ltr">5</bdi> שערים, <bdi dir="ltr">06–07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">12.</bdi> הכיסוי מתייחס לריבוע מדויק לפני איגום. שאלת פענוח <bdi dir="ltr">5+:5+</bdi> ל־<bdi dir="ltr">H/D/A</bdi> מופיעה ב־<bdi dir="ltr">S4/S5</bdi>; טרם מומש <bdi dir="ltr">M3</bdi> סופי.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D03</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">עשר השכיחות: <bdi dir="ltr">1:1</bdi> כ־<bdi dir="ltr">12.1%</bdi> (<bdi dir="ltr">523),</bdi> <bdi dir="ltr">1:0</bdi> <bdi dir="ltr">10.5%,</bdi> <bdi dir="ltr">0:0</bdi> <bdi dir="ltr">9.5%,</bdi> <bdi dir="ltr">0:1</bdi> <bdi dir="ltr">9.1%,</bdi> <bdi dir="ltr">2:0</bdi> <bdi dir="ltr">7.9%,</bdi> <bdi dir="ltr">2:1</bdi> <bdi dir="ltr">7.3%,</bdi> <bdi dir="ltr">1:2</bdi> <bdi dir="ltr">6.2%,</bdi> <bdi dir="ltr">0:2</bdi> <bdi dir="ltr">5.9%,</bdi> <bdi dir="ltr">2:2</bdi> <bdi dir="ltr">4.6%,</bdi> <bdi dir="ltr">3:0</bdi> <bdi dir="ltr">3.9%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">144–147</bdi> (כולל ערכי גרף)</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3:</bdi> מפת תוצאות לבדיקת כיול, <bdi dir="ltr">06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">14:</bdi> מעריכים התפלגות ולא רק ניחוש יחיד. לא מאמנים תחזית שכל המשחקים בה <bdi dir="ltr">1:1.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">Poisson</bdi> עם ממוצע ליגה בלבד: λ בית <bdi dir="ltr">1.324,</bdi> חוץ <bdi dir="ltr">1.140,</bdi> מסה מחוץ לריבוע <bdi dir="ltr">0–4</bdi> כ־<bdi dir="ltr">1.77%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">148–149</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא בשימוש ואין תוכנית</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קו השוואה תיאורי בלבד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אלה אומדנים מכל המדגם, לא λ הייצור ולא ערכי פתיחה היסטוריים. <bdi dir="ltr">M2</bdi> ילמד תוחלת לכל מפגש; אין תוכנית להשתמש בקבועים המדויקים האלה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">Poisson</bdi> לכל משחק בחצאים משלימים: תיקו <bdi dir="ltr">27.1%</bdi> נצפה מול <bdi dir="ltr">25.4%</bdi> צפוי. <bdi dir="ltr">0:0:</bdi> מול ממוצע גלובלי יחס <bdi dir="ltr">1.117,</bdi> אך מול כוח לכל משחק <bdi dir="ltr">0.928.</bdi> <bdi dir="ltr">1:1</bdi> יחס <bdi dir="ltr">1.126</bdi>; <bdi dir="ltr">2:2</bdi> <bdi dir="ltr">1.282.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">150–151</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3:</bdi> לבדוק תיקון כללי מתוך λ כרונולוגי, <bdi dir="ltr">06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חצאים אקראיים בתוך עונה הם <bdi dir="ltr">EDA</bdi> ולא <bdi dir="ltr">OOF</bdi> כרונולוגי. הגרף אינו מצדיק תיקון עצמאי ל־<bdi dir="ltr">1:1/2:2</bdi>; ראו <bdi dir="ltr">D06.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">200</bdi> ליגות אפס עם התאמה מחדש: יחס תיקו נצפה/צפוי <bdi dir="ltr">1.068</bdi> מול <bdi dir="ltr">0.999</bdi> באפס, <bdi dir="ltr">z=2.737</bdi> ו־<bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.015.</bdi> ל־<bdi dir="ltr">0:0/1:1/2:2:</bdi> <bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.119/0.473/0.114,</bdi> בלי הוכחה נפרדת.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">152–157</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3</bdi> <bdi dir="ltr">Dixon-Coles</bdi> מול ניפוח תיקו; בדיקת אפס מכוילת, <bdi dir="ltr">06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13.</bdi> עודף כ־<bdi dir="ltr">7%</bdi> יחסי הוא תיאור, לא מכפיל קבוע מראש. גבולות האפס אינם רווח סמך לאפקט.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפי שקילות המשחק: יחסי התיקו <bdi dir="ltr">0.937/1.035/1.317</bdi> מול <bdi dir="ltr">0.905/0.965/1.211</bdi> באפס; <bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.383/0.134/0.060.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">154–157</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3:</bdi> בדיקת כיול לפי פער λ, <bdi dir="ltr">06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין אישור לתיקון מיוחד למשחקים לא שקולים. נדרש λ מחוץ למדגם של <bdi dir="ltr">M2,</bdi> לא אומדן מעונה שלמה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">יציבות מפת התוצאות בין עונות: χ²=<bdi dir="ltr">161.26,</bdi> <bdi dir="ltr">df=153,</bdi> <bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.308</bdi>; תמורות <bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.319.</bdi> בעידן <bdi dir="ltr">14:</bdi> כ־<bdi dir="ltr">0.477/0.489.</bdi> לשיעור <bdi dir="ltr">12%</bdi> ב־<bdi dir="ltr">240</bdi> משחקים <bdi dir="ltr">SE</bdi> תיאורי כ־<bdi dir="ltr">2.1</bdi> נ״א.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">158–159</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3:</bdi> חוק משותף עם λ משתנה, ובדיקות כיול, <bdi dir="ltr">06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא נמצאה ראיה שכל עונה צריכה מטריצה חופשית משלה; אין הוכחת זהות מלאה בין עונות.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מגמות יתר התוצאות בנ״א לשנה: <bdi dir="ltr">1:1</bdi> <bdi dir="ltr">+0.169</bdi> (<bdi dir="ltr">p=.066),</bdi> <bdi dir="ltr">1:0</bdi> <bdi dir="ltr">−0.084</bdi> (.<bdi dir="ltr">364),</bdi> <bdi dir="ltr">0:1</bdi> <bdi dir="ltr">+0.025</bdi> (.<bdi dir="ltr">796),</bdi> <bdi dir="ltr">2:0</bdi> <bdi dir="ltr">+0.023</bdi> (.<bdi dir="ltr">658),</bdi> <bdi dir="ltr">2:1</bdi> <bdi dir="ltr">−0.056</bdi> (.<bdi dir="ltr">459),</bdi> <bdi dir="ltr">1:2</bdi> <bdi dir="ltr">+0.111</bdi> (.<bdi dir="ltr">061),</bdi> <bdi dir="ltr">0:2</bdi> <bdi dir="ltr">+0.157</bdi> (.<bdi dir="ltr">098),</bdi> <bdi dir="ltr">2:2</bdi> <bdi dir="ltr">−0.006</bdi> (.<bdi dir="ltr">923).</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">158–159</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3:</bdi> אבחון יציבות, לא תיקונים נפרדים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מגמת <bdi dir="ltr">0:0</bdi> מרוכזת ב־<bdi dir="ltr">B07,</bdi> ללא ספירה כפולה. אלו בדיקות חקרניות מרובות, לא רשת תיקונים שנבחרה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D10</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">פיזור גולמי בית/חוץ <bdi dir="ltr">1.089/1.082,</bdi> לסך השערים כ־<bdi dir="ltr">1.00,</bdi> ומתאם צדדים <bdi dir="ltr">−0.081.</bdi> אחרי חוזק: בית <bdi dir="ltr">1.069</bdi> מול <bdi dir="ltr">1.157</bdi> באפס (<bdi dir="ltr">p=.005),</bdi> חוץ <bdi dir="ltr">1.089</bdi> מול <bdi dir="ltr">1.139</bdi> (<bdi dir="ltr">p=.070)</bdi>; מתאם שאריות <bdi dir="ltr">0.008</bdi> מול <bdi dir="ltr">0.001</bdi> (<bdi dir="ltr">p=.711).</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">160–163</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M2</bdi> <bdi dir="ltr">Poisson</bdi>; <bdi dir="ltr">M3</bdi> עצמאי + תיקון תיקו, <bdi dir="ltr">05–06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13:</bdi> דו־משתני ובינומית שלילית לבדיקת שפיות בלבד. אי־מובהקות אינה הוכחת עצמאות מדויקת; אין פיזור יתר שיורי שמצדיק אוטומטית מודל מורכב.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D11</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ניצחונות נמוכים מול <bdi dir="ltr">Poisson:</bdi> <bdi dir="ltr">1:0</bdi> נצפה <bdi dir="ltr">10.5%,</bdi> גלובלי <bdi dir="ltr">11.3%</bdi> ופרטני <bdi dir="ltr">11.5%</bdi>; <bdi dir="ltr">0:1</bdi> נצפה <bdi dir="ltr">9.1%,</bdi> גלובלי <bdi dir="ltr">9.7%</bdi> ופרטני <bdi dir="ltr">10.1%.</bdi> יחסים פרטניים <bdi dir="ltr">0.912/0.895</bdi> מול אפס <bdi dir="ltr">0.982/0.977</bdi>; <bdi dir="ltr">p</bdi>≈<bdi dir="ltr">0.070/0.060.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">151,</bdi> <bdi dir="ltr">155</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3:</bdi> בדיקת כיול של כל מפת התוצאות, <bdi dir="ltr">06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין כאן אישור לתיקון עצמאי לתוצאות האלה; בחירת תיקון נבחנת עם λ כרונולוגי ובדיקת התהליך כולו.</td></tr></tbody></table></div></details>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — <bdi dir="ltr">D</bdi> — ⛔</h2>
<details><summary style="cursor:pointer;font-weight:bold;padding:12px;background:#edf3fa;border:1px solid #ccd7e4">תוך־משחק — נשמר לתיעוד בלבד: <bdi dir="ltr">13</bdi> ממצאים (<bdi dir="ltr">0</bdi> ישיר: פיצ׳ר/סרגל, <bdi dir="ltr">0</bdi> הנדסי, <bdi dir="ltr">0</bdi> עקיף, <bdi dir="ltr">13</bdi> נדחו, <bdi dir="ltr">0</bdi> מתוכננים, <bdi dir="ltr">0</bdi> ללא תוכנית)</summary><div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הממצא — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור במחברת <bdi dir="ltr">02</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מעמד</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">איפה בשרשרת</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הערה / החלטה</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X01</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">דוגמת דקות במצב משחק: <bdi dir="ltr">2023/2024,</bdi> קבוצה יכולה להיות <bdi dir="ltr">0</bdi> דקות ביתרון, <bdi dir="ltr">34</bdi> בשוויון ו־<bdi dir="ltr">56</bdi> בפיגור.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">164–165</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אירועים בתוך המשחק</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כל בלוק <bdi dir="ltr">164–211</bdi> הוקפא לתיעוד בלבד לפי ה-ס<bdi dir="ltr">13</bdi>; לא מידע לפני שריקה ולא פיצ׳ר ניהול משחק.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מובילה במחצית ניצחה ב־<bdi dir="ltr">77.3%</bdi> בבית וב־<bdi dir="ltr">72.4%</bdi> בחוץ; הפסידה ב־<bdi dir="ltr">6.2%/8.8%.</bdi> <bdi dir="ltr">28</bdi> שורות קבוצה הוחרגו מציר הזמן.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">166–169</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין יעד/פיצ׳ר מחצית במודלי <bdi dir="ltr">prematch</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; לא תכונת אופי ולא הסתברות למשחק שטרם התחיל.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X03</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">1,003</bdi> משחקים ללא שער עד דקה <bdi dir="ltr">60:</bdi> <bdi dir="ltr">40.9%</bdi> הסתיימו <bdi dir="ltr">0:0,</bdi> <bdi dir="ltr">31.0%</bdi> ניצחון בית, <bdi dir="ltr">21.4%</bdi> חוץ ו־<bdi dir="ltr">6.7%</bdi> תיקו עם שערים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">170–171</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין מודל תוך־משחקי בשרשרת</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; ידיעת הדקה והתוצאה אינה זמינה למשחק עתידי.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מבקיעה ראשונה ב־<bdi dir="ltr">3,890</bdi> משחקים עם סדר ידוע: בית ניצחה <bdi dir="ltr">72.5%</bdi> (<bdi dir="ltr">2.35</bdi> נקודות), חוץ <bdi dir="ltr">65.7%</bdi> (<bdi dir="ltr">2.19)</bdi>; התאוששות לניצחון אחרי ספיגה ראשונה <bdi dir="ltr">12.3%/10.2%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">172–175</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר ״מבקיעה ראשונה״</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; הבדלים בחוזק ובמועד השער אינם מנוטרלים. זה אינו אפקט סיבתי של שער ראשון.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מועד שער ראשון: <bdi dir="ltr">0–15</bdi> <bdi dir="ltr">27.6%,</bdi> <bdi dir="ltr">16–30</bdi> <bdi dir="ltr">20.7%,</bdi> <bdi dir="ltr">31–45</bdi> <bdi dir="ltr">16.2%,</bdi> תוספת ראשונה <bdi dir="ltr">0.3%,</bdi> <bdi dir="ltr">46–60</bdi> <bdi dir="ltr">11.8%,</bdi> <bdi dir="ltr">61–75</bdi> <bdi dir="ltr">7.2%,</bdi> <bdi dir="ltr">76–90</bdi> <bdi dir="ltr">5.7%,</bdi> תוספת שנייה <bdi dir="ltr">0.9%</bdi>; ללא שער <bdi dir="ltr">9.5%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">176–177</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר תזמון אירועים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; השיעורים מתוך כלל המשחקים התקינים, לא הסתברות הבקעה מותנית בדקה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">15</bdi> הדקות הראשונות <bdi dir="ltr">88.2%</bdi> מהשערים הגיעו בשוויון; בדקות <bdi dir="ltr">76–90:</bdi> <bdi dir="ltr">42.3%</bdi> בזמן יתרון, <bdi dir="ltr">26.8%</bdi> בשוויון ו־<bdi dir="ltr">30.8%</bdi> בפיגור.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">178–179</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין מודל מהלך משחק</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; חלוקת שערים אינה קצב לפי זמן החשיפה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קצב שערים ל־<bdi dir="ltr">90</bdi> דקות בפיגור/שוויון: בית <bdi dir="ltr">1.29/1.19,</bdi> חוץ <bdi dir="ltr">1.17/1.03</bdi>; ביתרון <bdi dir="ltr">1.62/1.35.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">180–183</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר ״ניהול מצב״</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; גם אחרי נרמול לחשיפה לא בוצע נטרול לחוזק ולבחירה למצב. אין להסיק שפיגור כשלעצמו מעלה קצב.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">דקות יתרון ממוצעות לקבוצות־עונה בטווח כ־<bdi dir="ltr">3.2–48.5</bdi>; בדוגמת <bdi dir="ltr">2023/2024</bdi> חיפה <bdi dir="ltr">37.33,</bdi> מכבי ת״א <bdi dir="ltr">38.44</bdi> והפועל פ״ת <bdi dir="ltr">9.15.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">184–187</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר זמן הובלה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; זהות הקבוצה ורמתה מעורבות עם הסגנון. נשמר לתיעוד בלבד.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נקודות שחולצו למשחק פיגור: באר שבע <bdi dir="ltr">0.80,</bdi> מכבי ת״א <bdi dir="ltr">0.79</bdi> והפועל ת״א <bdi dir="ltr">0.53</bdi>; איבוד למשחק יתרון <bdi dir="ltr">0.44/0.40/0.72</bdi> בהתאמה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">188–191</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר חוסן/ווינריות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; תנאי החשיפה שונה בין קבוצות, ואין זיהוי סיבתי של אופי.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X10</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חלק שערים מדקה <bdi dir="ltr">76:</bdi> מכבי ת״א <bdi dir="ltr">24.5%,</bdi> חיפה <bdi dir="ltr">22.1%,</bdi> פ״ת <bdi dir="ltr">28.3%</bdi>; חלק שערי המחצית השנייה <bdi dir="ltr">56.4%/54.6%</bdi> לשתי הראשונות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">192–195</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר ״מבקיעה מאוחר״</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; פירוק שערים לתזמון אינו מידע חדש מוכח מעל חוזק התקפה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X11</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">854</bdi> שערי פנדל ו־<bdi dir="ltr">244</bdi> עצמיים מתוך <bdi dir="ltr">10,632</bdi> שערים עם סוג ידוע; פנדלים כ־<bdi dir="ltr">8%,</bdi> שיעור עצמי כ־<bdi dir="ltr">1.8%→2.6%</bdi> בין קצות התקופה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">196–201</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר פנדלים/עצמיים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; שערים שכן הובקעו נכללים כרגיל בתוצאה, אך זו אינה הכנסת סוג השער לפיצ׳ר. אין מידע על החמצות ושריקות.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X12</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">עם אדום רק לקבוצה: <bdi dir="ltr">15.8%</bdi> ניצחונות ו־<bdi dir="ltr">0.73</bdi> נקודות, מול <bdi dir="ltr">58.8%</bdi> ו־<bdi dir="ltr">2.02</bdi> כשאדום רק ליריבה; ללא אדומים <bdi dir="ltr">1.36</bdi> נקודות, לשתיהן <bdi dir="ltr">1.35.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">202–205</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר משמעת ואין הפחתת משקל בגלל אדום</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; זה קשר תיאורי, לא אומדן סיבתי. בתצוגת הסיכום מופיעה עונה פיקטיבית <bdi dir="ltr">1998/1999</bdi> עקב סימון סיכום <bdi dir="ltr">0</bdi>; <bdi dir="ltr">S4.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">X13</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אדום ראשון עד דקה <bdi dir="ltr">60:</bdi> <bdi dir="ltr">432</bdi> שורות ו־<bdi dir="ltr">0.78</bdi> נקודות; מאוחר יותר <bdi dir="ltr">641</bdi> ו־<bdi dir="ltr">0.86.</bdi> ל־<bdi dir="ltr">12</bdi> כרטיסים חסרה דקה. חלק צהוב שני בקצות התקופה <bdi dir="ltr">30/76=39.5%</bdi> מול <bdi dir="ltr">21/68=30.9%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">206–211</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר תזמון/סוג אדום</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi>; משחקים תלויים ולא הושוו לפי כוח ומצב. אין תוכנית למודל תוך־משחקי במסגרת הנוכחית.</td></tr></tbody></table></div></details>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — <bdi dir="ltr">E</bdi></h2>
<details><summary style="cursor:pointer;font-weight:bold;padding:12px;background:#edf3fa;border:1px solid #ccd7e4">פרק <bdi dir="ltr">E:</bdi> <bdi dir="ltr">6</bdi> ממצאים (<bdi dir="ltr">1</bdi> ישיר: פיצ׳ר/סרגל, <bdi dir="ltr">1</bdi> הנדסי, <bdi dir="ltr">2</bdi> עקיף, <bdi dir="ltr">1</bdi> נדחו, <bdi dir="ltr">1</bdi> מתוכננים, <bdi dir="ltr">0</bdi> ללא תוכנית)</summary><div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הממצא — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור במחברת <bdi dir="ltr">02</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מעמד</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">איפה בשרשרת</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הערה / החלטה</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E01</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ניתוחים שנמחקו: אחרי החלפת מאמן שיפור גולמי כ־<bdi dir="ltr">0.33</bdi> נקודות מול כ־<bdi dir="ltr">0.09</bdi> בבקרה; בדרבי תיקו <bdi dir="ltr">26.8%</bdi> מול <bdi dir="ltr">27.3%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E,</bdi> <bdi dir="ltr">213</bdi> (תיעוד הניתוחים שהוסרו)</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין מאמן, דרבי או דירוג אצטדיונים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ס<bdi dir="ltr">13</bdi> ו־<bdi dir="ltr">FEATURES.</bdi> מספרי ארכיון מצוטטים, לא חושבו מחדש; אצטדיון משקף זהות מארחת ולא פיצ׳ר כוח נוסף.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קבוצה חלשה לפי שווי קודם: בפער קטן ניצחה <bdi dir="ltr">41.7%</bdi> בבית מול <bdi dir="ltr">33.2%</bdi> בחוץ; בפער גדול <bdi dir="ltr">25.8%</bdi> מול <bdi dir="ltr">15.9%.</bdi> תוספת כ־<bdi dir="ltr">8.5/9.9</bdi> נ״א, יחס <bdi dir="ltr">1.26/1.62.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E,</bdi> <bdi dir="ltr">214–219</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M2:</bdi> ביתיות אחת כפלית; אינטראקציה רק ב־<bdi dir="ltr">ablation,</bdi> <bdi dir="ltr">05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא מאמצים ביתיות אישית. ״חלשה״ נקבעה ממדד קודם, אך זהות ולוח עדיין מערבבים; אין הסקה סיבתית.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E03</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפי פער מיקום קודם <bdi dir="ltr">1–3:</bdi> החלשה ניצחה <bdi dir="ltr">35.4%</bdi> בבית מול <bdi dir="ltr">30.2%</bdi> בחוץ; בפער <bdi dir="ltr">8+:</bdi> <bdi dir="ltr">21.2%</bdi> מול <bdi dir="ltr">11.7%.</bdi> <bdi dir="ltr">18</bdi> משחקים חסרי שווי ו־<bdi dir="ltr">296</bdi> בלי עבר לשתיהן מוצגים בנפרד.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E,</bdi> <bdi dir="ltr">214–219</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: פיצ'ר או סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0:</bdi> <bdi dir="ltr">log</bdi> יחס מיקום ספורטיבי וטיפול בחסר</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">02</bdi> המיקום רשמי, בייצור ספורטיבי; אין העתקת לוג המיקום הרשמי. מיקום נשאר 🔬 לפי ה-ת<bdi dir="ltr">14.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">עולה מול ותיקה בלבד: <bdi dir="ltr">28.8%</bdi> ניצחונות כשהעולה בבית (<bdi dir="ltr">531</bdi> משחקים) ו־<bdi dir="ltr">23.7%</bdi> כשהיא בחוץ (<bdi dir="ltr">540).</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E,</bdi> <bdi dir="ltr">216–217</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש עקיף</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0:</bdi> פתיחה לעולה ודגל, ה-ס<bdi dir="ltr">5/</bdi>ה-ת<bdi dir="ltr">13</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">C6b</bdi> נמדדו כל משחקי העולות, ולכן <bdi dir="ltr">30.2%</bdi> שם אינו סותר <bdi dir="ltr">28.8%</bdi> כאן. אין להגדיר את העולה תמיד כחלשה יותר.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">LOSO</bdi> על אותם <bdi dir="ltr">2,930</bdi> משחקים: <bdi dir="ltr">B0</bdi> <bdi dir="ltr">1.0835,</bdi> שווי לוג <bdi dir="ltr">1.0484,</bdi> פער מיקום <bdi dir="ltr">1.0441,</bdi> לוג מיקום <bdi dir="ltr">1.0349,</bdi> שילוב <bdi dir="ltr">1.0319.</bdi> <bdi dir="ltr">RPS:</bdi> <bdi dir="ltr">0.2306/0.2179/0.2167/0.2137/0.2125.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E,</bdi> <bdi dir="ltr">220–221</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: כלל הנדסי או בקרה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בדיקת רגרסיה לשופט; רקע ל־<bdi dir="ltr">B0</bdi> ול־<bdi dir="ltr">EXP-003</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ללא עולות, ועל <bdi dir="ltr">17</bdi> עונות; <bdi dir="ltr">LOSO</bdi> רטרוספקטיבי, לא סרגל בר־השוואה ל־<bdi dir="ltr">E0050</bdi> הכרונולוגי. השופט משחזר את ה־<bdi dir="ltr">EDA</bdi> בנפרד.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שווי מול <bdi dir="ltr">B0:</bdi> הפרש <bdi dir="ltr">log</bdi> <bdi dir="ltr">loss</bdi> <bdi dir="ltr">−0.0351,</bdi> <bdi dir="ltr">CI</bdi> <bdi dir="ltr">−0.0475</bdi> עד <bdi dir="ltr">−0.0195,</bdi> טוב ב־<bdi dir="ltr">15/17</bdi>; שווי מול פער מיקום <bdi dir="ltr">+0.0136</bdi> (<bdi dir="ltr">CI</bdi> <bdi dir="ltr">−0.0028</bdi> עד <bdi dir="ltr">+0.0336).</bdi> שילוב מעל לוג מיקום <bdi dir="ltr">−0.003</bdi> (<bdi dir="ltr">CI</bdi> <bdi dir="ltr">−0.009</bdi> עד <bdi dir="ltr">+0.0038)</bdi>; מתאם שווי–מיקום <bdi dir="ltr">0.613.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E,</bdi> <bdi dir="ltr">220–221</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש עקיף</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">התכנון לבדוק תוספת מידע ב־<bdi dir="ltr">EXP-003</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא הוכחה שהשילוב מוסיף. ה-ת<bdi dir="ltr">14</bdi> קובע מעמדים אחרי הניסוי הכרונולוגי, ולא לפי ה־<bdi dir="ltr">CI</bdi> של <bdi dir="ltr">LOSO.</bdi></td></tr></tbody></table></div></details>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — ספירה ומה נשאר בלי שימוש</h2>
<p>הספירה היא של שורות המיפוי של <bdi dir="ltr">S2,</bdi> לא של תאי מחברת או של השערות עצמאיות. מקור המספרים: מיפוי <bdi dir="ltr">S2</bdi> שנכתב כאן; אין זו סטטיסטיקה חדשה על משחקים.</p>
<p>פריטי ״לא בשימוש ואין תוכנית״ מפורטים להלן בנפרד. אין בכך המלצה להכניס אותם: הם רק החלק שנותר תיאורי, בלי שימוש מוגדר בתוכנית הנוכחית.</p>
<div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מעמד</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מספר ממצאים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: פיצ'ר או סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">8</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שורות <bdi dir="ltr">S2,</bdi> לפי המעמד הראשי; חישוב ספירה בלבד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש ישיר: כלל הנדסי או בקרה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">6</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שורות <bdi dir="ltr">S2,</bdi> לפי המעמד הראשי; חישוב ספירה בלבד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בשימוש עקיף</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">6</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שורות <bdi dir="ltr">S2,</bdi> לפי המעמד הראשי; חישוב ספירה בלבד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נדחה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">25</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שורות <bdi dir="ltr">S2,</bdi> לפי המעמד הראשי; חישוב ספירה בלבד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מתוכנן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">29</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שורות <bdi dir="ltr">S2,</bdi> לפי המעמד הראשי; חישוב ספירה בלבד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא בשימוש ואין תוכנית</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">3</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שורות <bdi dir="ltr">S2,</bdi> לפי המעמד הראשי; חישוב ספירה בלבד</td></tr></tbody></table></div>
<p><bdi dir="ltr">14</bdi> ממצאים בשימוש מודלי: <bdi dir="ltr">8</bdi> ישירים ו־<bdi dir="ltr">6</bdi> עקיפים. <bdi dir="ltr">6</bdi> נוספים הם הנדסה/בקרה, ואינם מנפחים את ספירת השימוש במודלים. מקור: ספירת שורות <bdi dir="ltr">S2</bdi> בסיכום הזה; לא ניתוח חדש של משחקים.</p></div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S2</bdi> — הרשימה הנפרדת: לא בשימוש ואין תוכנית</h2>

<div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה חסר</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בדף העונה מכבי ת״א: משחק ללא ספיגה <bdi dir="ltr">44.4%,</bdi> ללא הבקעה <bdi dir="ltr">11.1%,</bdi> <bdi dir="ltr">BTTS</bdi> <bdi dir="ltr">47.2%,</bdi> שלושה שערים ומעלה <bdi dir="ltr">50%</bdi>; רצף ללא הפסד <bdi dir="ltr">14</bdi> וללא ניצחון <bdi dir="ltr">3.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A,</bdi> <bdi dir="ltr">7–8</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שיעורי האפסים והרצפים בדוגמה אינם פיצ׳רים שנבחרו, ואין תוכנית ייעודית להם. <bdi dir="ltr">gf/ga</bdi> נרשמו בנפרד ב־<bdi dir="ltr">A05</bdi>; מסקנת הסגנון הכללית ב־<bdi dir="ltr">A11.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B10</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מכבי ת״א וחיפה זכו יחד ב־<bdi dir="ltr">11</bdi> מ־<bdi dir="ltr">18</bdi> אליפויות (<bdi dir="ltr">6+5)</bdi>; בסך הכול שש אלופות שונות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B,</bdi> <bdi dir="ltr">82–83</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין פיצ׳ר מספר אליפויות/שם מועדון ואין תוכנית לנצל ספירה זו במודל. זיכרון תוצאות <bdi dir="ltr">Elo</bdi> אינו שקול להזרקת מספר תארים.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">Poisson</bdi> עם ממוצע ליגה בלבד: λ בית <bdi dir="ltr">1.324,</bdi> חוץ <bdi dir="ltr">1.140,</bdi> מסה מחוץ לריבוע <bdi dir="ltr">0–4</bdi> כ־<bdi dir="ltr">1.77%.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D,</bdi> <bdi dir="ltr">148–149</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אלה אומדנים מכל המדגם, לא λ הייצור ולא ערכי פתיחה היסטוריים. <bdi dir="ltr">M2</bdi> ילמד תוחלת לכל מפגש; אין תוכנית להשתמש בקבועים המדויקים האלה.</td></tr></tbody></table></div>
</div>
<div dir="rtl" style="text-align:right;line-height:1.7"><h2><bdi dir="ltr">S2</bdi> — מתוכנן לפי מחברת</h2><p>רשימת בדיקה להמשך: רק שורות שמעמדן הראשי ב־<bdi dir="ltr">S2</bdi> הוא ״מתוכנן״. ההקצאה נלקחה משדה ״איפה בשרשרת״ באותן שורות וממפת הרכיבים ב־<bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5–</bdi>§<bdi dir="ltr">6.</bdi></p><p>ממצא אחד יכול להופיע בכמה מחברות, ולכן סכומי המחברות אינם סכום הממצאים הייחודיים. ממצאים שכבר מסומנים בשימוש, גם אם יש להם המשך עתידי (למשל <bdi dir="ltr">C10),</bdi> אינם נספרים מחדש כ״מתוכנן״. לא נוסף שימוש או מעמד חדש.</p><div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מחברת</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מספר ממצאים מתוכננים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהי <bdi dir="ltr">S2</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M1:</bdi> חשיבות תחרותית</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">6</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B01,</bdi> <bdi dir="ltr">C08,</bdi> <bdi dir="ltr">C09,</bdi> <bdi dir="ltr">C11,</bdi> <bdi dir="ltr">C12,</bdi> <bdi dir="ltr">C14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2:</bdi> שורות אלה ושדה היעד שלהן; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5–</bdi>§<bdi dir="ltr">6</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M2:</bdi> שערים צפויים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">6</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A08,</bdi> <bdi dir="ltr">B03,</bdi> <bdi dir="ltr">B05,</bdi> <bdi dir="ltr">C12,</bdi> <bdi dir="ltr">D10,</bdi> <bdi dir="ltr">E02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2:</bdi> שורות אלה ושדה היעד שלהן; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5–</bdi>§<bdi dir="ltr">6</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3:</bdi> תוצאות ותיקו</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">11</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B07,</bdi> <bdi dir="ltr">C11,</bdi> <bdi dir="ltr">D02,</bdi> <bdi dir="ltr">D03,</bdi> <bdi dir="ltr">D05,</bdi> <bdi dir="ltr">D06,</bdi> <bdi dir="ltr">D07,</bdi> <bdi dir="ltr">D08,</bdi> <bdi dir="ltr">D09,</bdi> <bdi dir="ltr">D10,</bdi> <bdi dir="ltr">D11</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2:</bdi> שורות אלה ושדה היעד שלהן; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5–</bdi>§<bdi dir="ltr">6</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M4:</bdi> סימולציית עונה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">13</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A04,</bdi> <bdi dir="ltr">B01,</bdi> <bdi dir="ltr">B09,</bdi> <bdi dir="ltr">B11,</bdi> <bdi dir="ltr">C02,</bdi> <bdi dir="ltr">C03,</bdi> <bdi dir="ltr">C04,</bdi> <bdi dir="ltr">C05,</bdi> <bdi dir="ltr">C06,</bdi> <bdi dir="ltr">C08,</bdi> <bdi dir="ltr">C09,</bdi> <bdi dir="ltr">C13,</bdi> <bdi dir="ltr">D02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2:</bdi> שורות אלה ושדה היעד שלהן; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5–</bdi>§<bdi dir="ltr">6</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M5:</bdi> טבלה בלבד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">8</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">A04,</bdi> <bdi dir="ltr">B01,</bdi> <bdi dir="ltr">B11,</bdi> <bdi dir="ltr">C02,</bdi> <bdi dir="ltr">C03,</bdi> <bdi dir="ltr">C04,</bdi> <bdi dir="ltr">C05,</bdi> <bdi dir="ltr">C09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2:</bdi> שורות אלה ושדה היעד שלהן; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5–</bdi>§<bdi dir="ltr">6</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M6:</bdi> השוואה ושילוב</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">4</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B11,</bdi> <bdi dir="ltr">C03,</bdi> <bdi dir="ltr">C04,</bdi> <bdi dir="ltr">C06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2:</bdi> שורות אלה ושדה היעד שלהן; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5–</bdi>§<bdi dir="ltr">6</bdi></td></tr></tbody></table></div></div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S3</bdi> — כל רכיב: מה הוא מקבל, מה הוא חוזה ואיך הוא נבדק</h2>
<p>״נבנה״ אינו אומר שהמערכת הסופית נבחנה. <bdi dir="ltr">M0</bdi> ותשתיות הניסויים קיימים; שאר רכיבי השרשרת הם תכנון. לכל רכיב מתוכנן מצוין מה עדיין פתוח, בלי לבחור שיטה או סף. המקורות המחייבים: <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">4–</bdi>§<bdi dir="ltr">6,</bdi> <bdi dir="ltr">PROJECT_PRINCIPLES</bdi> §<bdi dir="ltr">6</bdi>; פרטי הסרגלים הקיימים: <bdi dir="ltr">EXPERIMENTS</bdi> <bdi dir="ltr">EXP-001/002/003</bdi> ומחברת <bdi dir="ltr">03,</bdi> <bdi dir="ltr">75,</bdi> <bdi dir="ltr">93,</bdi> <bdi dir="ltr">117,</bdi> <bdi dir="ltr">149,</bdi> <bdi dir="ltr">210.</bdi></p>
<p>כל רכיב נלמד משתמש בעבר בלבד ובמשקלי זמן שנבדקים מראש (ה-ת<bdi dir="ltr">7).</bdi> תחזית של רכיב נלמד שמוזנת לאימון הרכיב הבא צריכה <bdi dir="ltr">OOF</bdi> כרונולוגי מקונן, כולל כלל חימום. נוסחה סיבתית קבועה של <bdi dir="ltr">M0</bdi> אינה תחזית נלמדת. אין כאן אישור להתחיל לאמן אף רכיב.</p>
<div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">רכיב</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה זה, במילים — מה הוא חוזה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">קלט</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">פלט</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">שיטת האימון / הבנייה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מדד ההערכה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הסרגל לנצח</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מחברת / מקור</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מצב</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מה היה ידוע לפני המשחק; אינו חוזה תוצאה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תוצאות קודמות ליום <bdi dir="ltr">as_of,</bdi> שווי <bdi dir="ltr">t−1,</bdi> לוח ופורמט; בלוקים שמורים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">prematch_v1,</bdi> שורת קבוצה־משחק; <bdi dir="ltr">51</bdi> עמודות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">טבלה וכללים דטרמיניסטיים; <bdi dir="ltr">Elo</bdi> וחלונות מכוילים בניסויים נפרדים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אימות טבלה, דליפה, מפתחות, טיפוסים ו־<bdi dir="ltr">checksums</bdi>; אין <bdi dir="ltr">loss</bdi> יחיד ל־<bdi dir="ltr">M0</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שחזור כללי הליגה; סרגלי <bdi dir="ltr">Elo</bdi> ושערים נפרדים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">03</bdi>; <bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">1–</bdi>§<bdi dir="ltr">9</bdi>; <bdi dir="ltr">DATA_DICTIONARY</bdi> <bdi dir="ltr">prematch</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נבנה</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חשיבות תחרותית של המשחק לכל צד, לא מצב פסיכולוגי</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M0-A,</bdi> מעטפת נקודות, דגלים, חוקים ולוח; <bdi dir="ltr">Elo</bdi> מכויל כקנה מידה ולסימולטור</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">home_motivation</bdi> / <bdi dir="ltr">away_motivation,</bdi> בטווח <bdi dir="ltr">0–1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הגדרה פתוחה למחקר: חוקים/מרחק ליעד/רגישות תרחישים; אם יש פרמטרים נלמדים — עבר בלבד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין תווית אמת; תוקף דוגמאות ובסוף <bdi dir="ltr">ablation</bdi> ב־<bdi dir="ltr">M2</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בלי מוטיבציה / פיצ׳רי טבלה שקופים / ציון</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">04</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1</bdi> ו-§<bdi dir="ltr">7</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">פתוח: בחירת הגדרה</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M2</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כמה שערים תבקיע כל קבוצה מול היריבה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">פיצ׳רי <bdi dir="ltr">M0</bdi> הרלוונטיים + ציון <bdi dir="ltr">M1</bdi> ב־<bdi dir="ltr">OOF,</bdi> מול חלופות בלי ציון</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">λ בית ו-λ חוץ; גם כוח מול יריב ניטרלי</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">XGBoost</bdi> <bdi dir="ltr">count:poisson</bdi> מתוכנן מול <bdi dir="ltr">GLM/</bdi>מודל התקפה־הגנה; כיוונון ו־<bdi dir="ltr">ablation</bdi> בפיתוח</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">Poisson</bdi> <bdi dir="ltr">deviance</bdi> וכיול λ לפי דצילים; <bdi dir="ltr">NLL</bdi> של <bdi dir="ltr">EXP-002</bdi> כסרגל תואם־מדגם</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">GLM</bdi> <bdi dir="ltr">Poisson</bdi> / <bdi dir="ltr">Dixon-Coles</bdi> קלאסי; <bdi dir="ltr">EXP-002</bdi> <bdi dir="ltr">NLL</bdi> <bdi dir="ltr">1.409455</bdi> הוא סרגל קיים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">05</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M2</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">93,</bdi> <bdi dir="ltr">117</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בתכנון; מודל השופט של <bdi dir="ltr">EXP-002</bdi> כבר נבנה</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M3</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">התפלגות תוצאות מדויקות והסתברויות בית/תיקו/חוץ</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">רק λ של <bdi dir="ltr">M2</bdi> מחוץ למדגם ופרמטרי תיקון מוגבלים, ה-ת<bdi dir="ltr">5</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מטריצת <bdi dir="ltr">6×6</bdi> עם <bdi dir="ltr">5+</bdi>; <bdi dir="ltr">H/D/A</bdi> ממנה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">Poisson</bdi> עצמאי; השוואת <bdi dir="ltr">Dixon-Coles</bdi> לניפוח תיקו ב־<bdi dir="ltr">walk-forward</bdi>; לא לומד שוב את כל הפיצ׳רים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">RPS,</bdi> <bdi dir="ltr">log</bdi> <bdi dir="ltr">loss</bdi> ל־<bdi dir="ltr">H/D/A</bdi> ולתוצאה מדויקת, כיול</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">Poisson</bdi> בלי תיקון, <bdi dir="ltr">B0</bdi> ו־<bdi dir="ltr">B2</bdi>; <bdi dir="ltr">B4</bdi> לבדיקת אובדן מידע</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">06</bdi> ו־<bdi dir="ltr">06-B</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M3</bdi>; ה-ס<bdi dir="ltr">12–</bdi>ה-ס<bdi dir="ltr">14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בתכנון; פרשנות תא הזנב דורשת הבהרה ב־<bdi dir="ltr">S4</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M4</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לאן העונה תגיע אם נגריל את יתר המשחקים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מצב <bdi dir="ltr">M0</bdi> ולוח שנותר; <bdi dir="ltr">M1–M3</bdi> שמורים ומגורסנים; <bdi dir="ltr">rules.py</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">התפלגות נקודות ומיקום בפיצול ובסיום; בעונת היעד מטריצת <bdi dir="ltr">14×14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">Monte</bdi> <bdi dir="ltr">Carlo,</bdi> לא <bdi dir="ltr">ML</bdi> בפני עצמו; בכל מסלול עדכון טבלה, <bdi dir="ltr">Elo,</bdi> התקפה/הגנה ומוטיבציה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">RMSE</bdi> נקודות, שגיאת מיקום, ספירמן ולפי אזור; <bdi dir="ltr">RPS</bdi> מיקומים, <bdi dir="ltr">Brier</bdi> וכיול אירועים לפי <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">8</bdi>; גם <bdi dir="ltr">log</bdi> <bdi dir="ltr">loss</bdi> אירועים לפי <bdi dir="ltr">PRINCIPLES</bdi> §<bdi dir="ltr">6</bdi>; יציבות דגימה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B1</bdi> מתחילת עונה ומנקודות אמצע, ובדיקות הכיול של מחברת <bdi dir="ltr">02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">07</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M4,</bdi> §<bdi dir="ltr">8</bdi>; <bdi dir="ltr">PRINCIPLES</bdi> §<bdi dir="ltr">6</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בתכנון</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M5</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סוף העונה מהטבלה בלבד, בלי לדעת מי נגד מי</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נקודות, הפרש, מחזור/שלב, קבוצת פלייאוף ונקודות שאר קבוצות הפלייאוף; <bdi dir="ltr">prev_ppm/</bdi>הפרש <bdi dir="ltr">t−1</bdi> מועמדים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">התפלגות נקודות ומיקום</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">השיטה פתוחה: קצב נקודות, <bdi dir="ltr">Ridge,</bdi> <bdi dir="ltr">quantile</bdi> או חלופות מתועדות; כל פורמטי הפיתוח עם נרמול</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שגיאות נקודות ומיקום לפי מועד ואזור, <bdi dir="ltr">RPS</bdi> מיקומים, <bdi dir="ltr">Brier</bdi> וכיול אירועים; קריטריוני קבלה במשימת הרכיב</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B1:</bdi> הטבלה אחרי <bdi dir="ltr">k</bdi> היא טבלת הסיום</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">08</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M5</bdi> ו-§<bdi dir="ltr">8</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M5</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">פתוח: שיטה וחוזה התפלגות</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M6</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">האם שילוב תחזיות העונה מוסיף מעבר לכל אחת</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תחזיות <bdi dir="ltr">M4/M5</bdi> מחוץ למדגם והשלב; לא תוצאות העתיד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">התפלגות עונה משולבת וחוקית</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מעט פרמטרים, משקל תלוי מועד או <bdi dir="ltr">M5</bdi> כהשוואה בלבד; בחירה רק על <bdi dir="ltr">OOF</bdi> מקונן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מדדי הטבלה והאירועים, כולל <bdi dir="ltr">RPS</bdi> מיקומים, <bdi dir="ltr">Brier</bdi> וכיול לפי אזור; בודקים את השרשרת כולה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M4</bdi> לבדו, <bdi dir="ltr">M5</bdi> לבדו ו־<bdi dir="ltr">B1,</bdi> לא רק פער ביניהם</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">09</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M6,</bdi> §<bdi dir="ltr">8</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">פתוח: עצם השילוב וצורתו</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B0</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מה קורה בליגה בלי לדעת אילו קבוצות משחקות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שכיחויות בית/תיקו/חוץ בעונות האימון בלבד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שלוש הסתברויות קבועות למשחקי קיפול</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ספירת תוצאות באימון מחדש לכל קיפול</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">log</bdi> <bdi dir="ltr">loss</bdi> ו־<bdi dir="ltr">RPS</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שוויון הסתברויות לתצוגת הסרגל; <bdi dir="ltr">B0</bdi> עצמו בסיס לרכיבים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">01/03</bdi>; <bdi dir="ltr">EXPERIMENTS</bdi> <bdi dir="ltr">EXP-001</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">40–43,</bdi> <bdi dir="ltr">75</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נבנה</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">טבלת אשתקד או הטבלה הנוכחית מנבאות את הסיום</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">טבלה ספורטיבית מחושבת, בתחילת עונה או בנקודת <bdi dir="ltr">as_of</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בסיס נקודתי למיקום/טבלה; המרה להתפלגות טרם הוגדרה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין אימון לבסיס הישיר; נדרש חוזה לעולות ולפורמטים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שגיאת מיקום/נקודות לפי <bdi dir="ltr">k</bdi> ואזור; הסתברויות רק אחרי הגדרתן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סרגל ל־<bdi dir="ltr">M4/M5/M6,</bdi> ה-ס<bdi dir="ltr">8</bdi> וה-ס<bdi dir="ltr">9</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">07–09</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M5/M6</bdi>; <bdi dir="ltr">02,</bdi> <bdi dir="ltr">84–85,</bdi> <bdi dir="ltr">104–113</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בתכנון כמוצר סופי; <bdi dir="ltr">EDA</bdi> וכללי טבלה קיימים</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B2</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הסתברויות תוצאה על סמך זיכרון תוצאות <bdi dir="ltr">Elo</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">פער <bdi dir="ltr">Elo</bdi> לפני המשחק; המרה סדורה מאימון הקיפול</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">P(H),</bdi> <bdi dir="ltr">P(D),</bdi> <bdi dir="ltr">P(A)</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">רשת <bdi dir="ltr">EXP-001</bdi> קבועה מראש; <bdi dir="ltr">E0050</bdi> נבחר והמרה מותאמת בעבר בלבד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">log</bdi> <bdi dir="ltr">loss</bdi> לבחירה, <bdi dir="ltr">RPS</bdi> וכיול לדיווח</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B0:</bdi> <bdi dir="ltr">1.084974</bdi> מול <bdi dir="ltr">1.020424</bdi> של <bdi dir="ltr">E0050</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">03</bdi>; ה-ת<bdi dir="ltr">1/</bdi>ה-ת<bdi dir="ltr">11</bdi>; <bdi dir="ltr">EXP-001</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">75</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נבנה</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B3</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מודל תוצאה קטן ושקוף על מעט פיצ׳רים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שניים–שלושה פיצ׳רי <bdi dir="ltr">prematch</bdi> שייבחרו מראש</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">P(H),</bdi> <bdi dir="ltr">P(D),</bdi> <bdi dir="ltr">P(A)</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">רגרסיה לוגיסטית סדורה על עונות אימון</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">RPS/log</bdi> <bdi dir="ltr">loss</bdi> וכיול</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B0</bdi> ו־<bdi dir="ltr">B2</bdi> על אותו מדגם</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">05–06</bdi>; <bdi dir="ltr">PROJECT_PRINCIPLES</bdi> §<bdi dir="ltr">6</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">8</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בתכנון כסרגל סופי; תשתית סדורה ומודלי <bdi dir="ltr">EXP-003</bdi> קיימים</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B4</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תוצאה ישירות מכל המידע, כדי לבדוק מה אובד בדרך דרך λ</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כל הפיצ׳רים המותרים, לרבות <bdi dir="ltr">M1</bdi> רק ב־<bdi dir="ltr">OOF</bdi>; בלי פיצ׳רים שנדחו</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">P(H),</bdi> <bdi dir="ltr">P(D),</bdi> <bdi dir="ltr">P(A),</bdi> בלי מטריצת תוצאות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">XGBoost</bdi> סיווג ישיר; השוואה הוגנת על אותם קיפולים ומשחקים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">RPS/log</bdi> <bdi dir="ltr">loss</bdi> וכיול</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M2→M3</bdi>; גם <bdi dir="ltr">B2/B3</bdi> לבקרת סרגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">05–06,</bdi> מיקום מדויק לדיון; ה-ת<bdi dir="ltr">5</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M3</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בתכנון; אין תפקיד <bdi dir="ltr">B4</bdi> ב־<bdi dir="ltr">registry</bdi> עדיין</td></tr></tbody></table></div>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S3</bdi> — משפחות פיצ׳רים מול הרכיבים</h2>
<p>הטבלה מסכמת את התכנון ב־<bdi dir="ltr">FEATURES,</bdi> עם בדיקת הרישום בפועל ב־<bdi dir="ltr">registry.py.</bdi> היא אינה רשימת קלטים שאושרה מחדש. ✅ = ברירת מחדל/קנה מידה נדרש; 🔬 = מועמד; ⛔ = נדחה; — = אין הקצאה מוגדרת כאן. היעדר רישום למודל עתידי אינו החלטת פסילה.</p>
<p><bdi dir="ltr">M4</bdi> צורך את הפיצ׳רים דרך <bdi dir="ltr">M2</bdi> ומעדכן אותם במסלול: הסימון בעמודה שלו הוא התכנון הנובע מ־<bdi dir="ltr">M2,</bdi> ולא בחירת פיצ׳רים עצמאית. <bdi dir="ltr">B4</bdi> כולו מתוכנן, ולכן 🔬 בעמודתו פירושו משפחה מותרת לניסוי, ולא טענה שכבר נרשמה. המעמד של <bdi dir="ltr">Elo</bdi> ב־<bdi dir="ltr">M1</bdi> מתייחס לקנה המידה המכויל, לא לפיצ׳ר שמייצר את הגדרת החשיבות.</p>
<p>הפערים בין התכנון לרישום מפורטים ב־<bdi dir="ltr">S4:</bdi> למשל, <bdi dir="ltr">gf/ga</bdi> עדיין 🔬 ברישום, לשווי אין חריג ✅ ב־<bdi dir="ltr">M4,</bdi> ו־<bdi dir="ltr">M5</bdi> מקבל ברישום יותר מהקטלוג. אין להשתמש בטבלה זו כפתרון שקט לפערים האלה.</p>
<div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">משפחה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789"><bdi dir="ltr">M1</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789"><bdi dir="ltr">M2</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789"><bdi dir="ltr">M4</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789"><bdi dir="ltr">M5</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789"><bdi dir="ltr">B4</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור / גבול</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">טבלה ומעטפת</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נקודות, הפרש, מיקום, פערים ומשחקים שנותרו</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1/M2/M5</bdi>; <bdi dir="ltr">registry</bdi> <bdi dir="ltr">TABLE_STATE_COLUMNS</bdi>; <bdi dir="ltr">M1</bdi> משווה לחלופה השקופה</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">דגל אין יעד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הוכחה מתמטית שמאבקי היעד סגורים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0/M1</bdi>; <bdi dir="ltr">registry</bdi> <bdi dir="ltr">nothing_to_play</bdi> חריג <bdi dir="ltr">M2</bdi>; ההנחה בעדכון <bdi dir="ltr">Elo</bdi> נדחתה</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">Elo</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כוח מתוצאות והסתברויות מכוילות כקנה מידה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1/M2</bdi>; <bdi dir="ltr">registry</bdi> <bdi dir="ltr">elo</bdi> (שם כולו 🔬, <bdi dir="ltr">S4)</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">התקפה והגנה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">gf</bdi> של קבוצה מול <bdi dir="ltr">ga</bdi> של היריבה, <bdi dir="ltr">ewma5</bdi> לפי בית/חוץ</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0/M2/M4</bdi>; <bdi dir="ltr">registry</bdi> <bdi dir="ltr">FORM_COLUMNS</bdi> ו־<bdi dir="ltr">PREMATCH_OPPONENT_COLUMNS</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נקודות בחלון</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כושר נקודות, להבדיל מהתקפה והגנה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES:</bdi> רק ניסוי <bdi dir="ltr">XGBoost</bdi> ב־<bdi dir="ltr">M2</bdi>; אין החלטת הכללה ב־<bdi dir="ltr">M4.</bdi> <bdi dir="ltr">registry</bdi> רחב יותר, <bdi dir="ltr">S4</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שווי ממוצע <bdi dir="ltr">t−1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נתח שווי ו־<bdi dir="ltr">log</bdi> יחס קבוצה/יריבה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ת<bdi dir="ltr">14</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0</bdi>; <bdi dir="ltr">registry</bdi> חריג ✅ ל־<bdi dir="ltr">M2</bdi> בלבד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שווי כולל <bdi dir="ltr">t−1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סכום שווי במקום ממוצע לשחקן</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">⛔</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">⛔</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">⛔</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ת<bdi dir="ltr">14</bdi>; לא ב־<bdi dir="ltr">prematch</bdi>; נשאר בארכיון <bdi dir="ltr">season_context,</bdi> עדיין 🔬 ברישום</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מיקום <bdi dir="ltr">t−1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מיקום ספורטיבי יחסי ו־<bdi dir="ltr">log</bdi> יחס</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0</bdi>; ה-ת<bdi dir="ltr">13/</bdi>ה-ת<bdi dir="ltr">14</bdi>; <bdi dir="ltr">registry</bdi> כולל גם <bdi dir="ltr">M5,</bdi> <bdi dir="ltr">S4</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נקודות/הפרש <bdi dir="ltr">t−1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קצב והפרש שערים של העונה הקודמת</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0/M5</bdi>; <bdi dir="ltr">registry</bdi> <bdi dir="ltr">SEASON_CONTEXT_COLUMNS</bdi> וארבעת ההפרשים</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">עולות והשלמות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">דגל עולה, שווי מושלם וסימון שאין עבר</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0</bdi>; <bdi dir="ltr">registry</bdi> כולל גם <bdi dir="ltr">M5,</bdi> אך הקטלוג לא מגדיר תפקיד לכל דגל</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שלב ומרחק לסיום</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סדיר/פלייאוף וכמה נותר</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1/M2/M5</bdi>; <bdi dir="ltr">registry</bdi> <bdi dir="ltr">stage/rounds_to_stage_end</bdi> מסמן ✅ גם ל־<bdi dir="ltr">M1</bdi>; הגדרת המרחק ב־<bdi dir="ltr">S4</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בית/חוץ</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">איפה המשחק ביחס לקבוצה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">✅</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0/M2</bdi>; <bdi dir="ltr">registry</bdi> <bdi dir="ltr">is_home</bdi>; <bdi dir="ltr">M5</bdi> אינו נכנס למשחק בודד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ציון <bdi dir="ltr">M1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חשיבות תחרותית לפי הגדרה שתיבדק</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">—</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">🔬</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1/M2</bdi> וה-ת<bdi dir="ltr">5</bdi>; טרם נבנה/נרשם; אין החלטה אם להשאירו</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תוך־משחק וזהות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אדום, מחצית, תזמון, מאמן, דרבי, אצטדיון, מזהה קבוצה כקטגוריה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">⛔</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">⛔</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">⛔</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">⛔</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">⛔</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> ״נבדק ולא נכנס״, ה-ס<bdi dir="ltr">13</bdi>; מזהים מותרים כמפתחות בלבד</td></tr></tbody></table></div>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S4</bdi> — פערים בין המסמכים לבין מה שנבנה</h2>
<p>זו בדיקה עצמאית של החוזים מול המימוש, לא רשימת תיקונים שבוצעו. לא שיניתי מסמך, <bdi dir="ltr">registry,</bdi> קוד או תא קיים. חלק מהפריטים הם פערי ניסוח/תפקיד, וחלקם חוזים שטרם הוגדרו לרכיבים עתידיים; הם אינם כשל בבדיקות המחברת.</p>
<p>בפרט, נוכחות שדה בבלוק קפוא אינה מחייבת שהוא ייכנס למודל. הבעיה היא שהרישום כיום לא תמיד מבדיל בין ארכיון, שדה זמין לניסוי וקלט שמותר למודל ספציפי. גם המלצות סוף מחברת <bdi dir="ltr">02</bdi> הן תיעוד היסטורי: החלטות מאוחרות, למשל ה-ת<bdi dir="ltr">14,</bdi> גוברות עליהן בלי צורך לשכתב את העבר.</p>
<div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מזהה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה כתוב</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">איפה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה נבנה / נמצא בפועל</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">איזה קובץ/מפרט דורש טיפול</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D01</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מעמד התקפה והגנה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">ewma5/venue</bdi> ברירת מחדל, ✅.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0</bdi>; ה-ת<bdi dir="ltr">8</bdi>; <bdi dir="ltr">EXP-002</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FORM_COLUMNS</bdi> ורישום <bdi dir="ltr">opp_gf/opp_ga</bdi> עדיין 🔬 ב־<bdi dir="ltr">M2/M4,</bdi> בלי חריג לנבחרים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">ligat/features/registry.py</bdi>; החלטה על היקף החריג, בלי שינוי כאן</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כושר בנקודות: זמינות אינה היתר שימוש</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא נכנס ל־<bdi dir="ltr">M0</bdi>; מועמד רק ל־<bdi dir="ltr">XGBoost</bdi> ב־<bdi dir="ltr">M2.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0</bdi>; <bdi dir="ltr">EXP-002</bdi>; מחברת <bdi dir="ltr">03,</bdi> <bdi dir="ltr">108,</bdi> <bdi dir="ltr">117</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">pts_venue_ewma5</bdi> כן ב־<bdi dir="ltr">SELECTED_FORM_COLUMNS</bdi> וב־<bdi dir="ltr">prematch</bdi>; <bdi dir="ltr">registry</bdi> מאפשר <bdi dir="ltr">pts</bdi> ל־<bdi dir="ltr">M2/M4.</bdi> <bdi dir="ltr">DATA_DICTIONARY</bdi> כבר מתאר אותו בטבלה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> / <bdi dir="ltr">DATA_DICTIONARY</bdi> / <bdi dir="ltr">registry</bdi> / <bdi dir="ltr">pipeline.py:</bdi> להכריע אם שמירה זמינה לבנצ׳מרק היא הכוונה, ולבודד קלט למודל; לא להסיר עמודה בשקט</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D03</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שווי כולל בארכיון לעומת פיצ׳ר מותר</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">value_total</bdi> יצא ולא נכנס ל־<bdi dir="ltr">prematch</bdi>; נשאר בבלוק הקפוא.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0</bdi>; ה-ת<bdi dir="ltr">14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ההחרגה מ־<bdi dir="ltr">prematch</bdi> אכן בוצעה. ברישום עצמו <bdi dir="ltr">prev_value_share_total</bdi> עדיין 🔬 ל־<bdi dir="ltr">M2/M4/M5,</bdi> בלי פסילה לפי מודל.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">registry.py</bdi> וחוזה <bdi dir="ltr">allowlist</bdi> לכל מודל; להשאיר את הבלוק הקפוא כארכיון</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D04</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M5</bdi> הוא טבלה בלבד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין משחקים בודדים; <bdi dir="ltr">t−1</bdi> מועמדים רק נקודות/הפרש.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M5</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M5</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הרישום מאפשר ל־<bdi dir="ltr">M5</bdi> גם שווי, <bdi dir="ltr">log</bdi> יחסי יריבות, <bdi dir="ltr">wins/draws/losses</bdi> ושערים נפרדים. הוא רחב מחוזה המאקרו, ואין עדיין <bdi dir="ltr">M5</bdi> שמסנן אותם.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">registry.py</bdi> / <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M5</bdi>; להבחין ברישום טכני מול קלט מודל מאושר</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D05</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ברירת מחדל תלויה ברכיב</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שווי ממוצע ו־<bdi dir="ltr">gf/ga</bdi> מתוכננים ברירת מחדל ב־<bdi dir="ltr">M2,</bdi> ו־<bdi dir="ltr">M4</bdi> משתמש בהם דרכו; <bdi dir="ltr">Elo</bdi> הוא קנה מידה מחייב ל־<bdi dir="ltr">M1.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M0/M1/M4</bdi>; ה-ת<bdi dir="ltr">1/</bdi>ה-ת<bdi dir="ltr">8/</bdi>ה-ת<bdi dir="ltr">14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חריג ✅ לשווי קיים רק ל־<bdi dir="ltr">M2</bdi>; <bdi dir="ltr">Elo</bdi> כולו 🔬 גם ב־<bdi dir="ltr">M1.</bdi> הרישום אינו מבחין בקנה מידה לעומת פיצ׳ר, או בשימוש דרך <bdi dir="ltr">M2.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">registry.py</bdi> / <bdi dir="ltr">FEATURES:</bdi> להגדיר משמעות <bdi dir="ltr">models</bdi> ו־<bdi dir="ltr">model_status</bdi> לכל תפקיד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D06</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B4</bdi> מול רישום פיצ׳רים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B4</bdi> בודק סיווג ישיר עם כל הפיצ׳רים המותרים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M3</bdi>; ה-ת<bdi dir="ltr">5</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M3</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין <bdi dir="ltr">B4</bdi> באף <bdi dir="ltr">models</bdi> ב־<bdi dir="ltr">registry.</bdi> <bdi dir="ltr">validate_prematch_feature_columns</bdi> בודק רישום כללי, לא היתר לפי מודל/מעמד.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">registry.py</bdi> וחוזה קלט עתידי של <bdi dir="ltr">B4</bdi>; זה חוסר תכנון למימוש עתידי, לא תקלה בריצה הנוכחית</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D07</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ציפיית <bdi dir="ltr">Elo</bdi> מול הסתברות לתוצאה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">M1</bdi> צריך קנה מידה מכויל לתוצאה ולסימולטור.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1</bdi> ו-§<bdi dir="ltr">7.6</bdi>א; ה-ת<bdi dir="ltr">1</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">prematch</bdi> שומר <bdi dir="ltr">elo_expected</bdi> (ציון <bdi dir="ltr">1/0.5/0</bdi> צפוי), לא <bdi dir="ltr">P(win/draw/loss).</bdi> הסתברויות מכוילות כן שמורות ב־<bdi dir="ltr">EXP-001</bdi> לשורות הוולידציה; אין חיבור מלא שלהן כחוזה קלט <bdi dir="ltr">M1.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מפרט מחברת <bdi dir="ltr">04</bdi> ו־<bdi dir="ltr">FEATURES</bdi> / <bdi dir="ltr">DATA_DICTIONARY</bdi>; לחבר <bdi dir="ltr">OOF</bdi> וכיול לפי <bdi dir="ltr">cutoff,</bdi> עם כלל חימום, לא לקרוא ל־<bdi dir="ltr">elo_expected</bdi> ״סיכוי לנצח״</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D08</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">טבלת משחק נוכחי אינה כל לוח העתיד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סימולטור החשיבות של <bdi dir="ltr">M1</bdi> דורש את המשחקים שנותרו ואת <bdi dir="ltr">rules.py.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">prematch</bdi> כולל ספירות וקבוצה/יריבה של המשחק הנוכחי, לא את כל צמדי העתיד או מחולל לוח פלייאוף. מנוע <bdi dir="ltr">M1</bdi> טרם נבנה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מפרט מחברת <bdi dir="ltr">04:</bdi> מקור לוח <bdi dir="ltr">K</bdi> שהיה ידוע ב־<bdi dir="ltr">as_of,</bdi> תזמון פיצול וגרסת חוקים</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D09</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מרחק מסוף שלב: מחזור לעומת משחק</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">EDA</bdi> נבדק מחזור ממוספר אחרון פחות הנוכחי; לא מספר משחקים לקבוצה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">02,</bdi> <bdi dir="ltr">130–131</bdi>; <bdi dir="ltr">DATA_DICTIONARY</bdi> <bdi dir="ltr">C</bdi> <bdi dir="ltr">draw_rates_by_distance</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">rounds_to_stage_end</bdi> הוא <bdi dir="ltr">games_remaining_stage</bdi> פחות המשחק הנוכחי. מקור <bdi dir="ltr">03,</bdi> <bdi dir="ltr">23–24,</bdi> <bdi dir="ltr">188</bdi>; לא בהכרח אותו מדד במקרה דחייה/טכני.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> / <bdi dir="ltr">DATA_DICTIONARY</bdi> / מפרט <bdi dir="ltr">M1-M2:</bdi> להכריע מה המדד ולתעד את ההבדל, בלי לשנות <bdi dir="ltr">03</bdi> כאן</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D10</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נפח שער אימות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">10,040</bdi> שורות רשמיות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M0-A</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">באימות ההנדסי המאושר <bdi dir="ltr">9,956</bdi> שורות בעונות <bdi dir="ltr">8–27</bdi>; <bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">4</bdi> ופלט <bdi dir="ltr">03,</bdi> <bdi dir="ltr">8.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION.md</bdi> בלבד, בשלב תיקוני המסמכים</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D11</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מזהה ואצטדיון כקטגוריה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שורת ״קבוצות עולות״ עדיין מציגה מזהי קבוצה ואצטדיון כקטגוריאליים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M2</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> ״נבדק ולא נכנס״ פוסל אותם; בפלטים הם אינם פיצ׳רים קטגוריאליים מאושרים. <bdi dir="ltr">team_id</bdi> משמש מפתח.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION.md:</bdi> לנקות ניסוח ישן; אין צורך להוסיף קלט למימוש</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D12</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אירופה: קו שמרני אינו יעד הסתברותי</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">w_europe</bdi> קטן וקבוע מוזכר כקלט <bdi dir="ltr">M1.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">7.4</bdi> וה-ת<bdi dir="ltr">10</bdi> מחריגים חיזוי אירופה; הדגל כולל קו מקום רביעי כשמרנות בלבד, בלי מידע גביע. אין <bdi dir="ltr">w_europe</bdi> ב־<bdi dir="ltr">prematch.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1</bdi> / מפרט <bdi dir="ltr">04:</bdi> להכריע אם מוחקים את השורה הישנה או מגדירים מחקר נפרד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D13</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מעקב החלטות שלא עודכן אחרי מחברת <bdi dir="ltr">03</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">עוד פתוחים מימוש <bdi dir="ltr">S1/S5/as_of,</bdi> הסבר <bdi dir="ltr">Elo</bdi> ובדיקת השווי; טבלת העקרונות מסמנת את כל <bdi dir="ltr">t−1</bdi> כמועמדים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> ״פתוח למחברת <bdi dir="ltr">03</bdi>״; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M0</bdi> ״פתוח מ־<bdi dir="ltr">02</bdi>״; <bdi dir="ltr">PROJECT_PRINCIPLES</bdi> §<bdi dir="ltr">5</bdi> ו-§<bdi dir="ltr">8</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אלה נבנו/הוצגו ב־<bdi dir="ltr">03</bdi> <bdi dir="ltr">A–C</bdi>; <bdi dir="ltr">value_avg</bdi> כבר ✅ לפי ה-ת<bdi dir="ltr">14.</bdi> ההחלטות המעודכנות עצמן כן מופיעות בהמשך המסמכים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">FEATURES</bdi> / <bdi dir="ltr">VISION</bdi> / <bdi dir="ltr">PROJECT_PRINCIPLES:</bdi> לעדכן רשימות מצב; לבדוק בנפרד אם השוואת מיקום רשמי/ספורטיבי עדיין דורשת דוח נוסף</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מילון שדות הבלוקים</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">DATA_DICTIONARY</bdi> מתאר את <bdi dir="ltr">prematch</bdi> לפי קבוצות ואת ארבעת ההפרשים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">DATA_DICTIONARY,</bdi> ״מחברת <bdi dir="ltr">03</bdi>״</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אין שם פירוט מלא של כל שדה, יחידותיו ו־<bdi dir="ltr">NA</bdi> בארבעת הבלוקים. <bdi dir="ltr">T-010</bdi> והקוד מכילים מידע נוסף. טבלת <bdi dir="ltr">rules</bdi> כוללת חצייה היסטורית, לעומת <bdi dir="ltr">form/prev_ppm</bdi> שהם <bdi dir="ltr">3/1/0</bdi> ללא חצייה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">DATA_DICTIONARY.md:</bdi> להשלים שדות ומונחים, לא לשנות את הבלוקים</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D15</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מבחן הירידה בדגל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הדגל שמרני ביחס ליעדים שהוגדרו.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ת<bdi dir="ltr">3/</bdi>ה-ת<bdi dir="ltr">10</bdi>; מחברת <bdi dir="ltr">03,</bdi> <bdi dir="ltr">17</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מגבלת מבחן הירידה מתועדת: ההכרעה על קו הסיכון אינה בהכרח הבחנה בין מבחן לירידה ישירה בפורמטים ההיסטוריים. ההערה נשארת; אין שינוי <bdi dir="ltr">rules.py.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מפרט <bdi dir="ltr">M1:</bdi> להגדיר יעדים ומידת ההוכחה; להותיר את מגבלת הגרסה ברורה</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D16</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">פענוח <bdi dir="ltr">5+:5+</bdi> לתוצאת משחק</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">H/D/A</bdi> מסכום משולשים ואלכסון מטריצה, ו־<bdi dir="ltr">5+</bdi> נרשם כ־<bdi dir="ltr">5</bdi> בסימולציה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M3/M4</bdi>; <bdi dir="ltr">TECHNICAL_GUIDELINES</bdi> <bdi dir="ltr">score_matrix/outcome_probabilities</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תא <bdi dir="ltr">5+:5+</bdi> כולל גם <bdi dir="ltr">5:6</bdi> ו־<bdi dir="ltr">6:5.</bdi> <bdi dir="ltr">DATA_DICTIONARY</bdi> <bdi dir="ltr">D</bdi> כבר מבהיר ש־<bdi dir="ltr">all_draws</bdi> אינו אלכסון קטגוריות; <bdi dir="ltr">M3</bdi> הסופי טרם נבנה. מדובר באי־התאמה בחוזה הדיוק, לא שגיאה שהוכחה בתוצר <bdi dir="ltr">M3.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מפרט <bdi dir="ltr">06</bdi> ו־<bdi dir="ltr">VISION/TECHNICAL_GUIDELINES:</bdi> להכריע קירוב מפורש או פירוט פנימי של הזנב תוך שימור הפלט המאושר</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D17</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תווית מועמדות ישנה ב־<bdi dir="ltr">02</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בתצוגת התנאים נשארה השערת <bdi dir="ltr">walk-forward</bdi> עבור <bdi dir="ltr">SHARED.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מחברת <bdi dir="ltr">02,</bdi> <bdi dir="ltr">68</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בתאי <bdi dir="ltr">71/94</bdi> וב־<bdi dir="ltr">FEATURES</bdi> הוחלט ״נבדק ובוטל״. מקור ופלט התא הישן הוקפאו בסבב הזה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">02,</bdi> תצוגת תא <bdi dir="ltr">68,</bdi> רק במשימת תיקונים מאושרת</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D18</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תווית סיכום אדומים שמוצגת כעונה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סיכום כל התקופה מסומן <bdi dir="ltr">season_id=0,</bdi> לא עונה אמיתית.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">DATA_DICTIONARY</bdi> <bdi dir="ltr">D</bdi> <bdi dir="ltr">red_card_stats</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בטבלאות מחברת <bdi dir="ltr">02,</bdi> <bdi dir="ltr">205</bdi> ו־<bdi dir="ltr">207</bdi> מוצגת <bdi dir="ltr">1998/1999.</bdi> זה פגם תצוגה של שורת סיכום; אינו נתון מעונת מחקר נוספת.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תצוגת הבלוק המוקפא במחברת <bdi dir="ltr">02,</bdi> רק לאחר אישור; לא לשנות מקור נתונים</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D19</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה־<bdi dir="ltr">Elo</bdi> של מחברת <bdi dir="ltr">02</bdi> אינו <bdi dir="ltr">E0050</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">במחברת <bdi dir="ltr">02</bdi> <bdi dir="ltr">B04</bdi> נקרא ״אחרי <bdi dir="ltr">Elo</bdi>״, אך זה נטרול <bdi dir="ltr">EDA</bdi> עם דירוג אחר.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">02,</bdi> <bdi dir="ltr">67–68</bdi> ו־<bdi dir="ltr">S2</bdi> <bdi dir="ltr">B04</bdi>; <bdi dir="ltr">conditions.py:</bdi> <bdi dir="ltr">condition_comparison</bdi> → <bdi dir="ltr">elo_residuals</bdi> → <bdi dir="ltr">prematch_elo</bdi>; <bdi dir="ltr">E0050:</bdi> <bdi dir="ltr">EXP-001</bdi> ו־<bdi dir="ltr">03,</bdi> <bdi dir="ltr">75</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">B04</bdi> בלבד תלוי בדירוג הזה: השוואות <bdi dir="ltr">GHOST/LIMITED/SHARED/RELOCATED</bdi> ל־<bdi dir="ltr">NORMAL</bdi> משתמשות בשאריות ממנו. <bdi dir="ltr">prematch_elo</bdi> משתמש ב־<bdi dir="ltr">K=20</bdi> ובתוספת בית <bdi dir="ltr">50</bdi>; <bdi dir="ltr">E0050</bdi> ב־<bdi dir="ltr">K=10</bdi> ובתוספת בית <bdi dir="ltr">30.</bdi> גם האתחול/תזמון וההמרה שונים: נטרול <bdi dir="ltr">OLS</bdi> בתוך מדגם מול בנצ׳מרק סדור מכויל. <bdi dir="ltr">B05</bdi> (<bdi dir="ltr">02,</bdi> <bdi dir="ltr">70)</bdi> קורא מאותה מסגרת רק <bdi dir="ltr">condition</bdi> ותוצאות: אין לו תלות מספרית בדירוג. <bdi dir="ltr">C14</bdi> משתמש בקצב נקודות עבר, <bdi dir="ltr">C07</bdi> בכוח <bdi dir="ltr">gf/ga</bdi> מהסדיר, <bdi dir="ltr">D05–D07</bdi> ו־<bdi dir="ltr">D10–D11</bdi> בבדיקות <bdi dir="ltr">Poisson,</bdi> <bdi dir="ltr">D08–D09</bdi> בשכיחויות/מגמות תוצאות ו־<bdi dir="ltr">E06</bdi> במדדי <bdi dir="ltr">t−1</bdi>; אינם ממצאי ״אחרי <bdi dir="ltr">Elo</bdi>״ זה. המספרים של <bdi dir="ltr">B04</bdi> אינם עוברים אוטומטית ל־<bdi dir="ltr">E0050.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תיעוד פרק <bdi dir="ltr">B/</bdi>מפרט ניסוי עתידי: להבדיל גרסאות ולא לייחס ל־<bdi dir="ltr">E0050</bdi> את מספרי <bdi dir="ltr">B04.</bdi> אין החלפת <bdi dir="ltr">Elo</bdi> או חישוב מחדש בסבב הזה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">D20</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שווי–מיקום לעומת שווי–<bdi dir="ltr">Elo:</bdi> שני מקורות לחפיפה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E06</bdi> ב־<bdi dir="ltr">02:</bdi> מתאם שווי–מיקום <bdi dir="ltr">0.613.</bdi> ב־<bdi dir="ltr">03:</bdi> שווי–<bdi dir="ltr">Elo</bdi> כ־<bdi dir="ltr">0.76</bdi> בכל התקופה וכ־<bdi dir="ltr">0.86</bdi> בחמש האחרונות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">02,</bdi> <bdi dir="ltr">220–221</bdi> (<bdi dir="ltr">S2</bdi> <bdi dir="ltr">E06)</bdi>; <bdi dir="ltr">context.py:</bdi> <bdi dir="ltr">correlation_lv_rk</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">186</bdi>; <bdi dir="ltr">nb03_C5_correlations.md</bdi> §<bdi dir="ltr">4</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ב־<bdi dir="ltr">02:</bdi> <bdi dir="ltr">Pearson</bdi> בין <bdi dir="ltr">log</bdi> יחס נתח השווי הכולל אשתקד לבין פער המיקום הרשמי (חוץ פחות בית), על <bdi dir="ltr">2,930</bdi> משחקים ללא עולות/עבר חסר. ב־<bdi dir="ltr">03:</bdi> <bdi dir="ltr">Pearson</bdi> בין <bdi dir="ltr">value_avg_log_ratio</bdi> לבין <bdi dir="ltr">elo_diff</bdi> של <bdi dir="ltr">E0050,</bdi> כולל השלמות לעולות ושתי שורות למשחק; המקדם השמור <bdi dir="ltr">0.7579274,</bdi> ובחמש האחרונות <bdi dir="ltr">0.8615904.</bdi> אלה משתנים, הגדרות שווי ואוכלוסיות שונים. אין לפרש את ההבדל כמגמת התחזקות על אותו מדד או כהוכחת תוספת מעל <bdi dir="ltr">Elo.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הסברים והפניות בדיון התכנון: להבחין במפורש בין שני המתאמים; אין תיקון מספרים, קוד או קבצי תוצאה.</td></tr></tbody></table></div>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S5</bdi> — שאלות לדיון התכנון</h2>
<p>כל השאלות הן לדיון. אין כאן בחירת הגדרה, הורדת עמודה, ריצה חדשה או שינוי מעמד. המקורות הקיימים הם <bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">9</bdi> ״לדיון התכנון״ ומחברת <bdi dir="ltr">03,</bdi> <bdi dir="ltr">183–213</bdi>; שאלות החוזה הנוספות נובעות מ־<bdi dir="ltr">S2–S4.</bdi></p>
<div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">השאלה — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">למה חשובה</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">חלופות שאפשר לדון בהן</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מתי צריך להכריע</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">משחקים מתים: מה מוסיף הדגל מעבר לחוזק ולשלב?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">רק <bdi dir="ltr">245</bdi> משחקים באיחוד; תיקו <bdi dir="ltr">18.4%/18.1%</bdi> מול כ־<bdi dir="ltr">27.1%.</bdi> ממוצע <bdi dir="ltr">Elo</bdi> קרוב לאפס אינו בקרה מספקת.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">להשאיר דגל כקו בסיס; להשוות לפערי טבלה; לחקור רגישות לניצחון/תיקו עם בקרה עונתית. בלי לייחס סיבתיות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">9</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">205–208</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני מחברת <bdi dir="ltr">04.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מה חוזה ההסתברויות שמחברת <bdi dir="ltr">04</bdi> מקבלת?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">elo_expected</bdi> אינו <bdi dir="ltr">P(win).</bdi> אי אפשר לבדוק ״מעל הצפוי״ בלי להגדיר מה צפוי ועל אילו שנים יש <bdi dir="ltr">OOF.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לחבר תחזיות <bdi dir="ltr">EXP-001</bdi> למפתחות; לייצר מתאם כיול עם <bdi dir="ltr">cutoff</bdi> לכל חלון; כלל חימום מפורש. אם נבדקים שערים, להגדיר סרגל שערים נפרד.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S4</bdi> <bdi dir="ltr">D07</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1</bdi> ו-§<bdi dir="ltr">7.6</bdi>א; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">34–39,</bdi> <bdi dir="ltr">65</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני מחברת <bdi dir="ltr">04.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אילו מאבקים הדגל צריך לייצג?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מקום רביעי הוסף כשמרנות; מבחן ירידה ואירופה אינם אותו יעד. ״אין יעד״ תלוי בהגדרה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לשמור את יעד הגרסה הנוכחית; להגדיר יעדי מבחן/ישירה בנפרד; אירופה כקו מקום בלבד לעומת חיזוי אירופה ממש — להכרעת עמית.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ת<bdi dir="ltr">10</bdi>; <bdi dir="ltr">S4</bdi> <bdi dir="ltr">D12/D15</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">17</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני מחברת <bdi dir="ltr">04.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מחזורים או משחקים שנותרו?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">בדחייה ובמשחק טכני המרחק של ה־<bdi dir="ltr">EDA</bdi> אינו המונה של <bdi dir="ltr">M0.</bdi> למספר יש משמעות כדורגלנית שונה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">להשאיר מונה משחקים ולשנות תיאור; להוסיף בעתיד מרחק לוח ידוע; להשוות רק בניסוי מאושר, לא להחליף שדה קפוא.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S4</bdi> <bdi dir="ltr">D09</bdi>; <bdi dir="ltr">02,</bdi> <bdi dir="ltr">130–131</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">23–24</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני מחברת <bdi dir="ltr">04.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">איזה ייצוג להשאיר לעמודות הכפולות?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">games_remaining_stage</bdi> ו־<bdi dir="ltr">rounds_to_stage_end</bdi> נבדלים באחד; <bdi dir="ltr">stage</bdi> זהה; דגלי עולה/השלמה שונים ב־<bdi dir="ltr">35</bdi> שורות בלבד.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">להשאיר את כולן בזמינות ולהגדיר קלט לכל מודל; לבחור נציג ברשימת קלט, בלי למחוק בלוק קפוא; לבדוק רגישות משפחתית.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">03,</bdi> <bdi dir="ltr">187–188</bdi>; <bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">9</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני <bdi dir="ltr">05.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">האם <bdi dir="ltr">t−1</bdi> מוסיף מעל <bdi dir="ltr">Elo</bdi> גם כשהם מתכנסים?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חפיפה כ־<bdi dir="ltr">0.87</bdi> למדדי תוצאות; שווי מול <bdi dir="ltr">elo_diff</bdi> עולה מ־<bdi dir="ltr">0.758</bdi> לכל התקופה ל־<bdi dir="ltr">0.862</bdi> בחמש האחרונות. מתאם גבוה יכול לדור עם תוספת קטנה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">ablation</bdi> לפי משפחות וגם לפי תקופה; לשמור את המעמדים המאושרים עד ניסוי; לא להסיר לפי מתאם לבדו.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">9</bdi>; <bdi dir="ltr">03,</bdi> מטריצות תא <bdi dir="ltr">186</bdi>; דוח <bdi dir="ltr">nb03_C5_correlations.md</bdi> §<bdi dir="ltr">4</bdi>; ה-ת<bdi dir="ltr">14</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני <bdi dir="ltr">05.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">האם שווי מבדיל בין עולות?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כולן מתחילות ב־<bdi dir="ltr">Elo</bdi> <bdi dir="ltr">1450,</bdi> אבל מחברת <bdi dir="ltr">02</bdi> הראתה שונות בשווי. הממוצע יכול להסתיר עולה חזקה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לבדוק אינטראקציה מאושרת מראש; אבחון תת־קבוצה בלבד; להשאיר הכלל הפשוט עד שיש ראיה. לא לשנות את <bdi dir="ltr">E0050.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">9</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">203–204</bdi>; <bdi dir="ltr">02,</bdi> <bdi dir="ltr">37–44</bdi>; <bdi dir="ltr">EXP-001</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני <bdi dir="ltr">05.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">האם <bdi dir="ltr">t−1</bdi> חשוב בעיקר בתחילת העונה?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">החלונות לומדים בהדרגה והערכת אשתקד קבועה; תוספת שנתית ממוצעת עלולה להסתיר תפקיד מוקדם.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אבחון לפי מספר משחקים עם חיתוכים שנקבעים מראש; אינטראקציה במודל בלבד אם תיבדק; בלי מחקר על עונת ההחזקה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">9</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">84–87,</bdi> <bdi dir="ltr">213</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני <bdi dir="ltr">05.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מה מותר לכל מודל מתוך הטבלה המאוחדת?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">כושר בנקודות, <bdi dir="ltr">M5</bdi> ושווי כולל מדגימים פער בין נוכחות לרשות שימוש.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">רשימות קלט נפרדות עם תפקידי קנה מידה/פיצ׳ר/ארכיון; סטטוס מפורש לכל מודל; <bdi dir="ltr">B4</bdi> מקבל רק משפחות מותרות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S4</bdi> <bdi dir="ltr">D01–D06</bdi>; <bdi dir="ltr">FEATURES</bdi>; <bdi dir="ltr">registry.py</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני <bdi dir="ltr">05.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מה פירוש <bdi dir="ltr">H/D/A</bdi> עבור תא <bdi dir="ltr">5+:5+</bdi>?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קטגוריית זנב מאוחדת אינה יודעת אם שערי הצדדים שווים בדיוק.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קירוב מודע שבו כל התא תיקו, או פירוט פנימי של סדר השערים בזנב ושמירת <bdi dir="ltr">6×6</bdi> להצגה; נדרש אישור חוזה, בלי שינוי כעת.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S4</bdi> <bdi dir="ltr">D16</bdi>; ה-ס<bdi dir="ltr">12</bdi>; <bdi dir="ltr">DATA_DICTIONARY</bdi> <bdi dir="ltr">D</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני <bdi dir="ltr">06.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">התיקו לאורך השרשרת</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">E0050:</bdi> הסתברות תיקו מרבית כ־<bdi dir="ltr">31%</bdi> ואפס בחירות תיקו ב־<bdi dir="ltr">argmax</bdi> (<bdi dir="ltr">03,</bdi> <bdi dir="ltr">64–67,</bdi> <bdi dir="ltr">B1.5).</bdi> <bdi dir="ltr">Poisson</bdi> עצמאי: <bdi dir="ltr">25.5%</bdi> חזוי מול <bdi dir="ltr">26.8%</bdi> בפועל (<bdi dir="ltr">EXP-002</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">110–111).</bdi> ב־<bdi dir="ltr">02</bdi> עודף תיקו יחסי כ־<bdi dir="ltr">7%</bdi> מול תהליך אפס מכויל (<bdi dir="ltr">D06</bdi>; <bdi dir="ltr">154–157).</bdi> סוף שלב: <bdi dir="ltr">20.8%</bdi> מול <bdi dir="ltr">28.4%</bdi> (<bdi dir="ltr">C10</bdi>; <bdi dir="ltr">02,</bdi> <bdi dir="ltr">128–133).</bdi> משחקים מתים: כ־<bdi dir="ltr">18%</bdi> (<bdi dir="ltr">03,</bdi> <bdi dir="ltr">205–208,</bdi> <bdi dir="ltr">C12).</bdi> שני האחרונים באותו כיוון: הם עקביים זה עם זה, אבל זו לא הוכחה; לא הופרדו כאן שלב, חוזק ומדגמים חופפים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">האם תיקון התיקו ב־<bdi dir="ltr">M3</bdi> צריך להיות תלוי במצב התחרותי שלפני השריקה (למשל שלב/אין יעד), או להישאר כללי לפי ה-ת<bdi dir="ltr">5</bdi>? תלות נוספת מחייבת הכרעה מפורשת בחוזה <bdi dir="ltr">M3.</bdi> זה אינו מצב תוצאה בתוך המשחק. הסתברות חיובית לתיקו אינה מחייבת לבחור בו ב־<bdi dir="ltr">argmax.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">03,</bdi> <bdi dir="ltr">64–67,</bdi> <bdi dir="ltr">110–115,</bdi> <bdi dir="ltr">205–208</bdi>; <bdi dir="ltr">EXPERIMENTS</bdi> <bdi dir="ltr">EXP-002</bdi>; <bdi dir="ltr">S2</bdi> <bdi dir="ltr">D06/C10</bdi> ומקורותיהם; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">2.4</bdi> ה-ת<bdi dir="ltr">5</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני <bdi dir="ltr">06.</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">איך הופכים את לקחי <bdi dir="ltr">02</bdi> לבדיקות התפלגות בלי לכפות תוצאה?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סף <bdi dir="ltr">34</bdi> נקודות אינו הבטחת הישרדות; יעד שוויון <bdi dir="ltr">0.42</bdi> לכל עונה עלול להסתיר טעות מודל.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סרגל לפי עונה/שלב/אזור עם אי־ודאות; השוואה ל־<bdi dir="ltr">B1</bdi>; לא תיקון נקודות ידני או גורם כאוס.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2</bdi> <bdi dir="ltr">B01/B09/C02–C09</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M4–M6</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לפני <bdi dir="ltr">07</bdi> ומעלה.</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מה מעמד הממצאים ללא תוכנית?</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שיעורי אפס/רצפים, ספירת תארים ו־<bdi dir="ltr">Poisson</bdi> גלובלי נשארו תיאוריים. אין חובה להפוך כל שורה לפיצ׳ר.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">להשאיר לתיעוד בלבד; להציע בדיקת כיול מוגדרת בהמשך; פיצ׳ר חדש רק עם היגיון, זמינות ובדיקה מאושרת מראש.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2</bdi> <bdi dir="ltr">A02/B10/D04</bdi>; כללי ה-ס<bdi dir="ltr">6</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לא דחוף.</td></tr></tbody></table></div>
<p>מועדי ההכרעה הם סדר עבודה לדיון, לא החלטות מתודולוגיות חדשות. ״לפני מחברת <bdi dir="ltr">04</bdi>״ פירושו להגדיר את הקלט ואת מפרט המחקר; ההכרעה הסופית בציון <bdi dir="ltr">M1</bdi> נשארת בסוף המחקר. שאלת התיקו מאחדת את המקורות, בלי למחוק את שאלת המשחקים המתים או את שאר השאלות המקוריות.</p></div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S6</bdi> — מה <bdi dir="ltr">Codex</bdi> מציע לשקול</h2>
<h3>הצעות של <bdi dir="ltr">Codex</bdi> לדיון, לא החלטות</h3>
<p>אלה הצעות עצמאיות לאחר הקריאה. לא קבעתי ספים, לא שיניתי את השרשרת ולא הרצתי בדיקה חדשה. תחילה הנימוק מבחינת נתונים, ואז הפירוש הכדורגלני.</p>
<div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">הצעה — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">נימוק: <bdi dir="ltr">data</bdi> <bdi dir="ltr">scientist</bdi></th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">נימוק כדורגלני</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מאיפה עלתה</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לסכם כל לקח <bdi dir="ltr">EDA</bdi> כתפקיד: קלט, ערך פתיחה, סרגל או תיעוד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קבוע זהה בכל שורות המשחק אינו מבדיל בין משחקים. עדיף להראות שימוש מוגדר ולמנוע דליפה מהסטטיסטיקה המלאה לעבר.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ממוצע אלופה עוזר לבדוק אם תחזית העונה סבירה; ההתקפה הנוכחית עוזרת לחזות את המשחק עצמו. שני השימושים חשובים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2</bdi>; <bdi dir="ltr">PROJECT_PRINCIPLES</bdi> §<bdi dir="ltr">4:</bdi> <bdi dir="ltr">02</bdi> תיאורי בלבד</td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חוזה קנה מידה נפרד ל־<bdi dir="ltr">M1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לתייג <bdi dir="ltr">P(win/draw/loss),</bdi> <bdi dir="ltr">cutoff,</bdi> גרסת <bdi dir="ltr">Elo</bdi> וגרסת המרה; לא לחשב נקודות צפויות מהציון <bdi dir="ltr">elo_expected.</bdi> ב־<bdi dir="ltr">3/1/0</bdi> התוחלת היא <bdi dir="ltr">3P(win)+P(draw).</bdi> זו זהות לפי כלל הניקוד, לא תוצאה חדשה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קבוצה עשויה לקבל סיכוי תוצאה סביר גם אם אינה פייבוריטית. ההשוואה צריכה לשאול כמה עשתה מעבר לצפוי מחוזקה, ולא כמה מעל ציון דירוג גולמי.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S4</bdi> <bdi dir="ltr">D07</bdi>; <bdi dir="ltr">rules.py</bdi>; ה-ת<bdi dir="ltr">1</bdi> וה-ת<bdi dir="ltr">11</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">להתחיל את <bdi dir="ltr">M1</bdi> מהגדרה פשוטה ומכמה השוואות שנקבעות מראש</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">המדגם של משחקים בלי יעד קטן, והדגל קשור לשלב ולחוזק. בקרה והשוואה בין עונות עדיפות לחיפוש הרבה תתי־קבוצות עד שמתקבל אפקט.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לבדוק בנפרד קבוצה שחייבת ניצחון, קבוצה שתיקו משרת אותה וקבוצה שסגרה יעד; הדגל ״אין יעד״ אינו מודד את הרצון של כל היתר באותה סקאלה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">03,</bdi> <bdi dir="ltr">205–208</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1</bdi> ש־<bdi dir="ltr">1/</bdi>ש־<bdi dir="ltr">2</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">7</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">להפריד בין תפקיד טכני למעמד מודל</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">אפשר לשמור בלוק קפוא לניסוי בלי לאפשר לכל מודל לצרוך כל עמודה. רשימת קלט מפורשת יכולה לאכוף גם את החלטות הפסילה וגם את תפקיד <bdi dir="ltr">B4.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מודל טבלה בלבד לא אמור להכיר את היריבה הבאה; בנצ׳מרק עשיר כן יכול לבדוק אם הדרך דרך λ איבדה מידע.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S4</bdi> <bdi dir="ltr">D01–D06</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M5</bdi>; ה-ת<bdi dir="ltr">5</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לאחד את שפת הנקודות והשעות במסמכי החוזה</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">להבדיל בין נקודות טבלה עם חצייה היסטורית, קצב <bdi dir="ltr">3/1/0</bdi> ב־<bdi dir="ltr">form</bdi> ו־<bdi dir="ltr">t−1,</bdi> ויעד רשמי/ספורטיבי; לפרט <bdi dir="ltr">NA</bdi> ומרחק סוף־שלב.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חציית הנקודות משנה מאבק אליפות בלי לשנות כמה טוב הקבוצה שיחקה. משחק שנדחה משנה מה ידוע ביום נתון, גם אם מספר המחזור דומה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S4</bdi> <bdi dir="ltr">D09/D14</bdi>; ה-ס<bdi dir="ltr">3</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">9,</bdi> <bdi dir="ltr">23–24,</bdi> <bdi dir="ltr">181–182</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לתכנן מראש בדיקות כיול לעונה ולא רק <bdi dir="ltr">loss</bdi> למשחק</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הצטברות טעויות קטנות ב-λ ובתיקו עשויה לשנות פיזור נקודות וגבולות פלייאוף; אי אפשר לתקן זאת בהכרח בעזרת עוד מסלולים.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">סימולטור יכול לחזות היטב את האלופה ולפספס את מאבק הירידה. לבדוק קבוצות צמרת, קו עליון ותחתית בנפרד, בלי לכפות את ממוצעי ה־<bdi dir="ltr">EDA.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S2</bdi> <bdi dir="ltr">B09/C02–C06</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M4–M6</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">להכריע את חוזה הזנב לפני מימוש <bdi dir="ltr">M3</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">איגום שומר מסה אבל מאבד סדר בתוך תא הזנב. יש להבחין בין <bdi dir="ltr">log</bdi> <bdi dir="ltr">loss</bdi> לקטגוריה, הסתברות לתיקו אמיתי ותוצאה שנגרלת למסלול.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">תוצאה גבוהה של שני הצדדים אינה בהכרח תיקו. ההכרעה צריכה להיות גלויה לפני שהנקודות מוזנות לסימולציית עונה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">S4</bdi> <bdi dir="ltr">D16</bdi>; <bdi dir="ltr">DATA_DICTIONARY</bdi> <bdi dir="ltr">observed_vs_poisson</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לתת לשווי הזדמנות מאושרת בלי להתייחס אליו כאמת</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">שווי ממוצע נבחר בחריגה אחרי התוצאות. בדיקת תרומה ב־<bdi dir="ltr">M2</bdi> צריכה להישאר עצמאית, עם עומק אימון שהוכרז מראש ותיעוד חמש האחרונות.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הערכת שוק עשויה לזהות עולה חזקה לפני ש־<bdi dir="ltr">Elo</bdi> למד אותה, אך דף שווי אינו הרכב צפוי, תקציב אמיתי או הסבר לכל הפתעה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ה-ת<bdi dir="ltr">13/</bdi>ה-ת<bdi dir="ltr">14</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">132–133,</bdi> <bdi dir="ltr">170–172,</bdi> <bdi dir="ltr">191–192</bdi></td></tr></tbody></table></div>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S7</bdi> — גשר למחברת <bdi dir="ltr">04:</bdi> מה <bdi dir="ltr">M1</bdi> מקבל ומה עוד צריך להגדיר</h2>
<p><bdi dir="ltr">M1</bdi> מתחיל ממצב שמור לפני שריקה, ולא מחישוב מחדש של העונה או מטבלה רשמית. מחברת <bdi dir="ltr">03</bdi> מספקת את שכבת המידע; היא לא מייצרת עדיין ציון מוטיבציה. המקור: <bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">9,</bdi> <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1</bdi> ו-§<bdi dir="ltr">7,</bdi> <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1</bdi> ומחברת <bdi dir="ltr">03,</bdi> <bdi dir="ltr">175–213.</bdi></p>
<p>ציפיית <bdi dir="ltr">Elo</bdi> המכוילת קיימת בתוצרי <bdi dir="ltr">EXP-001,</bdi> אך אינה שקולה לעמודת <bdi dir="ltr">elo_expected</bdi> בטבלת <bdi dir="ltr">prematch.</bdi> לפני מחקר השאריות נדרש חיבור מתועד לפי מפתחות, גרסה וסוף אימון, כולל החלק שאין בו תחזיות <bdi dir="ltr">OOF</bdi> שמורות. גם חיבור לוח העתיד הוא שלב מפורש, ולא עמודה שכבר נמצאת בטבלה.</p>
<div style="overflow-x:auto"><table dir="rtl" style="width:100%;border-collapse:collapse;font-size:14px;line-height:1.6"><thead><tr><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מה מועבר — מה זה, במילים</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">זמין כעת</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">התפקיד ב־<bdi dir="ltr">M1</bdi> / מה עדיין חסר</th><th style="text-align:right;vertical-align:top;padding:8px;border-bottom:2px solid #789">מקור</th></tr></thead><tbody><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מפתחות ותאריך מידע</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">game_id,</bdi> <bdi dir="ltr">team_id,</bdi> <bdi dir="ltr">opponent_id,</bdi> <bdi dir="ltr">season_id,</bdi> <bdi dir="ltr">date,</bdi> <bdi dir="ltr">as_of</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לשמור שתי נקודות מבט של אותו משחק יחד; יום משחק אינו רואה תוצאות מאותו יום.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">DATA_DICTIONARY</bdi> <bdi dir="ltr">prematch</bdi>; <bdi dir="ltr">03,</bdi> <bdi dir="ltr">179–180</bdi>; <bdi dir="ltr">T-010</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">מצב הטבלה הספורטיבית</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">נקודות, הפרש, מיקום, פערים, משחקים שנותרו וחסרים, שלב</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קלט להגדרות חשיבות שקופות; ערך חסר בפער אחרי פיצול אינו אפס.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">03,</bdi> <bdi dir="ltr">9–14,</bdi> <bdi dir="ltr">177–182</bdi>; ה-ס<bdi dir="ltr">3</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">הוכחות יעד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">nothing_to_play</bdi> והדגלים <bdi dir="ltr">title/cut/relegation/places_decided</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קו בסיס שמרני, לא תווית אמת למוטיבציה; לשמור מגבלת מבחן הירידה ולהגדיר מה נחשב יעד.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">03,</bdi> <bdi dir="ltr">15–18,</bdi> <bdi dir="ltr">25–26</bdi>; ה-ת<bdi dir="ltr">3/</bdi>ה-ת<bdi dir="ltr">10</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חוזק <bdi dir="ltr">Elo</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">ארבעת שדות <bdi dir="ltr">Elo</bdi> ב־<bdi dir="ltr">prematch</bdi>; המרות ותחזיות מכוילות בתוצרי <bdi dir="ltr">EXP-001</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">קנה מידה לתוצאה וסימולטור חשיבות בגרסה קבועה מהעבר; חיבור <bdi dir="ltr">OOF</bdi> והתחממות עוד דורשים חוזה.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">03,</bdi> <bdi dir="ltr">32–43,</bdi> <bdi dir="ltr">75,</bdi> <bdi dir="ltr">178</bdi>; <bdi dir="ltr">S4</bdi> <bdi dir="ltr">D07</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">התקפה/הגנה והקשר אשתקד</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">ewma5/venue,</bdi> שדות <bdi dir="ltr">t−1</bdi> והפרשי יריבות</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">זמינים לשכבה הבאה; אין החלטה להכניסם ישירות להגדרת <bdi dir="ltr">M1,</bdi> שקלטה לפי <bdi dir="ltr">VISION</bdi> הוא <bdi dir="ltr">M0-A.</bdi> אפשר לדון בהם כבקרת חוזק נוספת, לא להכניסם בשקט.</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">03,</bdi> <bdi dir="ltr">177–180,</bdi> <bdi dir="ltr">210</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1/M2</bdi>; <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חוקים ולוח</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">rules.py</bdi> וקונפיגורציית עונות; ספירות במשחק הנוכחי</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">לרגישות תרחישים דרוש גם <bdi dir="ltr">snapshot</bdi> של צמדי המשחקים שנותרו ולוח פלייאוף לפי המצב המדומה. אין להסתמך על שיבוץ עתידי שלא היה ידוע ב־<bdi dir="ltr">as_of.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1/M4</bdi>; <bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">2</bdi>; <bdi dir="ltr">S4</bdi> <bdi dir="ltr">D08</bdi></td></tr><tr><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">חוזה יציאה להמשך</td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">טרם נבנה ציון <bdi dir="ltr">M1</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3">להגדיר ציון לכל צד, גרסה, סוף אימון אם יש למידה וכלל חימום; המבחן הקובע בהמשך הוא <bdi dir="ltr">ablation</bdi> בלי/טבלה/ציון ב־<bdi dir="ltr">M2.</bdi></td><td style="text-align:right;vertical-align:top;padding:8px;border-bottom:1px solid #d5dce3"><bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">4.4/</bdi>§<bdi dir="ltr">4.14,</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1/M2</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1</bdi></td></tr></tbody></table></div>
</div>

<div dir="rtl" style="text-align:right;line-height:1.7">
<h2><bdi dir="ltr">S7</bdi> — השאלות הראשונות של <bdi dir="ltr">M1</bdi></h2>
<p>מהו בדיוק היעד התחרותי שנמדד, ואילו תוצאות משנות אותו? האם ״תיקו מספיק״ שונה מ״חייבת לנצח״ גם אחרי שמתחשבים בחוזק, בשלב וביריבה? האם השארית של הפלייאוף התחתון נשארת לאחר שינוי היריבות המכני? אלה שאלות מחקר, לא תשובות שכבר מצאנו.</p>
<p>איך מודדים את הפער מהצפוי: תוצאת בית/תיקו/חוץ או נקודות צפויות לפי ההמרה המכוילת? אם רוצים לבחון שערים, מה קו הבסיס המאושר לכך לפני <bdi dir="ltr">M2</bdi>? איך מחשבים את החשיבות מחדש במסלול מדומה, ומה המקור שהיה ידוע ללוח העתיד?</p>
<p>סדר מוצע לדיון בלבד: להכריע בחוזי הקלט וביעדים, לבחור השוואות מראש, ואז לכתוב מפרט מחקר למחברת <bdi dir="ltr">04.</bdi> לא להחליף את <bdi dir="ltr">Elo,</bdi> לא להוסיף פיצ׳ר ולא להסיק סיבתיות מתוך שיעור התיקו של המשחקים המתים.</p>
<p>מקורות: <bdi dir="ltr">VISION</bdi> §<bdi dir="ltr">5</bdi> <bdi dir="ltr">M1</bdi> ו-§<bdi dir="ltr">7</bdi>; <bdi dir="ltr">FEATURES</bdi> <bdi dir="ltr">M1</bdi> ש־<bdi dir="ltr">1/</bdi>ש־<bdi dir="ltr">2/</bdi>ש־<bdi dir="ltr">4</bdi>; <bdi dir="ltr">T-010</bdi> §<bdi dir="ltr">9</bdi>; מחברת <bdi dir="ltr">03,</bdi> <bdi dir="ltr">205–213</bdi>; <bdi dir="ltr">S4–S6</bdi> לעיל.</p>
</div>